In [1]:
#mohammedammar79845@gmail.com
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")

In [3]:
#import data.csv file.
from google.colab import files
upload = files.upload()

Saving truthABCpoorDQ.txt to truthABCpoorDQ.txt


In [4]:
from __future__ import annotations
import csv
import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional


def _normalize_recid(x: object) -> str:
    """Normalize record ID to uppercase and stripped string."""
    return str(x).strip().upper()


def _normalize_value(val: object) -> str:
    """
    Normalize field values by uppercasing, removing punctuation,
    and collapsing whitespace. Keeps only A-Z, 0-9 and spaces.
    """
    if val is None:
        return ""
    s = str(val).strip()
    if not s:
        return ""
    s = s.upper()
    s = re.sub(r"[^A-Z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _find_index_case_insensitive(names: List[str], target: str) -> Optional[int]:
    """Find column index by case-insensitive name match."""
    tgt = target.lower()
    for i, n in enumerate(names):
        if str(n).strip().lower() == tgt:
            return i
    return None


def _truth_id_col(fieldnames: List[str]) -> Optional[str]:
    """Identify the truth/group ID column from common naming patterns."""
    candidates = [
        "idtruth", "truthid", "truth_id",
        "clusterid", "cluster_id", "cluster",
        "groupid", "group_id", "group"
    ]
    lower_map = {fn.lower(): fn for fn in fieldnames}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def read_truth_map(truth_path: Path) -> Dict[str, int]:
    """
    Read truth file and build mapping from RecID to truth group ID.
    Returns dictionary mapping normalized RecIDs to integer group IDs.
    """
    truth_map: Dict[str, int] = {}
    with truth_path.open("r", newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        tf_fields = reader.fieldnames or []

        recid_col = None
        for cand in ["recid", "RecID"]:
            recid_col = recid_col or (cand if cand in tf_fields else None)
        if not recid_col:
            for fn in tf_fields:
                if fn.lower() == "recid":
                    recid_col = fn
                    break

        id_col = _truth_id_col(tf_fields)
        if not recid_col or not id_col:
            return truth_map

        for row in reader:
            recid_raw = row.get(recid_col, "")
            idtruth_raw = row.get(id_col, "")
            recid = _normalize_recid(recid_raw)
            if not recid:
                continue
            if idtruth_raw is None or str(idtruth_raw).strip() == "":
                continue
            try:
                idtruth = int(str(idtruth_raw).strip())
            except ValueError:
                continue
            truth_map[recid] = idtruth
    return truth_map


def process_files(
    source_path: Path,
    truth_path: Path,
    output_path: Optional[Path] = None,
) -> Path:
    """
    Process source file using truth mapping to group and relabel records.

    Args:
        source_path: Path to input CSV file
        truth_path: Path to truth mapping file
        output_path: Optional output path (defaults to 'a_concatenated.csv')

    Returns:
        Path to the generated output file
    """
    truth_map = read_truth_map(truth_path)

    with source_path.open("r", newline="", encoding="utf-8-sig") as f:
        rdr = csv.reader(f)
        rows_raw = list(rdr)

    if not rows_raw:
        raise ValueError("Input file is empty.")

    header = rows_raw[0]
    data_rows = rows_raw[1:]

    recid_idx = _find_index_case_insensitive(header, "recid")
    if recid_idx is None:
        recid_idx = 0

    items = []
    for idx, cells in enumerate(data_rows):
        if len(cells) <= recid_idx:
            recid_val = ""
        else:
            recid_val = cells[recid_idx]
        items.append({
            "__idx": idx,
            "cells": cells,
            "recid": recid_val,
            "__recid_norm__": _normalize_recid(recid_val),
        })

    matched = []
    unmatched = []
    for it in items:
        if it["__recid_norm__"] in truth_map:
            matched.append(it)
        else:
            unmatched.append(it)

    present_idtruths = sorted({truth_map[it["__recid_norm__"]] for it in matched}) if matched else []
    idtruth_remap: Dict[int, int] = {orig: i + 1 for i, orig in enumerate(present_idtruths)}

    per_truth_counters: Dict[int, int] = {}
    relabeled = []
    for it in matched:
        orig_truth = truth_map[it["__recid_norm__"]]
        grp = idtruth_remap[orig_truth]
        per_truth_counters[orig_truth] = per_truth_counters.get(orig_truth, 0) + 1
        k = per_truth_counters[orig_truth]
        it2 = dict(it)
        it2["recid"] = f"{grp}.{k}"
        relabeled.append(it2)

    def parse_new_recid(val: str) -> Tuple[int, int]:
        try:
            g_str, k_str = val.split(".", 1)
            return (int(g_str), int(k_str))
        except Exception:
            return (10**12, 10**12)

    relabeled_sorted = sorted(relabeled, key=lambda r: parse_new_recid(str(r["recid"])))
    final_items = relabeled_sorted + unmatched

    out_rows = []
    for it in final_items:
        cells = it["cells"]
        parts: List[str] = []
        for j, cell in enumerate(cells):
            if j == recid_idx:
                continue
            norm = _normalize_value(cell)
            if norm:
                parts.append(norm)
        concatenated = " ".join(parts)
        out_rows.append({
            "RecID": it["recid"],
            "concatenated": concatenated,
        })

    out_path = output_path if output_path else Path("a_concatenated.csv")
    with out_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["RecID", "concatenated"])
        writer.writeheader()
        writer.writerows(out_rows)

    print(f"Matched rows: {len(matched)}  |  Unmatched rows: {len(unmatched)}")
    if len(matched) == 0 and truth_map:
        print("Note: Truth map loaded, but no RecIDs matched after normalization.")
    if not truth_map:
        print("Warning: No truth IDs were loaded (missing/unknown truth column names in truth file?).")

    return out_path


def main() -> None:
    """Main entry point for interactive command-line usage."""
    import sys
    try:
        src = input("Enter the input filename (e.g., S1G.txt): ").strip()
        if not src:
            print("No input filename provided.")
            sys.exit(1)
        src_path = Path(src)

        default_truth = "truthABCgoodDQ.txt"
        truth_in = input(f"Enter the truth filename [default: {default_truth}]: ").strip() or default_truth
        truth_path = Path(truth_in)

        if not src_path.exists():
            print(f"Input file not found: {src_path}")
            sys.exit(1)
        if not truth_path.exists():
            print(f"Truth file not found: {truth_path}")
            sys.exit(1)

        out = process_files(src_path, truth_path)
        print(f"Output written to: {out}")

    except KeyboardInterrupt:
        print("\nAborted by user.")
    except Exception as e:
        print(f"Error: {e}")
        sys.exit(1)


if __name__ == "__main__":
    main()

Enter the input filename (e.g., S1G.txt): S18PX.txt
Enter the truth filename [default: truthABCgoodDQ.txt]: truthABCpoorDQ.txt
Matched rows: 10000  |  Unmatched rows: 0
Output written to: a_concatenated.csv


In [5]:
from __future__ import annotations

import asyncio
import csv
import json
import os
import re
from collections import Counter
from typing import Iterable, List, Dict, Optional, Tuple

import pandas as pd
from tqdm import tqdm

# Configuration
INPUT_CSV  = "a_concatenated.csv"
MODEL      = "gpt-4.1"
BATCH_SIZE = 32
MAX_CONCURRENCY = 12
RETRIES    = 3
BACKOFF_S  = 1.5
ADD_PARSED_BY_TO_MAIN = False


def chunkify(lst: List[dict], n: int) -> Iterable[List[dict]]:
    """Split list into chunks of size n."""
    for i in range(0, len(lst), n):
        yield lst[i : i + n]


def collapse_spaces(s: str) -> str:
    """Normalize whitespace to single spaces."""
    return re.sub(r"\s+", " ", s.strip())


# Rule-Based Parsing Components
ZIP_LONG_RUN_RE      = re.compile(r"\d{8,}")
NUMERIC_DOTTED_RE    = re.compile(r"\s+[0-9]*\.[0-9]+")
HYPHENATED_TOKENS_RE = re.compile(r"\s+\S*-\S*")
POBOX_RE             = re.compile(r"(?i)\bPO\s*BOX\b")

ZIP5_RE              = re.compile(r"\b(\d{5})(?![A-Za-z])\b")
ZIP9_RE              = re.compile(r"\b(\d{9})\b")
NOISY5_A23_RE        = re.compile(r"\b(\d{2})[A-Za-z](\d{3})\b")
NOISY5_A32_RE        = re.compile(r"\b(\d{3})[A-Za-z](\d{2})\b")


def find_zip_span(addr: str, start_idx: int) -> Optional[Tuple[int, int, str]]:
    """
    Find a ZIP-like span in address string starting at given index.

    Returns:
        Tuple of (zip_start, zip_end, zip_text) or None if not found.
        Priority: ZIP5 > NOISY5 > ZIP9 > 4-digit fallback rule
    """
    tail = addr[start_idx:]

    m = ZIP5_RE.search(tail)
    if m:
        s, e = m.span()
        return (start_idx + s, start_idx + e, m.group(1))

    m = NOISY5_A23_RE.search(tail)
    if m:
        s, e = m.span()
        digits = m.group(1) + m.group(2)
        return (start_idx + s, start_idx + e, digits)

    m = NOISY5_A32_RE.search(tail)
    if m:
        s, e = m.span()
        digits = m.group(1) + m.group(2)
        return (start_idx + s, start_idx + e, digits)

    m = ZIP9_RE.search(tail)
    if m:
        s, e = m.span()
        return (start_idx + s, start_idx + s + 5, m.group(1)[:5])

    runs = [(m.start(), m.end(), m.group()) for m in re.finditer(r"\d+", tail)]
    for i, (rs, re_, txt) in enumerate(runs):
        if len(txt) == 4:
            later = next((r for r in runs[i+1:] if len(r[2]) >= 6), None)
            if later:
                return (start_idx + rs, start_idx + rs + 4, txt)

    return None


def rule_based_address_truncate(address: str) -> str:
    """Apply PO BOX removal, ZIP detection, truncation, and cleanup."""
    addr = collapse_spaces(address)

    m_po = POBOX_RE.search(addr)
    if m_po:
        addr = addr[: m_po.start()].strip()

    digit_runs = list(re.finditer(r"\d+", addr))
    if digit_runs:
        search_from = digit_runs[0].end()
    else:
        search_from = 0

    zip_span = find_zip_span(addr, search_from)
    if zip_span is not None:
        _, zip_end, _ = zip_span
        truncated = addr[:zip_end].strip()
    else:
        truncated = addr

    truncated = re.split(r"\s*\(", truncated)[0].strip()
    truncated = re.split(ZIP_LONG_RUN_RE, truncated)[0].strip()
    truncated = re.split(HYPHENATED_TOKENS_RE, truncated)[0].strip()
    truncated = re.split(NUMERIC_DOTTED_RE, truncated)[0].strip()

    return collapse_spaces(truncated)


def rule_parse_record(rec: Dict[str, str], tag: str = "rule") -> Dict[str, str]:
    """Parse a single record using rule-based approach."""
    recid = str(rec.get("RecID", "")).strip()
    concat = str(rec.get("concatenated", "")).strip()

    if not concat:
        return {"RecID": recid, "Name": "", "Address": "", "ParsedBy": tag}

    m = re.search(r"\d", concat)
    if m:
        name = collapse_spaces(concat[: m.start()])
        addr_candidate = collapse_spaces(concat[m.start() :])
    else:
        return {"RecID": recid, "Name": collapse_spaces(concat), "Address": "", "ParsedBy": tag}

    address = rule_based_address_truncate(addr_candidate)
    return {"RecID": recid, "Name": name, "Address": address, "ParsedBy": tag}


def parse_records_rule_based(records: List[Dict[str, str]], tag: str = "rule") -> List[Dict[str, str]]:
    """Parse all records using rule-based approach."""
    return [rule_parse_record(rec, tag=tag) for rec in tqdm(records, desc="Rule-based parsing")]


def extract_results_array(json_text: str) -> List[dict]:
    """Parse JSON response from LLM, with fallback to regex extraction."""
    try:
        obj = json.loads(json_text)
        if isinstance(obj, dict) and isinstance(obj.get("results"), list):
            return obj["results"]
    except Exception:
        pass

    m = re.search(r"\[.*\]", json_text, re.DOTALL)
    if m:
        try:
            arr = json.loads(m.group(0))
            return arr if isinstance(arr, list) else []
        except Exception:
            return []
    return []


async def async_llm_parse_batch(batch: List[Dict[str, str]], model: str, client, sem: asyncio.Semaphore) -> List[Dict[str, str]]:
    """Process a single batch of records using OpenAI API with retries and concurrency control."""
    system_prompt = {
        "role": "system",
        "content": (
            "You are a meticulous data extraction service. For each record, split the 'concatenated' string into 'Name' and 'Address'.\n"
            "Output STRICT JSON with a single top-level key 'results' that maps to an array.\n"
            "Each array item MUST be an object with keys: 'RecID', 'Name', 'Address'.\n\n"
            "Rules to follow exactly:\n"
            "1) Name = all text BEFORE the first digit in 'concatenated'.\n"
            "2) Address = starts at the FIRST digit and extends THROUGH the ZIP code. The ZIP heuristics are:\n"
            "   - Prefer a 5-digit run NOT followed by letters; OR\n"
            "   - A 4-digit run followed later by a run of >=6 digits. Use the 4-digit point.\n"
            "3) If 'PO BOX' appears, drop it and everything after.\n"
            "4) EXCLUDE anything after the ZIP: apartment numbers, phone numbers, SSNs, hyphenated tokens, numeric dotted tokens.\n"
            "5) Preserve dotted tokens with letters (e.g., 'St.').\n"
            "6) Trim extra spaces; return clean strings only."
        ),
    }

    user_prompt = {
        "role": "user",
        "content": (
            'Return ONLY JSON like: {"results":[{"RecID":"...","Name":"...","Address":"..."}, ...]}\n'
            + json.dumps(batch, ensure_ascii=False)
        ),
    }

    for attempt in range(1, RETRIES + 1):
        try:
            async with sem:
                resp = await client.chat.completions.create(
                    model=model,
                    temperature=0.0,
                    response_format={"type": "json_object"},
                    messages=[system_prompt, user_prompt],
                )
            raw = resp.choices[0].message.content or ""
            arr = extract_results_array(raw)

            results = []
            for r in arr:
                results.append({
                    "RecID": str(r.get("RecID", "")),
                    "Name": collapse_spaces(str(r.get("Name", ""))),
                    "Address": collapse_spaces(str(r.get("Address", ""))),
                    "ParsedBy": "llm",
                })
            return results
        except Exception as e:
            if attempt == RETRIES:
                tqdm.write(f"LLM API error (final) on batch starting RecID={batch[0].get('RecID','?')}: {e}")
            else:
                wait = BACKOFF_S * (2 ** (attempt - 1))
                tqdm.write(f"LLM API error (attempt {attempt}) on batch starting RecID={batch[0].get('RecID','?')}: {e}. Retrying in {wait:.1f}s...")
                await asyncio.sleep(wait)

    return []


async def parse_records_llm_async(records: List[Dict[str, str]], model: str, batch_size: int) -> List[Dict[str, str]]:
    """Parse records using LLM with concurrent batch processing."""
    try:
        from openai import AsyncOpenAI  # type: ignore
    except Exception as e:
        raise RuntimeError("OpenAI package not available. Install `openai`.") from e

    if "OPENAI_API_KEY" not in os.environ:
        raise RuntimeError("OPENAI_API_KEY not set in environment for LLM mode.")

    client = AsyncOpenAI()
    out: List[Dict[str, str]] = []
    batches = list(chunkify(records, batch_size))

    print(f"Submitting {len(batches)} batches to LLM concurrently (cap={MAX_CONCURRENCY})...")
    sem = asyncio.Semaphore(MAX_CONCURRENCY)
    tasks = [async_llm_parse_batch(batch, model, client, sem) for batch in batches]

    all_parsed_results = await asyncio.gather(*tasks)
    print("LLM processing complete. Applying fallbacks and finalizing...")

    for original_batch, parsed_results in tqdm(zip(batches, all_parsed_results), total=len(batches), desc="Processing results"):
        parsed_map = {r.get("RecID"): r for r in parsed_results if r.get("RecID") is not None}
        for rec in original_batch:
            rid = rec["RecID"]
            if rid in parsed_map:
                out.append(parsed_map[rid])
            else:
                out.append(rule_parse_record(rec, tag="fallback_rule"))
    return out


# Sanity Check Patterns
ZIP5_END_RE = re.compile(r"\b\d{5}\b$")
PHONE_RE    = re.compile(r"(\b\d{10}\b|\(\d{3}\)\s*\d{3}[-.\s]?\d{4}|\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b)")
SSN_RE      = re.compile(r"\b\d{3}-?\d{2}-?\d{4}\b")
APT_RE      = re.compile(r"\b(apt|apartment|unit|#)\b", re.IGNORECASE)


def sanity_check_row(row: Dict[str, str]) -> Dict[str, str]:
    """Perform quality checks on parsed row and generate audit metadata."""
    recid   = str(row.get("RecID",""))
    name    = str(row.get("Name",""))
    address = str(row.get("Address",""))
    parsed  = str(row.get("ParsedBy",""))

    issues = []

    if not name: issues.append("empty_name")
    if not address: issues.append("empty_address")
    if re.search(r"\d", name): issues.append("name_has_digits")
    if not ZIP5_END_RE.search(address): issues.append("zip_missing_or_not_at_end")
    if PHONE_RE.search(address): issues.append("phone_in_address")
    if SSN_RE.search(address): issues.append("ssn_in_address")
    if APT_RE.search(address): issues.append("apt_mention")

    if "empty_address" in issues or "ssn_in_address" in issues:
        quality = "FAIL"
    elif issues:
        quality = "WARN"
    else:
        quality = "OK"

    return {
        "RecID": recid,
        "ParsedBy": parsed or "",
        "NameLen": len(name),
        "AddressLen": len(address),
        "Quality": quality,
        "Issues": ";".join(issues) if issues else "",
        "ZipAtEnd": bool(ZIP5_END_RE.search(address)),
        "NameHasDigits": bool(re.search(r"\d", name)),
        "PhoneLike": bool(PHONE_RE.search(address)),
        "SSNLike": bool(SSN_RE.search(address)),
        "AptMention": bool(APT_RE.search(address)),
    }


async def main():
    """Main execution function supporting both rule-based and LLM parsing modes."""
    choice = input("Choose parsing mode (type 'rule' or 'llm'): ").strip().lower()
    if choice not in {"rule", "llm"}:
        print("Invalid choice. Please run again and type 'rule' or 'llm'.")
        return

    OUTPUT_CSV = "b_processed_results_LLM.csv" if choice == "llm" else "b_processed_results.csv"

    try:
        df = pd.read_csv(INPUT_CSV, dtype=str).fillna("")
    except FileNotFoundError:
        print(f"Error: Input file '{INPUT_CSV}' not found in the current folder.")
        return

    required = {"RecID", "concatenated"}
    missing = required - set(df.columns)
    if missing:
        print(f"Error: Input is missing required columns: {sorted(missing)}")
        return

    records: List[Dict[str, str]] = df[["RecID", "concatenated"]].to_dict(orient="records")

    if choice == "rule":
        results = parse_records_rule_based(records, tag="rule")
    else:
        results = await parse_records_llm_async(records, model=MODEL, batch_size=BATCH_SIZE)

    out_df = pd.DataFrame(results, dtype=str).fillna("")
    out_df.rename(columns={"name": "Name", "address": "Address"}, inplace=True)

    main_cols = ["RecID", "Name", "Address"]
    if ADD_PARSED_BY_TO_MAIN:
        if "ParsedBy" not in out_df.columns:
            out_df["ParsedBy"] = ""
        main_cols.append("ParsedBy")

    out_df = out_df[main_cols]
    out_df.to_csv(OUTPUT_CSV, index=False, quoting=csv.QUOTE_ALL)

    AUDIT_CSV = OUTPUT_CSV.replace(".csv", "_audit.csv")
    audit_rows = [sanity_check_row(r) for r in tqdm(results, desc="Sanity checks")]
    audit_df = pd.DataFrame(audit_rows)
    audit_df.to_csv(AUDIT_CSV, index=False, quoting=csv.QUOTE_ALL)

    print("\n=== Parse Provenance ===")
    by_source = audit_df["ParsedBy"].value_counts(dropna=False).to_dict()
    for k in ("llm", "fallback_rule", "rule", ""):
        if k in by_source and by_source[k] > 0:
            label = k if k else "(unknown)"
            print(f"{label:>14}: {by_source[k]}")

    print("\n=== Quality Summary ===")
    by_quality = audit_df["Quality"].value_counts(dropna=False).to_dict()
    for k in ("OK", "WARN", "FAIL"):
        if k in by_quality:
            print(f"{k:>5}: {by_quality[k]}")

    common_issues = Counter(";".join([i for i in audit_df["Issues"] if i]).split(";"))
    if common_issues:
        print("\nTop issues:")
        for issue, cnt in common_issues.most_common(10):
            if issue:
                print(f" - {issue}: {cnt}")

    print(f"\n✔ Execution complete. Saved: {OUTPUT_CSV}")
    print(f"✔ Audit saved: {AUDIT_CSV}")

await main()

Choose parsing mode (type 'rule' or 'llm'): llm
Submitting 313 batches to LLM concurrently (cap=12)...
LLM processing complete. Applying fallbacks and finalizing...


Sanity checks: 100%|██████████| 10000/10000 [00:00<00:00, 81796.51it/s]



=== Parse Provenance ===
           llm: 10000

=== Quality Summary ===
   OK: 8469
 WARN: 1531

Top issues:
 - apt_mention: 804
 - zip_missing_or_not_at_end: 771
 - name_has_digits: 1

✔ Execution complete. Saved: b_processed_results_LLM.csv
✔ Audit saved: b_processed_results_LLM_audit.csv


In [ ]:
import os
import json
import re
import time
import glob
import asyncio
from typing import List, Dict, Optional

import pandas as pd
from openai import AsyncOpenAI

# Configuration
client = AsyncOpenAI(
    api_key=os.getenv(
        "OPENAI_API_KEY",
        ""
    )
)

PRIMARY_MODEL = "o4-mini"
FALLBACK_MODEL = "o3"
PRIMARY_ATTEMPTS = 2

INPUT_CSV = "b_processed_results_LLM.csv"
OUTPUT_CSV = "c_parsed_names.csv"
CHUNK_SIZE = 64
OVERLAP_SIZE = 16
CONCURRENT_REQUESTS = 10

TMP_DIR = "test_c_tmp_parsed_names_batches"
RESUME_FROM_CHECKPOINTS = True


def ensure_dir(path: str) -> None:
    """Create directory if it doesn't exist."""
    if not os.path.isdir(path):
        os.makedirs(path, exist_ok=True)


def chunk_records_with_overlap(records: List[Dict], chunk_size: int, overlap_size: int):
    """
    Create chunks with overlap to ensure name variations appear together in at least one batch.
    The overlap ensures that records near batch boundaries get processed with context from both sides.
    """
    yield records[:chunk_size]

    i = chunk_size
    while i < len(records):
        start_idx = max(0, i - overlap_size)
        end_idx = min(i + chunk_size - overlap_size, len(records))

        chunk = records[start_idx:end_idx]

        chunk_with_metadata = []
        for j, record in enumerate(chunk):
            record_copy = record.copy()
            record_copy['_is_overlap'] = (start_idx + j) < i
            record_copy['_original_index'] = start_idx + j
            chunk_with_metadata.append(record_copy)

        yield chunk_with_metadata

        i += (chunk_size - overlap_size)


def extract_json_array(text: str) -> str:
    """Extract JSON array from potentially malformed LLM response."""
    start = text.find('[')
    end = text.rfind(']')
    if start == -1 or end == -1 or start > end:
        raise ValueError("No JSON array delimiters found")
    return text[start:end + 1]


def load_checkpoint(batch_idx: int) -> Optional[List[Dict]]:
    """
    Load an existing checkpoint for the given batch index.
    Returns parsed list if available and valid, else None.
    """
    path = os.path.join(TMP_DIR, f"batch_{batch_idx:05d}.json")
    part = path + ".part"

    for candidate in (path, part):
        if os.path.exists(candidate) and os.path.getsize(candidate) > 0:
            try:
                with open(candidate, "r", encoding="utf-8") as f:
                    data = json.load(f)
                if isinstance(data, list):
                    return data
            except Exception:
                pass
    return None


def save_checkpoint(batch_idx: int, data: List[Dict]) -> None:
    """Atomically write the batch results to a checkpoint file."""
    ensure_dir(TMP_DIR)
    path = os.path.join(TMP_DIR, f"batch_{batch_idx:05d}.json")
    tmp_path = path + ".part"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    os.replace(tmp_path, path)


def consolidate_checkpoints_with_dedup() -> pd.DataFrame:
    """
    Read all batch checkpoint files and return a DataFrame of parsed rows.
    Handles deduplication of overlapped records by preferring the most complete version.
    """
    ensure_dir(TMP_DIR)
    files = sorted(glob.glob(os.path.join(TMP_DIR, "batch_*.json")))

    best_records = {}

    for fp in files:
        try:
            with open(fp, "r", encoding="utf-8") as f:
                rows = json.load(f)
            if isinstance(rows, list):
                for record in rows:
                    if isinstance(record, dict) and 'RecID' in record:
                        rec_id = record['RecID']

                        if rec_id not in best_records:
                            best_records[rec_id] = record
                        else:
                            current = best_records[rec_id]
                            new_completeness = sum(1 for v in record.values() if v and str(v).strip())
                            current_completeness = sum(1 for v in current.values() if v and str(v).strip())

                            new_last = str(record.get('last_name', ''))
                            current_last = str(current.get('last_name', ''))

                            if new_completeness > current_completeness or \
                               (new_completeness == current_completeness and len(new_last) > len(current_last)):
                                best_records[rec_id] = record
        except Exception as e:
            print(f"Warning: Could not read checkpoint file {fp}: {e}")
            pass

    if not best_records:
        return pd.DataFrame(columns=["RecID", "first_name", "middle_name", "last_name"])

    return pd.DataFrame(list(best_records.values()), dtype=str)


async def parse_chunk_async(chunk: List[Dict], model: str) -> str:
    """Process a chunk of records using the LLM API."""
    clean_chunk = []
    for record in chunk:
        clean_record = {k: v for k, v in record.items() if not k.startswith('_')}
        clean_chunk.append(clean_record)

    system_message = {
        "role": "system",
        "content": (
            "You are an expert in batch name standardization and normalization. Process the entire batch systematically.\n"
            "\n"
            "## STEP-BY-STEP PROCESS:\n"
            "\n"
            "### 1. NORMALIZE\n"
            "• Convert to uppercase\n"
            "• Remove all punctuation except spaces\n"
            "• Replace multiple spaces with single spaces\n"
            "• Trim whitespace\n"
            "\n"
            "### 2. ANALYZE BATCH PATTERNS\n"
            "• Identify all unique tokens across the entire batch\n"
            "• Count frequency of each token in each position (first/middle/last)\n"
            "• Build a comprehensive nickname-to-formal name mapping from the batch data\n"
            "\n"
            "### 3. NICKNAME & ABBREVIATION EXPANSION\n"
            "Apply these rules in order:\n"
            "• **Common nicknames**: MIKE→MITCHELL, MIKE→MICHAEL, BOB→ROBERT, JIM→JAMES, etc.\n"
            "• **Initials**: Single letters (M, J, S) → most frequent full name in that position\n"
            "• **Partial names**: MITCH→MITCHELL, JACKIE→JACQUELINE if the full form exists in batch\n"
            "• **Fuzzy matches**: Levenshtein distance ≤2 with 80%+ similarity\n"
            "\n"
            "### 4. BATCH-BASED INFERENCE\n"
            "For incomplete names, infer missing parts by:\n"
            "• Finding records with same last name and complete information\n"
            "• Using most frequent first/middle name combination for that family\n"
            "• Prioritizing exact matches over partial matches\n"
            "\n"
            "### 5. STANDARDIZE TO CANONICAL FORM\n"
            "• Choose the most complete/formal version found in the batch\n"
            "• MITCHELL is more formal than MIKE, MICHAEL, MITCH\n"
            "• JACQUELINE is more formal than JACKIE, JACKI\n"
            "• Always prefer the longest, most complete form\n"
            "\n"
            "### 6. FINAL PARSING\n"
            "Split standardized names:\n"
            "• 3+ tokens: first_name=token[0], middle_name=token[1], last_name=remaining_tokens\n"
            "• 2 tokens: first_name=token[0], middle_name='', last_name=token[1]\n"
            "• 1 token: first_name='', middle_name='', last_name=token[0]\n"
            "\n"
            "## CRITICAL REQUIREMENTS:\n"
            "• Process ALL records in the batch consistently\n"
            "• Use the batch context to resolve ambiguities\n"
            "• Map ALL variations to the same canonical form\n"
            "• Handle nicknames intelligently (MIKE→MITCHELL if MITCHELL exists in batch)\n"
            "• Output ONLY valid JSON array\n"
            "\n"
            "## EXAMPLE:\n"
            "Input batch contains: 'MIKE S', 'MITCHELL SANDERS', 'M SANDERS', 'MICHAEL S'\n"
            "Analysis: MITCHELL is the most complete form\n"
            "Result: All should become 'MITCHELL' as first_name\n"
            "\n"
            "Output format: [{\"RecID\":\"...\", \"first_name\":\"...\", \"middle_name\":\"...\", \"last_name\":\"...\"}]\n"
        )
    }
    user_message = {
        "role": "user",
        "content": json.dumps(clean_chunk, ensure_ascii=False)
    }
    resp = await client.chat.completions.create(model=model, messages=[system_message, user_message])
    return resp.choices[0].message.content


def safe_parse(raw: str) -> List[Dict]:
    """Safely parse JSON response with fallback extraction."""
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        snippet = extract_json_array(raw)
        return json.loads(snippet)


async def process_batch_with_fallback(batch_idx: int, chunk: List[Dict], total_batches: int, semaphore: asyncio.Semaphore) -> None:
    """Process a single batch with fallback model support."""
    async with semaphore:
        if RESUME_FROM_CHECKPOINTS:
            existing = load_checkpoint(batch_idx)
            if existing is not None:
                print(f"Batch {batch_idx}/{total_batches} ({len(chunk)} recs) ✓ loaded from checkpoint")
                return

        t0 = time.perf_counter()
        last_error = None

        new_records = sum(1 for r in chunk if not r.get('_is_overlap', False))
        overlap_records = len(chunk) - new_records

        for attempt in range(PRIMARY_ATTEMPTS):
            try:
                raw = await parse_chunk_async(chunk, PRIMARY_MODEL)
                data = safe_parse(raw)

                save_checkpoint(batch_idx, data)
                dt = time.perf_counter() - t0

                print(f"Batch {batch_idx}/{total_batches} ({new_records} new + {overlap_records} overlap) "
                      f"parsed in {dt:.2f}s - {PRIMARY_MODEL}")
                return

            except Exception as e:
                last_error = e
                if attempt < PRIMARY_ATTEMPTS - 1:
                    await asyncio.sleep(1)
                    continue

        print(f"Batch {batch_idx}/{total_batches} - Primary model failed, trying fallback {FALLBACK_MODEL}")
        try:
            raw = await parse_chunk_async(chunk, FALLBACK_MODEL)
            data = safe_parse(raw)

            save_checkpoint(batch_idx, data)
            dt = time.perf_counter() - t0

            print(f"Batch {batch_idx}/{total_batches} ({new_records} new + {overlap_records} overlap) "
                  f"parsed in {dt:.2f}s - {FALLBACK_MODEL}")

        except Exception as e:
            print(f"Batch {batch_idx}/{total_batches} ✗ failed with both models. Last error: {e}")


async def process_batches_async(records: List[Dict]) -> None:
    """Process records in chunks with overlap for better context at boundaries."""
    total = len(records)

    batches = list(chunk_records_with_overlap(records, CHUNK_SIZE, OVERLAP_SIZE))
    total_batches = len(batches)

    print(f"Starting concurrent batch processing: {total} records in {total_batches} batches")
    print(f"Batch size: {CHUNK_SIZE}, Overlap: {OVERLAP_SIZE} records")
    print(f"Concurrent requests: {CONCURRENT_REQUESTS}")

    semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)

    tasks = []
    for idx, batch in enumerate(batches, start=1):
        task = process_batch_with_fallback(idx, batch, total_batches, semaphore)
        tasks.append(task)

    await asyncio.gather(*tasks)


def merge_with_source_and_write(df_source: pd.DataFrame) -> str:
    """
    Consolidate all checkpoints, merge with source dataframe, and write final output CSV.
    Uses deduplication to handle overlapped records.
    """
    parsed_df = consolidate_checkpoints_with_dedup()
    if not parsed_df.empty and "RecID" in parsed_df.columns:
        parsed_df["RecID"] = parsed_df["RecID"].astype(str)
    else:
        parsed_df = pd.DataFrame(columns=["RecID", "first_name", "middle_name", "last_name"])

    out = df_source.merge(parsed_df, on="RecID", how="left")

    tmp_out = OUTPUT_CSV + ".part"
    out.to_csv(tmp_out, index=False)
    os.replace(tmp_out, OUTPUT_CSV)
    return OUTPUT_CSV


async def main():
    """Main execution function for name parsing pipeline."""
    start_total = time.perf_counter()

    df = pd.read_csv(INPUT_CSV, dtype=str).fillna("")
    df["RecID"] = df["RecID"].astype(str)
    records = df[["RecID", "Name"]].to_dict(orient="records")

    await process_batches_async(records)

    out_path = merge_with_source_and_write(df)

    total_elapsed = time.perf_counter() - start_total
    print(f"✔ Finalized → {out_path}")
    print(f"Total elapsed time: {total_elapsed:.2f} seconds")
    print(f"(If some batches are missing, re-run to resume. Existing checkpoints will be reused.)")


await main()

Starting concurrent batch processing: 10000 records in 208 batches
Batch size: 64, Overlap: 16 records
Concurrent requests: 10
Batch 9/208 (48 new + 16 overlap) parsed in 36.76s - o4-mini
Batch 7/208 (48 new + 16 overlap) parsed in 42.81s - o4-mini
Batch 3/208 (48 new + 16 overlap) parsed in 43.40s - o4-mini
Batch 4/208 (48 new + 16 overlap) parsed in 46.16s - o4-mini
Batch 10/208 (48 new + 16 overlap) parsed in 51.04s - o4-mini
Batch 2/208 (48 new + 16 overlap) parsed in 52.60s - o4-mini
Batch 5/208 (48 new + 16 overlap) parsed in 65.00s - o4-mini
Batch 6/208 (48 new + 16 overlap) parsed in 66.33s - o4-mini
Batch 8/208 (48 new + 16 overlap) parsed in 75.48s - o4-mini
Batch 11/208 (48 new + 16 overlap) parsed in 44.04s - o4-mini
Batch 1/208 (64 new + 0 overlap) parsed in 82.79s - o4-mini
Batch 13/208 (48 new + 16 overlap) parsed in 41.07s - o4-mini
Batch 15/208 (48 new + 16 overlap) parsed in 33.60s - o4-mini
Batch 14/208 (48 new + 16 overlap) parsed in 45.85s - o4-mini
Batch 12/208 (4

In [7]:
import os
import re
import sys
import json
import time
import glob
import asyncio
import hashlib
from typing import List, Dict, Optional

import pandas as pd
from openai import AsyncOpenAI

# Configuration
INPUT_CSV = "b_processed_results_LLM.csv"
OUTPUT_CSV = "c_parsed_addresses.csv"

CHUNK_SIZE = 64
CONCURRENT_REQUESTS = 10

PRIMARY_MODEL = "o4-mini"
FALLBACK_MODEL = "o3"
PRIMARY_ATTEMPTS = 3

client = AsyncOpenAI()

TMP_DIR = "tmp_parsed_address_batches"
CACHE_DIR = "api_response_cache"
RESUME_FROM_CHECKPOINTS = True

REQUIRED_KEYS = ["RecID", "house_number", "street_name", "city", "state", "zip"]


def ensure_dir(path: str) -> None:
    """Create directory if it doesn't exist."""
    if not os.path.isdir(path):
        os.makedirs(path, exist_ok=True)


def chunk_records(records: List[Dict], size: int):
    """Split records into chunks of specified size."""
    for i in range(0, len(records), size):
        yield records[i:i + size]


def extract_json_array(text: str) -> str:
    """Extract JSON array from potentially malformed response."""
    start = text.find('[')
    end = text.rfind(']')
    if start == -1 or end == -1 or start > end:
        raise ValueError("No JSON array delimiters found")
    return text[start:end + 1]


def load_checkpoint(batch_idx: int) -> Optional[List[Dict]]:
    """Load an existing checkpoint for the given batch index if present and valid."""
    path = os.path.join(TMP_DIR, f"batch_{batch_idx:05d}.json")
    if os.path.exists(path) and os.path.getsize(path) > 0:
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, list):
                if all(isinstance(x, dict) for x in data):
                    return data
        except (json.JSONDecodeError, IOError):
            return None
    return None


def save_checkpoint(batch_idx: int, data: List[Dict]) -> None:
    """Atomically write the batch results to a checkpoint file."""
    ensure_dir(TMP_DIR)
    path = os.path.join(TMP_DIR, f"batch_{batch_idx:05d}.json")
    tmp_path = path + ".part"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp_path, path)


def get_cache_key(chunk: List[Dict], model: str) -> str:
    """Generate cache key for a chunk and model combination."""
    content = json.dumps(chunk, sort_keys=True) + model
    return hashlib.md5(content.encode()).hexdigest()


def load_from_cache(cache_key: str) -> Optional[List[Dict]]:
    """Load cached API response if it exists."""
    ensure_dir(CACHE_DIR)
    cache_file = os.path.join(CACHE_DIR, f"{cache_key}.json")

    if os.path.exists(cache_file):
        try:
            with open(cache_file, 'r', encoding='utf-8') as f:
                cached_data = json.load(f)
                if isinstance(cached_data, list):
                    return cached_data
        except (json.JSONDecodeError, IOError):
            pass
    return None


def save_to_cache(cache_key: str, data: List[Dict]) -> None:
    """Save successful API response to cache."""
    ensure_dir(CACHE_DIR)
    cache_file = os.path.join(CACHE_DIR, f"{cache_key}.json")
    tmp_file = cache_file + ".tmp"

    try:
        with open(tmp_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        os.replace(tmp_file, cache_file)
    except Exception as e:
        print(f"Warning: Failed to save cache {cache_key}: {e}", file=sys.stderr)
        if os.path.exists(tmp_file):
            os.remove(tmp_file)


def consolidate_checkpoints() -> pd.DataFrame:
    """Read all batch checkpoint files and return a DataFrame."""
    ensure_dir(TMP_DIR)
    files = sorted(glob.glob(os.path.join(TMP_DIR, "batch_*.json")))
    all_rows = []
    for fp in files:
        try:
            with open(fp, "r", encoding="utf-8") as f:
                rows = json.load(f)
            if isinstance(rows, list):
                for item in rows:
                    if isinstance(item, dict):
                        all_rows.append(item)
        except Exception:
            print(f"Warning: Could not read or parse checkpoint file: {fp}", file=sys.stderr)

    if not all_rows:
        return pd.DataFrame(columns=["RecID", "house_number", "street_name", "city", "state", "zip"])
    return pd.DataFrame(all_rows, dtype=str)


SYSTEM_PROMPT = """You are an expert in US address normalization and component extraction for large batches. Your job: STANDARDIZE and SPLIT each address into exactly these fields:

- house_number
- street_name   (street name + type + directionals + unit/building info; for PO Boxes use "PO BOX")
- city
- state         (2-letter USPS)
- zip           (5 digits only)

You must:
1) Analyze the WHOLE BATCH first to find families of near-duplicate addresses (same/near street and same house number) and choose ONE consistent, most-complete canonical form within each family.
2) Fix typos, spacing, fused tokens, and abbreviations CONSISTENTLY across that family using the tables below.
3) DO NOT invent facts: if a component cannot be confidently resolved from batch evidence, leave it as "" (empty string). Never make up a ZIP or city without batch support.
4) Use uppercase for all tokens; trim all extra spaces; remove stray punctuation.

COMPONENT RULES
• house_number:
  - For street addresses: leading digits only (e.g., "621 GARDENIA ST..." → 621).
  - For PO Boxes/APARTADO/etc.: house_number is the box number (e.g., "PO BOX 150706" → 150706).
  - Keep only digits; drop non-digits. If missing or unclear, set "".
• street_name:
  - Include street name + street type (USPS abbreviations) + directionals + unit/building info (APT/BLDG/UNIT/STE/LOT).
  - For PO boxes, set street_name = "PO BOX".
  - For numbered streets, keep ordinals as given (e.g., 118TH).
• city:
  - Use the most frequent correct spelling from the batch family (e.g., RELDDING → REDDING).
• state:
  - 2-letter USPS only. Correct common errors using the table below (TEAS/TEXA/TEXPAS → TX; CALIF./CALI/CALIFORNIA → CA; FLE/FLE./FLORIDYA → FL; ARK./ARKANSAS → AR).
• zip:
  - 5 digits only. If you see a 4-digit or corrupted ZIP, replace it ONLY when the same address family in the batch clearly shows the correct 5-digit ZIP. Otherwise set "".
  - Strip any letter prefixes/suffixes (e.g., H33025 → 33025).
  - Do NOT add ZIP+4.

NORMALIZATION TABLES
• Directionals → abbreviate:
  NORTH→N, SOUTH→S, EAST→E, WEST→W, NORTHEAST→NE, NORTHWEST→NW, SOUTHEAST→SE, SOUTHWEST→SW
• Street types → USPS abbreviations (examples):
  AVENUE/AVEN/AVENU/AVN/AOVE → AVE
  STREET/ST/STR/STRT → ST
  DRIVE/DRIV/DRV/DR → DR
  LANE/LAN/LN → LN
  ROAD/ROD/RD → RD
  BOULEVARD/BLVD → BLVD
  CIRCLE/CIRC/CRCL → CIR
  COURT/CRT/CT → CT
  TERRACE/TERR/TERRACE → TER
  PLACE/PLC/PL → PL
  PARKWAY/PKWAY/PKWY → PKWY
  HIGHWAY/HWY → HWY
  WAY → WAY
• Common PO BOX terms → normalize to "PO BOX":
  APARTADO/APTDO/BUZON/BÚZON/POST OFFICE/POST OFFICE BOX/POST BOX/OFFICE BOX/CALLER → PO BOX
  (e.g., "3127 APARTADO..." → "PO BOX 3127", "150706 CALLER..." → "PO BOX 150706")
• Units/Buildings (append to street_name):
  APT/AP/APTO/APT./UNIT/STE/SUITE/BLDG/BUILDING/LOT/# → normalize to APT/UNIT/STE/BLDG/LOT/# (use APT for AP)
• State cleanup → 2-letter:
  CALI/CALIF./CALIFORNIA → CA
  FLA/FLOR/FLE/FLE./FLORIDA/FLORIDYA → FL
  TEXAS/TEXA/TEAS/TEXPAS → TX
  ARK./ARKANSAS → AR
  CALIF → CA
  C → CA only if clearly used as a truncated state in a family already resolved to CA
• City typos → fix using batch frequency:
  LOS ANGELGES → LOS ANGELES
  MRAMAR → MIRAMAR
  HOUSTEON → HOUSTON
  RELDDING/REDDIN → REDDING
  MCLLEN/MCALLEN → MCALLEN
  WELLINGJTON/WABURWTON/WBURTON → WELLINGTON (use most common family spelling)
  LKEWOOD/YLAKEWOOD → LAKEWOOD
  DESOTO (keep DESOTO; do not split)

FUSED & NOISE TOKENS
• Insert missing spaces between numbers/letters (e.g., 861SOUTH → 861 SOUTH; 10THAVENU → 10TH AVENUE).
• Remove stray single-letter tokens near city/state (e.g., "ELK GROVE B CA" → "ELK GROVE CA").
• Remove leading garbage around ZIP (e.g., H33025 → 33025).
• Drop trailing state words after the 2-letter state (e.g., "... TX TEXAS" → TX).

ZIP CORRECTION POLICY (STRICT)
• If multiple records in the SAME BATCH FAMILY show the same address and a proper 5-digit ZIP, use that ZIP for all members.
• If the zip is corupted by characters, then strip the zip of charcters and output only the digits
• If no clear consensus exists, set zip = "" rather than guessing.

OUTPUT FORMAT (STRICT)
• Return ONLY a JSON object with a single key "records" containing the array of outputs.
• Each object in "records" MUST contain exactly: RecID, house_number, street_name, city, state, zip.
• Uppercase all text fields; zip must be 5 digits or "".

QUALITY CHECKS BEFORE RETURN
1) For any PO BOX: house_number must be digits of the box; street_name must be "PO BOX".
2) Ensure state is 2 letters; if not resolvable, set "".
3) Ensure no hallucinated components (no invented cities/ZIPs not supported by batch evidence)."""

USER_PREFIX = """Process this batch of address records. Use collective batch context to standardize all addresses. Apply the SYSTEM rules and tables strictly.

FOCUSED MICRO-EXAMPLES (covering typical failures):

1) PO BOX variants
Input:
  "3127 APARTADO MISSION TX 78573"
  "52840 BUZON MCALLEN TEXAS 78505"
  "881605 POST OFFICE LOS ANGELES CA 90009"
  "150706 CALLER CAPE CORAL FL 33915"
  "150706 OFFICE BOX CAPE CORAL FL 33915"
Output:
  [{"RecID":"X","house_number":"3127","street_name":"PO BOX","city":"MISSION","state":"TX","zip":"78573"},
   {"RecID":"X","house_number":"52840","street_name":"PO BOX","city":"MCALLEN","state":"TX","zip":"78505"},
   {"RecID":"X","house_number":"881605","street_name":"PO BOX","city":"LOS ANGELES","state":"CA","zip":"90009"},
   {"RecID":"X","house_number":"150706","street_name":"PO BOX","city":"CAPE CORAL","state":"FL","zip":"33915"},
   {"RecID":"X","house_number":"150706","street_name":"PO BOX","city":"CAPE CORAL","state":"FL","zip":"33915"}]

2) State typos and street types
Input:
  "621 GARDENIA ST DESOTO TEAS 75115"
  "21122 JADE BLUFF LANE KATY TEAS 77450"
  "6590 ELMHURST DRV TUJUNGA CALIF. 91042"
Output:
  [{"RecID":"X","house_number":"621","street_name":"GARDENIA ST","city":"DESOTO","state":"TX","zip":"75115"},
   {"RecID":"X","house_number":"21122","street_name":"JADE BLUFF LN","city":"KATY","state":"TX","zip":"77450"},
   {"RecID":"X","house_number":"6590","street_name":"ELMHURST DR","city":"TUJUNGA","state":"CA","zip":"91042"}]

3) Fused tokens and consensus ZIP
Input (same family appears multiple times):
  "861 SOUTH ST REDDING CALIFORNIA 9600"
  "861SOUTH ST REDDING CA 96001"
  "861 SOUTH STRT REDDING CA 96001"
  "861 SOZUTH STRT REDDING CA 96001"
Output (use family consensus ZIP 96001; fix fused and street type):
  [{"RecID":"X","house_number":"861","street_name":"SOUTH ST","city":"REDDING","state":"CA","zip":"96001"},
   {"RecID":"X","house_number":"861","street_name":"SOUTH ST","city":"REDDING","state":"CA","zip":"96001"},
   {"RecID":"X","house_number":"861","street_name":"SOUTH ST","city":"REDDING","state":"CA","zip":"96001"},
   {"RecID":"X","house_number":"861","street_name":"SOUTH ST","city":"REDDING","state":"CA","zip":"96001"}]

4) Units and AP→APT
Input:
  "301 PINE FOREST DR APT 20 MAUMELLE AR 72113"
  "7250 FRANKLIN AVE AP 609 LOS ANGELGES CALIF. 90046"
Output:
  [{"RecID":"X","house_number":"301","street_name":"PINE FOREST DR APT 20","city":"MAUMELLE","state":"AR","zip":"72113"},
   {"RecID":"X","house_number":"7250","street_name":"FRANKLIN AVE APT 609","city":"LOS ANGELES","state":"CA","zip":"90046"}]

5) Street-family canonicalization
Input (same family; correct typos and keep most complete form):
  "12438 JASMINE BROOK LN HOUSTON TX 77089"
  "12438 JASMINE BQROOK LN HOUSTON TX 77089"
  "12438 JASMINE BROOKX LN HOUSTEON TX 77089"
  "12438 JASMINE BROOK LANE HOUSTON TX 77089"
Output (canonical "JASMINE BROOK LN", city from consensus):
  [{"RecID":"X","house_number":"12438","street_name":"JASMINE BROOK LN","city":"HOUSTON","state":"TX","zip":"77089"},
   {"RecID":"X","house_number":"12438","street_name":"JASMINE BROOK LN","city":"HOUSTON","state":"TX","zip":"77089"},
   {"RecID":"X","house_number":"12438","street_name":"JASMINE BROOK LN","city":"HOUSTON","state":"TX","zip":"77089"},
   {"RecID":"X","house_number":"12438","street_name":"JASMINE BROOK LN","city":"HOUSTON","state":"TX","zip":"77089"}]

NOW PROCESS THIS BATCH USING THE SAME RULES:

"""

USER_SUFFIX = """

Return ONLY a JSON object with a single key "records" that maps to the array. Each object must have: RecID, house_number, street_name, city, state, zip. Uppercase all strings; zip is 5 digits or ""."""


async def parse_chunk_async(chunk: List[Dict], model: str, use_streaming: bool = True) -> str:
    """Calls the model with enhanced address normalization prompt asynchronously with optional streaming."""
    cache_key = get_cache_key(chunk, model)
    cached_result = load_from_cache(cache_key)
    if cached_result is not None:
        print(f"  → Cache hit for chunk (model: {model})")
        return json.dumps({"records": cached_result})

    system_message = {"role": "system", "content": SYSTEM_PROMPT}
    user_message = {
        "role": "user",
        "content": USER_PREFIX + json.dumps(chunk, ensure_ascii=False) + USER_SUFFIX,
    }

    response_format = {
        "type": "json_schema",
        "json_schema": {
            "name": "AddressBatch",
            "schema": {
                "type": "object",
                "properties": {
                    "records": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "RecID": {"type": "string"},
                                "house_number": {"type": "string"},
                                "street_name": {"type": "string"},
                                "city": {"type": "string"},
                                "state": {"type": "string"},
                                "zip": {"type": "string"},
                            },
                            "required": ["RecID", "house_number", "street_name", "city", "state", "zip"],
                            "additionalProperties": False,
                        },
                    }
                },
                "required": ["records"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }

    if use_streaming:
        stream = await client.chat.completions.create(
            model=model,
            messages=[system_message, user_message],
            response_format=response_format,
            temperature=1,
            stream=True,
        )

        content_chunks = []
        async for chunk_response in stream:
            if chunk_response.choices[0].delta.content:
                content_chunks.append(chunk_response.choices[0].delta.content)

        result = ''.join(content_chunks)
    else:
        resp = await client.chat.completions.create(
            model=model,
            messages=[system_message, user_message],
            response_format=response_format,
            temperature=1,
        )
        result = resp.choices[0].message.content

    try:
        parsed = safe_parse(result)
        if parsed:
            save_to_cache(cache_key, parsed)
    except:
        pass

    return result


async def smart_retry_on_mismatch(chunk: List[Dict], parsed: List[Dict], model: str, attempt: int) -> List[Dict]:
    """If we got fewer records, try to process missing ones separately."""
    if len(parsed) == len(chunk):
        return parsed

    if len(parsed) < len(chunk):
        parsed_ids = {r.get('RecID', '') for r in parsed if r.get('RecID')}
        chunk_ids = {r['RecID'] for r in chunk}
        missing_ids = chunk_ids - parsed_ids

        if missing_ids:
            missing_records = [r for r in chunk if r['RecID'] in missing_ids]
            print(f"  → Smart retry: Processing {len(missing_records)} missing records separately (attempt {attempt})")

            if len(missing_records) > 10:
                sub_chunks = list(chunk_records(missing_records, 10))
                all_missing_parsed = []

                for sub_chunk in sub_chunks:
                    try:
                        raw = await parse_chunk_async(sub_chunk, model, use_streaming=False)
                        sub_parsed = safe_parse(raw)
                        ok, _ = validate_batch_output(sub_parsed, expected_len=len(sub_chunk))
                        if ok:
                            all_missing_parsed.extend(sub_parsed)
                    except Exception as e:
                        print(f"    → Sub-chunk failed: {e}", file=sys.stderr)

                if all_missing_parsed:
                    return parsed + all_missing_parsed
            else:
                try:
                    raw = await parse_chunk_async(missing_records, model, use_streaming=False)
                    missing_parsed = safe_parse(raw)
                    ok, _ = validate_batch_output(missing_parsed, expected_len=len(missing_records))
                    if ok:
                        return parsed + missing_parsed
                except Exception as e:
                    print(f"  → Smart retry failed: {e}", file=sys.stderr)

    elif len(parsed) > len(chunk):
        chunk_ids = {r['RecID'] for r in chunk}
        filtered = [r for r in parsed if r.get('RecID') in chunk_ids]
        if len(filtered) == len(chunk):
            print(f"  → Filtered extra records: kept {len(filtered)} of {len(parsed)}")
            return filtered

    return parsed


def safe_parse(raw_json_string: str) -> List[Dict]:
    """
    Safely parses a JSON string by trying multiple strategies.
    Does NOT modify values (no normalization here).
    """
    try:
        data = json.loads(raw_json_string)
        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            for value in data.values():
                if isinstance(value, list):
                    return value
    except json.JSONDecodeError:
        match = re.search(r"```json\s*(\[.*?\])\s*```", raw_json_string, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1))
            except json.JSONDecodeError:
                pass
        try:
            json_array_str = extract_json_array(raw_json_string)
            return json.loads(json_array_str)
        except (ValueError, json.JSONDecodeError) as e:
            raise ValueError("Failed to find or parse a valid JSON array in the response.") from e

    raise ValueError("No JSON list found in the parsed object.")


def validate_batch_output(parsed: List[Dict], expected_len: int) -> (bool, str):
    """Minimal structural validation only (no rule-based normalization)."""
    if not isinstance(parsed, list):
        return False, "Model output is not a list"
    if len(parsed) != expected_len:
        return False, f"Length mismatch (got {len(parsed)}, expected {expected_len})"
    for i, o in enumerate(parsed):
        if not isinstance(o, dict):
            return False, f"Item {i} is not an object"
        for k in REQUIRED_KEYS:
            if k not in o:
                return False, f"Item {i} missing key: {k}"
            if not isinstance(o[k], str):
                return False, f"Item {i} key {k} is not a string"
    return True, ""


async def run_with_fallback(chunk: List[Dict], batch_idx: int, total_batches: int) -> List[Dict]:
    """Try primary model with retries, then fallback to secondary model."""
    last_err = ""
    for attempt in range(1, PRIMARY_ATTEMPTS + 1):
        try:
            raw = await parse_chunk_async(chunk, model=PRIMARY_MODEL)
            parsed = safe_parse(raw)

            ok, reason = validate_batch_output(parsed, expected_len=len(chunk))

            if not ok and "Length mismatch" in reason:
                parsed = await smart_retry_on_mismatch(chunk, parsed, PRIMARY_MODEL, attempt)
                ok, reason = validate_batch_output(parsed, expected_len=len(chunk))

            if ok:
                if attempt > 1:
                    print(f"Batch {batch_idx:04d}/{total_batches} ✓ primary recovered on attempt {attempt}")
                return parsed

            last_err = f"validation failed: {reason}"
            print(f"Batch {batch_idx:04d} primary attempt {attempt} failed: {reason}", file=sys.stderr)
        except Exception as e:
            last_err = str(e)
            print(f"Batch {batch_idx:04d} primary attempt {attempt} exception: {e}", file=sys.stderr)
        if attempt < PRIMARY_ATTEMPTS:
            await asyncio.sleep(2 ** (attempt - 1))

    print(f"Batch {batch_idx:04d}/{total_batches} → falling back to {FALLBACK_MODEL} after primary failures ({last_err})")
    raw = await parse_chunk_async(chunk, model=FALLBACK_MODEL, use_streaming=False)
    parsed = safe_parse(raw)

    ok, reason = validate_batch_output(parsed, expected_len=len(chunk))
    if not ok and "Length mismatch" in reason:
        parsed = await smart_retry_on_mismatch(chunk, parsed, FALLBACK_MODEL, 1)
        ok, reason = validate_batch_output(parsed, expected_len=len(chunk))

    if ok:
        print(f"Batch {batch_idx:04d}/{total_batches} ✓ parsed via fallback {FALLBACK_MODEL}")
        return parsed

    raise ValueError(f"Fallback {FALLBACK_MODEL} validation failed: {reason}")


async def process_one_batch(idx: int, chunk: List[Dict], total_batches: int, semaphore: asyncio.Semaphore):
    """Processes a single batch asynchronously, with primary→fallback model logic."""
    async with semaphore:
        if RESUME_FROM_CHECKPOINTS and load_checkpoint(idx) is not None:
            print(f"Batch {idx:04d}/{total_batches} ✓ loaded from checkpoint.")
            return

        t0 = time.perf_counter()
        try:
            parsed_data = await run_with_fallback(chunk, batch_idx=idx, total_batches=total_batches)
            save_checkpoint(idx, parsed_data)
            dt = time.perf_counter() - t0
            print(f"Batch {idx:04d}/{total_batches} ({len(chunk)} recs) ✓ parsed in {dt:.2f}s")
            return
        except Exception as e:
            print(f"Batch {idx:04d}/{total_batches} ✗ failed after primary + fallback: {e}", file=sys.stderr)
            print(f"--- Failing Data for Batch {idx:04d} ---", file=sys.stderr)
            print(json.dumps(chunk, indent=2), file=sys.stderr)
            print(f"--- End of Failing Data ---", file=sys.stderr)


async def process_batches_async(records: List[Dict]):
    """Creates and runs all batch processing tasks concurrently."""
    semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)
    batches = list(chunk_records(records, CHUNK_SIZE))
    total_batches = len(batches)
    print(f"Starting async address processing: {len(records)} records in {total_batches} batches.")
    print(f"Cache directory: {CACHE_DIR}")

    tasks = [
        process_one_batch(idx, chunk, total_batches, semaphore)
        for idx, chunk in enumerate(batches, start=1)
    ]
    await asyncio.gather(*tasks)


def merge_with_source_and_write(df_source: pd.DataFrame) -> str:
    """Consolidates checkpoints, merges with the source, and writes the final CSV."""
    parsed_df = consolidate_checkpoints()
    if not parsed_df.empty and "RecID" in parsed_df.columns:
        parsed_df["RecID"] = parsed_df["RecID"].astype(str)
    else:
        parsed_df = pd.DataFrame(columns=["RecID", "house_number", "street_name", "city", "state", "zip"])

    out_df = df_source.merge(parsed_df, on="RecID", how="left")

    for col in ["house_number", "street_name", "city", "state", "zip"]:
        if col not in out_df.columns:
            out_df[col] = ""
    out_df.fillna("", inplace=True)

    tmp_out = OUTPUT_CSV + ".part"
    out_df.to_csv(tmp_out, index=False)
    os.replace(tmp_out, OUTPUT_CSV)
    return OUTPUT_CSV


async def main():
    """Main execution function for address parsing pipeline."""
    start_total = time.perf_counter()

    if not os.path.exists(INPUT_CSV):
        print(f"Error: Input file not found at '{INPUT_CSV}'", file=sys.stderr)
        return

    df = pd.read_csv(INPUT_CSV, dtype=str).fillna("")
    if "RecID" not in df.columns or "Address" not in df.columns:
        print("Error: INPUT_CSV must have columns 'RecID' and 'Address'.", file=sys.stderr)
        return
    df["RecID"] = df["RecID"].astype(str)

    records = df[["RecID", "Address"]].to_dict(orient="records")

    await process_batches_async(records)

    out_path = merge_with_source_and_write(df)

    total_elapsed = time.perf_counter() - start_total
    print("\n--------------------------------------------------")
    print(f"✔ Finalized → {out_path}")
    print(f"Total elapsed time: {total_elapsed:.2f} seconds")
    print("(Re-run the script to resume processing any failed or missing batches.)")


await main()

Starting async address processing: 2000 records in 32 batches.
Cache directory: api_response_cache
Batch 0005/32 (64 recs) ✓ parsed in 59.55s
Batch 0001/32 (64 recs) ✓ parsed in 60.97s
Batch 0004/32 (64 recs) ✓ parsed in 64.46s
Batch 0003/32 (64 recs) ✓ parsed in 65.25s
Batch 0006/32 (64 recs) ✓ parsed in 78.14s
Batch 0007/32 (64 recs) ✓ parsed in 84.85s
Batch 0008/32 (64 recs) ✓ parsed in 87.31s
Batch 0010/32 (64 recs) ✓ parsed in 91.13s
Batch 0002/32 (64 recs) ✓ parsed in 92.98s
Batch 0012/32 (64 recs) ✓ parsed in 71.74s
Batch 0009/32 (64 recs) ✓ parsed in 134.04s
Batch 0014/32 (64 recs) ✓ parsed in 74.32s
Batch 0015/32 (64 recs) ✓ parsed in 71.06s
Batch 0017/32 (64 recs) ✓ parsed in 71.52s
Batch 0013/32 (64 recs) ✓ parsed in 97.70s
Batch 0016/32 (64 recs) ✓ parsed in 80.49s
Batch 0019/32 (64 recs) ✓ parsed in 77.45s
Batch 0011/32 (64 recs) ✓ parsed in 133.96s
Batch 0022/32 (64 recs) ✓ parsed in 64.84s
Batch 0021/32 (64 recs) ✓ parsed in 72.79s
Batch 0023/32 (64 recs) ✓ parsed in 64.

In [8]:
import time
import sys
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from sklearn.cluster import DBSCAN

warnings.filterwarnings("ignore", message=".*force_all_finite.*", category=FutureWarning)


def simple_normalize(text):
    """Normalize text to lowercase and strip whitespace."""
    if pd.isnull(text):
        return ""
    return str(text).lower().strip()


def l2_normalize_matrix(mat: np.ndarray) -> np.ndarray:
    """Apply L2 normalization to matrix rows."""
    return normalize(mat, norm="l2", axis=1, copy=False)


def recid_key(s: str):
    """
    Generate numeric key for sorting RecIDs like '88.10' > '88.2' correctly.
    Returns (major, minor) as integers; non-numeric parts become 0.
    """
    s = str(s)
    if "." in s:
        a, b = s.split(".", 1)
    else:
        a, b = s, "0"

    def to_int(x):
        try:
            return int(x)
        except:
            digits = "".join(ch for ch in x if ch.isdigit())
            return int(digits) if digits else 0

    return (to_int(a), to_int(b))


def detect_duplicate_boundary(features: np.ndarray, n_samples: int = 20) -> dict:
    """
    Sample random records and find the gap between duplicates and non-duplicates.
    Returns epsilon values at p85, p90, p95 percentiles.
    """
    np.random.seed(42)
    n_records = features.shape[0]
    sample_size = min(n_samples, n_records)

    nn = NearestNeighbors(n_neighbors=min(11, n_records), metric="cosine")
    nn.fit(features)

    sample_indices = np.random.choice(n_records, sample_size, replace=False)
    close_distances = []

    for idx in sample_indices:
        distances = nn.kneighbors([features[idx]], n_neighbors=11)[0][0][1:]

        gaps = np.diff(distances)
        for i, gap in enumerate(gaps):
            if gap > distances[i] * 0.5 and gap > 0.05:
                close_distances.extend(distances[:i+1])
                break
        else:
            threshold = min(distances[0] * 2, 0.15)
            close_distances.extend(distances[distances <= threshold])

    if not close_distances:
        return None

    close_distances = [d for d in close_distances if d > 1e-9]
    if not close_distances:
        return None

    return {
        85: float(np.percentile(close_distances, 85)),
        90: float(np.percentile(close_distances, 90)),
        95: float(np.percentile(close_distances, 95))
    }


def cluster_embeddings_dbscan(df: pd.DataFrame,
                              eps: float,
                              min_samples: int = 1,
                              metric: str = "cosine") -> pd.DataFrame:
    """
    Perform DBSCAN clustering on all columns except 'RecID'.
    Sort RecIDs within each cluster ascending (numeric).
    Remap cluster IDs so clusters are ordered by the smallest RecID.

    Returns:
        DataFrame with ClusterID and RecIDs columns
    """
    features = df.loc[:, df.columns != "RecID"].values
    rec_ids = df["RecID"].astype(str).reset_index(drop=True)

    db = DBSCAN(eps=float(eps), min_samples=int(min_samples), metric=metric)
    labels = db.fit_predict(features)

    df_lab = pd.DataFrame({"RecID": rec_ids, "cluster_label": labels.astype(int)})

    groups = {}
    for cid, grp in tqdm(df_lab.groupby("cluster_label"), desc="Sorting cluster members", unit="cluster"):
        items = sorted(grp["RecID"].tolist(), key=recid_key)
        groups[int(cid)] = items

    ordered = sorted(groups.items(), key=lambda kv: recid_key(kv[1][0] if kv[1] else "9999999"))
    rows = [{"ClusterID": new_id, "RecIDs": ";".join(items)} for new_id, (_, items) in enumerate(ordered)]
    return pd.DataFrame(rows, columns=["ClusterID", "RecIDs"])


def main():
    """Main execution function for DBSCAN clustering pipeline."""
    start_time = time.time()

    print("Loading parsed names and addresses...")
    names_df = pd.read_csv("c_parsed_names.csv", dtype=str).fillna("")
    names_df["Name"] = names_df[["first_name", "middle_name", "last_name"]].apply(
        lambda parts: " ".join(p for p in parts if p), axis=1
    )

    addr_df = pd.read_csv("c_parsed_addresses.csv", dtype=str).fillna("")
    addr_df["Address"] = addr_df[["house_number", "street_name", "city", "state", "zip"]].apply(
        lambda parts: " ".join(p for p in parts if p), axis=1
    )

    df = pd.merge(
        names_df[["RecID", "Name"]],
        addr_df[["RecID", "Address"]],
        on="RecID",
        how="inner"
    )

    print("Normalizing Name and Address...")
    df["Name_clean"] = df["Name"].apply(simple_normalize)
    df["Address_clean"] = df["Address"].apply(simple_normalize)

    print("Loading embedding model (BAAI/bge-M3)...")
    model = SentenceTransformer("BAAI/bge-M3")

    print("Computing name embeddings...")
    name_embs = model.encode(df["Name_clean"].tolist(), show_progress_bar=True)
    name_embs = l2_normalize_matrix(np.array(name_embs))
    name_emb_df = pd.DataFrame(name_embs, columns=[f"dim_{i}" for i in range(name_embs.shape[1])])
    name_emb_df.insert(0, "RecID", df["RecID"])
    name_emb_df.to_csv("d_bge_M3_name_embeddings.csv", index=False)
    print("→ Saved name embeddings to 'd_bge_M3_name_embeddings.csv'")

    print("Computing address embeddings...")
    addr_embs = model.encode(df["Address_clean"].tolist(), show_progress_bar=True)
    addr_embs = l2_normalize_matrix(np.array(addr_embs))
    addr_emb_df = pd.DataFrame(addr_embs, columns=[f"dim_{i}" for i in range(addr_embs.shape[1])])
    addr_emb_df.insert(0, "RecID", df["RecID"])
    addr_emb_df.to_csv("d_bge_M3_address_embeddings.csv", index=False)
    print("→ Saved address embeddings to 'd_bge_M3_address_embeddings.csv'")

    print("\nCombining embeddings for joint clustering...")
    combined = pd.merge(name_emb_df, addr_emb_df, on="RecID", suffixes=("_name", "_addr"))
    combined_feats = combined.loc[:, combined.columns != "RecID"].values
    combined_feats = l2_normalize_matrix(combined_feats)
    combined.to_csv("d_combined_embeddings.csv", index=False)
    print("→ Combined embeddings saved to 'd_combined_embeddings.csv'")

    standard_eps = {"name": 0.15, "address": 0.15, "combined": 0.08}

    print("\nDetecting epsilon values from duplicate boundaries...")
    suggestions = {}

    for label, features, standard in [
        ("name", name_embs, standard_eps["name"]),
        ("address", addr_embs, standard_eps["address"]),
        ("combined", combined_feats, standard_eps["combined"])
    ]:
        detected = detect_duplicate_boundary(features)
        if detected:
            suggestions[label] = detected
            print(f"  {label:8s}: p85={detected[85]:.4f}, p90={detected[90]:.4f}, p95={detected[95]:.4f}")
        else:
            suggestions[label] = {85: standard, 90: standard, 95: standard}
            print(f"  {label:8s}: Could not detect boundary. Using standard value: {standard:.4f}")

    print("\nChoose epsilon values:")
    print("  [1] p85 - Conservative (fewer false positives)")
    print("  [2] p90 - Balanced (recommended)")
    print("  [3] p95 - Aggressive (more matches)")
    print("  [4] Use standard values (0.15, 0.15, 0.08)")

    choice = input("Selection (1-4) [2]: ").strip() or "2"

    if choice == "1":
        percentile = 85
    elif choice == "2":
        percentile = 90
    elif choice == "3":
        percentile = 95
    else:
        percentile = None

    if percentile:
        chosen_eps = {
            'name': suggestions['name'][percentile],
            'address': suggestions['address'][percentile],
            'combined': suggestions['combined'][percentile]
        }
    else:
        chosen_eps = standard_eps.copy()

    print("\nDBSCAN parameters summary:")
    print(f"  min_samples = 1 (singleton-friendly)")
    print(f"  metric      = 'cosine'")
    print(f"  eps (name)     = {chosen_eps['name']:.4f}")
    print(f"  eps (address)  = {chosen_eps['address']:.4f}")
    print(f"  eps (combined) = {chosen_eps['combined']:.4f}")

    print("\nClustering name embeddings with DBSCAN...")
    name_clusters = cluster_embeddings_dbscan(
        name_emb_df.copy(),
        eps=chosen_eps["name"],
        min_samples=1,
        metric="cosine"
    )
    name_clusters.to_csv("d1_name_clusters.csv", index=False)
    print("→ Name clusters saved to 'd1_name_clusters.csv'")

    print("Clustering address embeddings with DBSCAN...")
    address_clusters = cluster_embeddings_dbscan(
        addr_emb_df.copy(),
        eps=chosen_eps["address"],
        min_samples=1,
        metric="cosine"
    )
    address_clusters.to_csv("d1_address_clusters.csv", index=False)
    print("→ Address clusters saved to 'd1_address_clusters.csv'")

    print("Clustering combined embeddings with DBSCAN...")
    combined_clusters = cluster_embeddings_dbscan(
        combined.copy(),
        eps=chosen_eps["combined"],
        min_samples=1,
        metric="cosine"
    )
    combined_clusters.to_csv("d1_combined_clusters.csv", index=False)
    print("→ Combined clusters saved to 'd1_combined_clusters.csv'")

    elapsed = time.time() - start_time
    print(f"\nTotal processing time: {elapsed:.2f} seconds.")


if __name__ == "__main__":
    main()

Loading parsed names and addresses...
Normalizing Name and Address...
Loading embedding model (BAAI/bge-M3)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

→ Name clusters saved to 'd1_name_clusters.csv'
Clustering address embeddings with DBSCAN...



Sorting cluster members: 100%|██████████| 638/638 [00:00<00:00, 20490.73cluster/s]


→ Address clusters saved to 'd1_address_clusters.csv'
Clustering combined embeddings with DBSCAN...


Sorting cluster members: 100%|██████████| 995/995 [00:00<00:00, 20162.68cluster/s]

→ Combined clusters saved to 'd1_combined_clusters.csv'

Total processing time: 61.83 seconds.


In [9]:
#Step 5 : Ask the LLM to choose the best name and best address (Optimized with Ordered Output)
import sys
import os
import time
import csv
import asyncio
from typing import List, Dict, Tuple, Optional
from collections import Counter

import pandas as pd
from tqdm import tqdm
from openai import AsyncOpenAI

# ──────────────────────────── Configuration ────────────────────────────
# NOTE: for safety, we do not hardcode a default API key.
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))
if not client.api_key:
    sys.stderr.write("ERROR: OPENAI_API_KEY is not set in environment.\n")
    sys.exit(1)

# Fixed inputs (no resolver / no alternatives)
NAME_CLUSTERS = "d1_name_clusters.csv"
PARSED_NAMES  = "c_parsed_names.csv"

ADDRESS_CLUSTERS = "d1_address_clusters.csv"
PARSED_ADDR      = "c_parsed_addresses.csv"

CSV_SEP = ","
MODEL_NAME = "gpt-4o-mini"

# Outputs
NAMES_OUT_CSV     = "e_cluster_complete_names.csv"
ADDRESSES_OUT_CSV = "e_cluster_complete_addresses.csv"


# ──────────────────────────── Helpers ────────────────────────────
def atomic_write_csv(df: pd.DataFrame, path: str, sep: str = CSV_SEP) -> None:
    tmp = path + ".part"
    df.to_csv(tmp, index=False, sep=sep, quoting=csv.QUOTE_ALL)
    os.replace(tmp, path)


def dict_dedupe(seq: List[str]) -> List[str]:
    """Order-preserving dedupe."""
    return list(dict.fromkeys(seq))


def normalize_for_comparison(text: str) -> str:
    """Normalize text for comparison (remove extra spaces, standardize case)."""
    return " ".join(text.strip().upper().split())


def get_min_recid(rec_ids_str: str) -> int:
    """Extract the minimum RecID from a semicolon-separated string of RecIDs."""
    rec_ids = [r.strip() for r in str(rec_ids_str).split(";") if r.strip()]
    numeric_ids = []
    for rid in rec_ids:
        try:
            # Handle different RecID formats (e.g., "R123", "123", etc.)
            # Extract numeric portion
            numeric_part = ''.join(filter(str.isdigit, rid))
            if numeric_part:
                numeric_ids.append(int(numeric_part))
        except (ValueError, AttributeError):
            continue
    return min(numeric_ids) if numeric_ids else float('inf')


def sort_and_reassign_cluster_ids(df: pd.DataFrame) -> pd.DataFrame:
    """
    Sort dataframe by minimum RecID in each cluster and reassign ClusterIDs.
    """
    if df.empty:
        return df

    # Calculate minimum RecID for each cluster
    df['MinRecID'] = df['RecIDs'].apply(get_min_recid)

    # Sort by minimum RecID
    df_sorted = df.sort_values('MinRecID').reset_index(drop=True)

    # Reassign ClusterIDs based on sorted order
    df_sorted['ClusterID'] = range(1, len(df_sorted) + 1)
    df_sorted['ClusterID'] = 'C' + df_sorted['ClusterID'].astype(str).str.zfill(6)

    # Drop the temporary MinRecID column
    df_sorted = df_sorted.drop(columns=['MinRecID'])

    return df_sorted


# ──────────────────────────── Smart Selection Logic ────────────────────────────
def smart_select_name(candidates: List[str], raw_names: List[str]) -> Tuple[str, str]:
    """
    Smart name selection with multiple fallback strategies before LLM.
    Returns (chosen_name, selection_method)
    """
    if not candidates:
        if raw_names:
            chosen = max(raw_names, key=len)
            return chosen, "fallback_raw_longest"
        return "", "no_candidates"

    if len(candidates) == 1:
        return candidates[0], "single_candidate"

    # Normalize candidates for comparison
    normalized_candidates = [normalize_for_comparison(c) for c in candidates]

    # Check if all normalized candidates are identical
    if len(set(normalized_candidates)) == 1:
        # Return the longest original version (preserves better formatting)
        chosen = max(candidates, key=len)
        return chosen, "identical_normalized"

    # Check for clear "most complete" name (contains all parts of others)
    complete_candidate = find_most_complete_name(candidates)
    if complete_candidate:
        return complete_candidate, "most_complete"

    # Check for majority consensus (if one name appears multiple times)
    name_counts = Counter(normalized_candidates)
    if name_counts.most_common(1)[0][1] > 1:
        most_common_normalized = name_counts.most_common(1)[0][0]
        # Find original version of most common
        for orig, norm in zip(candidates, normalized_candidates):
            if norm == most_common_normalized:
                return orig, "majority_consensus"

    # Need LLM decision
    return "", "needs_llm"


def find_most_complete_name(candidates: List[str]) -> Optional[str]:
    """
    Find a name that contains all parts of other names (is most complete).
    Returns None if no clear winner.
    """
    def name_parts(name: str) -> set:
        return set(name.strip().upper().split())

    candidates_parts = [(c, name_parts(c)) for c in candidates]

    for candidate, parts in candidates_parts:
        # Check if this candidate contains all parts of all other candidates
        is_most_complete = True
        for other_candidate, other_parts in candidates_parts:
            if candidate != other_candidate and not other_parts.issubset(parts):
                is_most_complete = False
                break

        if is_most_complete and len(parts) > 1:  # Ensure it's not just a single part
            return candidate

    return None


def smart_select_address(candidates: List[str], raw_variants: List[str]) -> Tuple[str, str]:
    """
    Smart address selection with multiple fallback strategies before LLM.
    Returns (chosen_address, selection_method)
    """
    if not candidates:
        if raw_variants:
            chosen = max(raw_variants, key=len).upper()
            return chosen, "fallback_raw_longest"
        return "", "no_candidates"

    if len(candidates) == 1:
        return candidates[0].upper(), "single_candidate"

    # Normalize candidates for comparison
    normalized_candidates = [normalize_for_comparison(c) for c in candidates]

    # Check if all normalized candidates are identical
    if len(set(normalized_candidates)) == 1:
        chosen = max(candidates, key=len).upper()
        return chosen, "identical_normalized"

    # Check for most complete address (contains most components)
    complete_candidate = find_most_complete_address(candidates)
    if complete_candidate:
        return complete_candidate.upper(), "most_complete"

    # Check for majority consensus
    addr_counts = Counter(normalized_candidates)
    if addr_counts.most_common(1)[0][1] > 1:
        most_common_normalized = addr_counts.most_common(1)[0][0]
        for orig, norm in zip(candidates, normalized_candidates):
            if norm == most_common_normalized:
                return orig.upper(), "majority_consensus"

    # Need LLM decision
    return "", "needs_llm"


def find_most_complete_address(candidates: List[str]) -> Optional[str]:
    """
    Find address with most components (house number, street, city, state, zip).
    """
    def count_address_components(addr: str) -> int:
        # Simple heuristic: count likely address components
        parts = addr.strip().split()
        components = 0

        # Check for house number (starts with digit)
        if parts and parts[0][0].isdigit():
            components += 1

        # Check for state (2-letter word, often near end)
        for part in parts:
            if len(part) == 2 and part.isalpha():
                components += 1
                break

        # Check for ZIP (5 or 9 digits, possibly with dash)
        for part in parts:
            if part.replace('-', '').isdigit() and len(part.replace('-', '')) in [5, 9]:
                components += 1
                break

        # Remaining parts likely street name and city
        remaining_parts = len(parts) - components
        if remaining_parts >= 2:  # At least street and city
            components += 2
        elif remaining_parts == 1:
            components += 1

        return components

    if not candidates:
        return None

    # Find candidate with most components
    scored_candidates = [(c, count_address_components(c)) for c in candidates]
    max_score = max(score for _, score in scored_candidates)

    # Check if there's a clear winner
    winners = [c for c, score in scored_candidates if score == max_score]
    if len(winners) == 1 and max_score > 2:  # At least 3 components
        return winners[0]

    return None


# ──────────────────────────── Async LLM Selectors ────────────────────────────
async def llm_select_single(system_prompt: str, user_prompt: str, candidates: List[str]) -> Optional[str]:
    """
    Ask the LLM to pick exactly one candidate. Returns the chosen string or None.
    """
    try:
        resp = await client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt}
            ],
            temperature=0.0,
            max_completion_tokens=80,
        )
        choice = (resp.choices[0].message.content or "").strip()
        return choice if choice else None
    except Exception as e:
        print(f"OpenAI API error: {e}", file=sys.stderr)
        return None


# Cache repeated decisions to avoid duplicate LLM calls
decision_cache: Dict[Tuple[str, Tuple[str, ...], Tuple[str, ...]], str] = {}


async def select_representative_name_async(candidates: List[str], raw_names: List[str], rec_ids: List[str]) -> str:
    """
    Async LLM picks the best combined name, with smart pre-filtering.
    """
    # Try smart selection first
    result, method = smart_select_name(candidates, raw_names)
    if method != "needs_llm":
        return result

    # Need LLM decision
    key = ("name", tuple(candidates), tuple(raw_names))
    if key in decision_cache:
        return decision_cache[key]

    system_prompt = (
        "You are a world-class name-normalization expert. "
        "From the given list of properly combined names (First Middle Last), "
        "select exactly one as the correct, most complete form."
    )
    user_prompt = (
        "Candidates (combined):\n" +
        "\n".join(f"- {n}" for n in candidates) +
        "\n\nRaw variants:\n" +
        "\n".join(f"- {r}" for r in raw_names) +
        "\n\nRespond with ONLY the chosen combined name."
    )

    chosen = await llm_select_single(system_prompt, user_prompt, candidates)
    if chosen and chosen in candidates:
        decision_cache[key] = chosen
        return chosen

    # Fallback
    fallback = max(candidates, key=len)
    decision_cache[key] = fallback
    return fallback


async def select_representative_address_async(candidates: List[str], raw_variants: List[str], rec_ids: List[str]) -> str:
    """
    Async LLM picks the best fully-standardized address, with smart pre-filtering.
    """
    # Try smart selection first
    result, method = smart_select_address(candidates, raw_variants)
    if method != "needs_llm":
        return result

    # Need LLM decision
    key = ("address", tuple(candidates), tuple(raw_variants))
    if key in decision_cache:
        return decision_cache[key]

    system_prompt = (
        "You are a world-class address normalization expert. "
        "From the given list of combined address candidates, "
        "choose exactly one correct, fully-standardized address."
    )
    user_prompt = (
        "Candidates (combined):\n" +
        "\n".join(f"- {a}" for a in candidates) +
        "\n\nRaw variants:\n" +
        "\n".join(f"- {v}" for v in raw_variants) +
        "\n\nRespond with ONLY the chosen combined address."
    )

    chosen = await llm_select_single(system_prompt, user_prompt, candidates)
    if chosen:
        for c in candidates:
            if chosen.strip().upper() == c.strip().upper():
                decision_cache[key] = c.strip().upper()
                return decision_cache[key]

    fallback = max(candidates, key=len).upper()
    decision_cache[key] = fallback
    return fallback


# ──────────────────────────── Async Processing Functions ────────────────────────────
async def process_name_cluster(row, parsed_df: pd.DataFrame) -> Dict:
    """Process a single name cluster asynchronously."""
    recs = [r.strip() for r in str(row.RecIDs).split(";") if r.strip()]
    raw_variants = []
    combined_candidates = []

    for r in recs:
        match = parsed_df[parsed_df.RecID == r]
        if match.empty:
            continue

        raw_name = (match.iloc[0].get("Name") or "").strip()
        if raw_name:
            raw_variants.append(raw_name)

        fn = (match.iloc[0].get("first_name") or "").strip()
        mn = (match.iloc[0].get("middle_name") or "").strip()
        ln = (match.iloc[0].get("last_name") or "").strip()
        if fn and ln:
            combined = f"{fn} {mn} {ln}".replace("  ", " ").strip() if mn else f"{fn} {ln}"
            combined_candidates.append(combined)

    if not combined_candidates:
        if raw_variants:
            chosen = max(dict_dedupe(raw_variants), key=len)
            return {
                "ClusterID": row.ClusterID,  # Keep original for now, will be reassigned
                "CompleteName": chosen,
                "RecIDs": ";".join(recs),
                "Names": ";".join(dict_dedupe(raw_variants))
            }
        return None

    unique_raw = dict_dedupe(raw_variants)
    unique_combined = dict_dedupe(combined_candidates)

    chosen = await select_representative_name_async(unique_combined, unique_raw, recs)
    return {
        "ClusterID":   row.ClusterID,  # Keep original for now, will be reassigned
        "CompleteName": chosen,
        "RecIDs":      ";".join(recs),
        "Names":       ";".join(unique_raw)
    }


async def process_address_cluster(row, parsed_addr_df: pd.DataFrame) -> Dict:
    """Process a single address cluster asynchronously."""
    rec_ids = [r.strip() for r in str(row.RecIDs).split(";") if r.strip()]
    raw_variants = []
    combined_candidates = []

    for rid in rec_ids:
        match = parsed_addr_df[parsed_addr_df.RecID == rid]
        if match.empty:
            continue

        raw_addr = (match.iloc[0].get("Address") or "").strip()
        if raw_addr:
            raw_variants.append(raw_addr)

        hn   = (match.iloc[0].get("house_number") or "").strip()
        sn   = (match.iloc[0].get("street_name") or "").strip()
        city = (match.iloc[0].get("city") or "").strip()
        st   = (match.iloc[0].get("state") or "").strip()
        zp   = (match.iloc[0].get("zip") or "").strip()

        parts = [p for p in [hn, sn, city, st, zp] if p]
        if parts:
            combined = " ".join(parts).strip()
            combined_candidates.append(combined)

    if not combined_candidates:
        if raw_variants:
            chosen = max(dict_dedupe(raw_variants), key=len).upper()
            return {
                "ClusterID":       row.ClusterID,  # Keep original for now, will be reassigned
                "CompleteAddress": chosen,
                "RecIDs":          ";".join(rec_ids),
                "Addresses":       ";".join(dict_dedupe(raw_variants))
            }
        return None

    unique_raw = dict_dedupe(raw_variants)
    unique_combined = dict_dedupe(combined_candidates)

    chosen = await select_representative_address_async(unique_combined, unique_raw, rec_ids)
    return {
        "ClusterID":       row.ClusterID,  # Keep original for now, will be reassigned
        "CompleteAddress": chosen,
        "RecIDs":          ";".join(rec_ids),
        "Addresses":       ";".join(unique_raw)
    }


async def run_names_pipeline() -> None:
    """Run the names pipeline with async processing."""
    name_clusters_path = NAME_CLUSTERS
    parsed_names_path  = PARSED_NAMES

    if not os.path.exists(name_clusters_path) or not os.path.exists(parsed_names_path):
        print("Skipping NAME clusters: required files not found.", file=sys.stderr)
        return

    clusters = pd.read_csv(name_clusters_path, dtype=str, sep=CSV_SEP).fillna("")
    parsed_df = pd.read_csv(parsed_names_path, dtype=str, sep=CSV_SEP).fillna("")

    print(f"\nProcessing NAME clusters from '{name_clusters_path}'…")

    # Create tasks for all clusters
    tasks = [process_name_cluster(row, parsed_df) for _, row in clusters.iterrows()]

    # Process with progress bar and concurrency control
    semaphore = asyncio.Semaphore(10)  # Limit concurrent LLM calls

    async def bounded_task(task):
        async with semaphore:
            return await task

    bounded_tasks = [bounded_task(task) for task in tasks]

    # Process all tasks with progress tracking
    results = []
    for coro in tqdm(asyncio.as_completed(bounded_tasks), total=len(bounded_tasks), desc="Name clusters"):
        result = await coro
        if result:
            results.append(result)

    if results:
        out_df = pd.DataFrame(results)
        # Sort by minimum RecID and reassign ClusterIDs
        out_df = sort_and_reassign_cluster_ids(out_df)
        atomic_write_csv(out_df, NAMES_OUT_CSV, sep=CSV_SEP)
        print(f"→ Names result written to {NAMES_OUT_CSV} (sorted by RecID with reassigned ClusterIDs)")
    else:
        print("No NAME clusters produced output.", file=sys.stderr)


async def run_addresses_pipeline() -> None:
    """Run the addresses pipeline with async processing."""
    addr_clusters_path = ADDRESS_CLUSTERS
    parsed_addr_path   = PARSED_ADDR

    if not os.path.exists(addr_clusters_path) or not os.path.exists(parsed_addr_path):
        print("Skipping ADDRESS clusters: required files not found.", file=sys.stderr)
        return

    clusters_df = pd.read_csv(addr_clusters_path, dtype=str, sep=CSV_SEP).fillna("")
    parsed_addr_df = pd.read_csv(parsed_addr_path, dtype=str, sep=CSV_SEP).fillna("")

    print(f"\nProcessing ADDRESS clusters from '{addr_clusters_path}'…")

    # Create tasks for all clusters
    tasks = [process_address_cluster(row, parsed_addr_df) for _, row in clusters_df.iterrows()]

    # Process with progress bar and concurrency control
    semaphore = asyncio.Semaphore(10)  # Limit concurrent LLM calls

    async def bounded_task(task):
        async with semaphore:
            return await task

    bounded_tasks = [bounded_task(task) for task in tasks]

    # Process all tasks with progress tracking
    results = []
    for coro in tqdm(asyncio.as_completed(bounded_tasks), total=len(bounded_tasks), desc="Address clusters"):
        result = await coro
        if result:
            results.append(result)

    if results:
        out_df = pd.DataFrame(results)
        # Sort by minimum RecID and reassign ClusterIDs
        out_df = sort_and_reassign_cluster_ids(out_df)
        atomic_write_csv(out_df, ADDRESSES_OUT_CSV, sep=CSV_SEP)
        print(f"→ Addresses result written to {ADDRESSES_OUT_CSV} (sorted by RecID with reassigned ClusterIDs)")
    else:
        print("No ADDRESS clusters produced output.", file=sys.stderr)


# ──────────────────────────── Main ────────────────────────────
async def main_async():
    """Main async function."""
    print("Starting combined Name+Address cluster summarization…")
    print("Output will be sorted by RecID with reassigned ClusterIDs based on order.")

    await asyncio.gather(
        run_names_pipeline(),
        run_addresses_pipeline()
    )


def main():
    """Main entry point."""
    t0 = time.time()

    # Handle both regular Python and Jupyter notebook environments
    try:
        loop = asyncio.get_running_loop()
        # We're in a Jupyter notebook - create a task
        import nest_asyncio
        nest_asyncio.apply()
        asyncio.run(main_async())
    except RuntimeError:
        # We're in regular Python
        asyncio.run(main_async())
    except ImportError:
        # nest_asyncio not available, try alternative approach
        print("Note: Running in Jupyter without nest_asyncio. Consider installing: pip install nest-asyncio")
        print("For now, running synchronously...")
        # Fall back to sync version if needed
        asyncio.run(main_async())

    # Print optimization statistics
    total_decisions = len(decision_cache)
    print(f"\nOptimization Statistics:")
    print(f"Total LLM calls made: {total_decisions}")
    print(f"Done in {time.time() - t0:.2f}s.")


# Alternative function for Jupyter notebooks
async def main_jupyter():
    """Use this function directly in Jupyter notebooks."""
    t0 = time.time()

    await main_async()

    # Print optimization statistics
    total_decisions = len(decision_cache)
    print(f"\nOptimization Statistics:")
    print(f"Total LLM calls made: {total_decisions}")
    print(f"Done in {time.time() - t0:.2f}s.")


if __name__ == "__main__":
    main()

Starting combined Name+Address cluster summarization…
Output will be sorted by RecID with reassigned ClusterIDs based on order.

Processing NAME clusters from 'd1_name_clusters.csv'…


Name clusters:   0%|          | 0/732 [00:00<?, ?it/s]


Processing ADDRESS clusters from 'd1_address_clusters.csv'…



Name clusters: 100%|██████████| 732/732 [00:07<00:00, 99.97it/s]

Address clusters:  83%|████████▎ | 527/638 [00:07<00:02, 49.55it/s]

→ Names result written to e_cluster_complete_names.csv (sorted by RecID with reassigned ClusterIDs)



Address clusters: 100%|██████████| 638/638 [00:09<00:00, 67.05it/s] 

→ Addresses result written to e_cluster_complete_addresses.csv (sorted by RecID with reassigned ClusterIDs)

Optimization Statistics:
Total LLM calls made: 137
Done in 9.60s.


In [10]:
#Step 6: the standardized results of LLM are put together
#outouts a major file that is relevant - direct links
import os
import sys
import re
import time
from collections import Counter
import pandas as pd

def debug_read_csv(path, sep=','):
    """Read CSV, print and clean column names."""
    try:
        df = pd.read_csv(path, sep=sep, dtype=str)
    except Exception as e:
        print(f"Error reading {path}: {e}", file=sys.stderr)
        sys.exit(1)
    print(f"Raw columns in {path}: {df.columns.tolist()}")
    df.columns = df.columns.str.strip().str.replace('"', '')
    print(f"Clean columns in {path}: {df.columns.tolist()}")
    return df

def build_map_from_complete(path, complete_col):
    """
    Build a dict mapping each RecID to its exact CompleteName or CompleteAddress.
    Expects columns: complete_col, RecIDs.
    """
    df = debug_read_csv(path)
    if complete_col not in df.columns or 'RecIDs' not in df.columns:
        raise KeyError(f"{path} must contain '{complete_col}' and 'RecIDs'")
    mapping = {}
    for _, row in df.iterrows():
        val = row[complete_col].strip()
        recids = [
            r.strip() for r in re.split(r'[;,]\s*', row['RecIDs'])
            if r.strip()
        ]
        for rid in recids:
            mapping[rid] = val
    return mapping

def combine_clusters(combined_path, names_map, addrs_map):
    """
    Reads combined_clusters.csv and for each ClusterID & its RecIDs:
      • collects CompleteName values,
      • picks the single most frequent name (ties broken by first occurrence),
      • collects CompleteAddress values (deduped & joined by ';'),
      • calculates Direct links as n*(n-1)/2.
    """
    df = debug_read_csv(combined_path)
    if 'ClusterID' not in df.columns or 'RecIDs' not in df.columns:
        raise KeyError(f"{combined_path} must contain 'ClusterID' and 'RecIDs'")

    best_names = []
    best_addrs = []
    direct_links = []

    for _, row in df.iterrows():
        recids = [
            r.strip() for r in re.split(r'[;,]\s*', row['RecIDs'])
            if r.strip()
        ]
        n = len(recids)
        direct_links.append(n * (n - 1) // 2)

        # Gather all names for these RecIDs
        names = [names_map.get(rid, '') for rid in recids]
        names = [n for n in names if n]  # drop empties

        # Choose the single most frequent name
        if names:
            counts = Counter(names)
            # preserve first-occurrence order
            unique_names = list(dict.fromkeys(names))
            max_count = max(counts.values())
            # pick the first name with highest count
            chosen_name = next(n for n in unique_names if counts[n] == max_count)
        else:
            chosen_name = ''

        best_names.append(chosen_name)

        # Addresses: dedupe preserving order
        addrs = [addrs_map.get(rid, '') for rid in recids]
        unique_addrs = list(dict.fromkeys(a for a in addrs if a))
        best_addrs.append(";".join(unique_addrs))

    df['BestName'] = best_names
    df['BestAddress'] = best_addrs
    df['Direct links'] = direct_links
    return df

def main():
    start = time.time()
    names_file     = 'e_cluster_complete_names.csv'
    addresses_file = 'e_cluster_complete_addresses.csv'
    combined_file  = 'd1_combined_clusters.csv'
    output_file    = 'f_combined_clusters_with_best.csv'

    for f in (names_file, addresses_file, combined_file):
        if not os.path.exists(f):
            print(f"Missing file: {f}", file=sys.stderr)
            sys.exit(1)

    names_map = build_map_from_complete(names_file, 'CompleteName')
    addrs_map = build_map_from_complete(addresses_file, 'CompleteAddress')

    result_df = combine_clusters(combined_file, names_map, addrs_map)
    result_df.to_csv(output_file, index=False)
    elapsed = time.time() - start
    print(f"Saved '{output_file}' ({len(result_df)} rows) in {elapsed:.2f}s")

    # Compute and display sum of all Direct links
    total_direct_links = result_df['Direct links'].sum()
    print(f"Sum of all Direct links: {total_direct_links}")

if __name__ == "__main__":
    main()

Raw columns in e_cluster_complete_names.csv: ['ClusterID', 'CompleteName', 'RecIDs', 'Names']
Clean columns in e_cluster_complete_names.csv: ['ClusterID', 'CompleteName', 'RecIDs', 'Names']
Raw columns in e_cluster_complete_addresses.csv: ['ClusterID', 'CompleteAddress', 'RecIDs', 'Addresses']
Clean columns in e_cluster_complete_addresses.csv: ['ClusterID', 'CompleteAddress', 'RecIDs', 'Addresses']
Raw columns in d1_combined_clusters.csv: ['ClusterID', 'RecIDs']
Clean columns in d1_combined_clusters.csv: ['ClusterID', 'RecIDs']
Saved 'f_combined_clusters_with_best.csv' (995 rows) in 0.18s
Sum of all Direct links: 1675


In [11]:
#Step 7
import pandas as pd

INPUT_CSV = "f_combined_clusters_with_best.csv"
OUT_BY_NAME = "g_grouped_by_name.csv"
OUT_BY_ADDRESS = "g_grouped_by_address.csv"

# ---------- helpers ----------
def recid_key(s: str):
    """
    Turn a RecID like '88.10' or '003.2' into a sortable (major, minor) tuple of ints.
    Non-digits are ignored; missing minor → 0.
    """
    s = str(s).strip()
    if "." in s:
        a, b = s.split(".", 1)
    else:
        a, b = s, "0"
    def to_int(x: str) -> int:
        digits = "".join(ch for ch in str(x) if ch.isdigit())
        return int(digits) if digits else 0
    return (to_int(a), to_int(b))

def dedupe_preserve_order(items):
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def parse_recids_cell(cell: str):
    if pd.isna(cell) or str(cell).strip() == "":
        return []
    return [x.strip() for x in str(cell).split(";") if x.strip()]

def flatten_links(vals):
    """
    vals: series of "Direct links" cells (strings, possibly ';'-separated).
    Returns a list of unique links (order preserved).
    """
    acc = []
    for v in vals:
        if pd.isna(v) or str(v).strip() == "":
            continue
        parts = [p.strip() for p in str(v).split(";") if p.strip()]
        acc.extend(parts)
    return dedupe_preserve_order(acc)

# ---------- core grouping ----------
def group_by_name(df: pd.DataFrame) -> pd.DataFrame:
    # split and aggregate
    df["RecIDs_list"] = df["RecIDs"].apply(parse_recids_cell)
    grouped = (
        df.groupby("BestName", sort=False)
          .agg({
              "RecIDs_list": "sum",
              "BestAddress": lambda addrs: dedupe_preserve_order(
                  [str(a).strip() for a in addrs if pd.notna(a) and str(a).strip() != ""]
              ),
              "Direct links": flatten_links
          })
          .reset_index()
    )

    # sort RecIDs within groups
    grouped["RecIDs_list"] = grouped["RecIDs_list"].apply(
        lambda lst: sorted([r for r in lst if r], key=recid_key)
    )

    # cluster order by smallest RecID
    grouped["__first_recid_key__"] = grouped["RecIDs_list"].apply(
        lambda lst: recid_key(lst[0]) if lst else (float("inf"), float("inf"))
    )
    grouped = grouped.sort_values("__first_recid_key__", kind="mergesort").reset_index(drop=True)
    grouped.insert(0, "ClusterID", range(len(grouped)))

    # finalize columns
    grouped["RecIDs"] = grouped["RecIDs_list"].apply(lambda lst: ";".join(lst))
    grouped["Addresses"] = grouped["BestAddress"].apply(lambda addrs: "; ".join(addrs))
    grouped["Direct links"] = grouped["Direct links"].apply(lambda links: ";".join(links))

    out = grouped[["ClusterID", "BestName", "RecIDs", "Addresses", "Direct links"]].copy()
    out = out.rename(columns={"BestName": "Name"})
    return out

def group_by_address(df: pd.DataFrame) -> pd.DataFrame:
    # split and aggregate
    df["RecIDs_list"] = df["RecIDs"].apply(parse_recids_cell)
    grouped = (
        df.groupby("BestAddress", sort=False)
          .agg({
              "RecIDs_list": "sum",
              "BestName": lambda names: dedupe_preserve_order(
                  [str(n).strip() for n in names if pd.notna(n) and str(n).strip() != ""]
              ),
              "Direct links": flatten_links
          })
          .reset_index()
    )

    # sort RecIDs within groups
    grouped["RecIDs_list"] = grouped["RecIDs_list"].apply(
        lambda lst: sorted([r for r in lst if r], key=recid_key)
    )

    # cluster order by smallest RecID
    grouped["__first_recid_key__"] = grouped["RecIDs_list"].apply(
        lambda lst: recid_key(lst[0]) if lst else (float("inf"), float("inf"))
    )
    grouped = grouped.sort_values("__first_recid_key__", kind="mergesort").reset_index(drop=True)
    grouped.insert(0, "ClusterID", range(len(grouped)))

    # finalize columns
    grouped["RecIDs"] = grouped["RecIDs_list"].apply(lambda lst: ";".join(lst))
    grouped["Name"] = grouped["BestName"].apply(lambda names: "; ".join(names))
    grouped["Direct links"] = grouped["Direct links"].apply(lambda links: ";".join(links))

    out = grouped[["ClusterID", "BestAddress", "RecIDs", "Name", "Direct links"]].copy()
    out = out.rename(columns={"BestAddress": "Addresses"})
    return out

# ---------- run both in one program ----------
def main():
    df = pd.read_csv(INPUT_CSV, dtype=str).fillna("")
    if "Direct links" not in df.columns:
        df["Direct links"] = ""  # ensure column exists

    out_name = group_by_name(df.copy())
    out_name.to_csv(OUT_BY_NAME, index=False)
    print(f"Saved (grouped by BestName, ordered by RecID): {OUT_BY_NAME}")

    out_addr = group_by_address(df.copy())
    out_addr.to_csv(OUT_BY_ADDRESS, index=False)
    print(f"Saved (grouped by BestAddress, ordered by RecID): {OUT_BY_ADDRESS}")

if __name__ == "__main__":
    main()

Saved (grouped by BestName, ordered by RecID): g_grouped_by_name.csv
Saved (grouped by BestAddress, ordered by RecID): g_grouped_by_address.csv


In [12]:
#Just to check if there is overlap or not
#explains where the LLM failed
import pandas as pd
import io

# This part simulates your CSV file.
# In your actual use, you would replace this with:
# file_path = 'd_name_clusters.csv'
# df = pd.read_csv(file_path)

csv_data = "f_combined_clusters_with_best.csv"

# Read the data into a pandas DataFrame
df = pd.read_csv(csv_data)

# --- Main Logic ---

# Loop through each row in the DataFrame
for index, row in df.iterrows():
    cluster_id = row['ClusterID']
    rec_ids_str = row['RecIDs']

    # Split the RecIDs string by the semicolon to get a list of individual IDs
    rec_id_list = rec_ids_str.split(';')

    # Extract the leading digit (the part before the '.') from each RecID
    # A set is used to store them, as it automatically handles duplicates.
    leading_digits = set(rec_id.split('.')[0] for rec_id in rec_id_list)

    # Check if the set contains exactly one unique leading digit
    if len(leading_digits) == 1:
        # If yes, the check passes for this cluster
        print(f"✅ ClusterID {cluster_id}: OK. All RecIDs start with '{list(leading_digits)[0]}'.")
    else:
        # If no, the check fails
        print(f"❌ ClusterID {cluster_id}: FAIL. Mismatched leading digits found: {sorted(list(leading_digits))}.")

✅ ClusterID 0: OK. All RecIDs start with '1'.
✅ ClusterID 1: OK. All RecIDs start with '1'.
✅ ClusterID 2: OK. All RecIDs start with '2'.
✅ ClusterID 3: OK. All RecIDs start with '3'.
✅ ClusterID 4: OK. All RecIDs start with '4'.
✅ ClusterID 5: OK. All RecIDs start with '4'.
✅ ClusterID 6: OK. All RecIDs start with '5'.
✅ ClusterID 7: OK. All RecIDs start with '6'.
✅ ClusterID 8: OK. All RecIDs start with '7'.
✅ ClusterID 9: OK. All RecIDs start with '8'.
✅ ClusterID 10: OK. All RecIDs start with '9'.
✅ ClusterID 11: OK. All RecIDs start with '9'.
✅ ClusterID 12: OK. All RecIDs start with '10'.
✅ ClusterID 13: OK. All RecIDs start with '11'.
✅ ClusterID 14: OK. All RecIDs start with '12'.
✅ ClusterID 15: OK. All RecIDs start with '13'.
✅ ClusterID 16: OK. All RecIDs start with '14'.
✅ ClusterID 17: OK. All RecIDs start with '15'.
✅ ClusterID 18: OK. All RecIDs start with '16'.
✅ ClusterID 19: OK. All RecIDs start with '17'.
✅ ClusterID 20: OK. All RecIDs start with '17'.
✅ ClusterID 21

In [13]:
# Step 8 — refined merge with parallel, batch, and async API calls
import os
import pandas as pd
import json
import asyncio
import aiohttp
from itertools import combinations
from collections import defaultdict
from openai import OpenAI
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
from typing import List, Tuple, Dict, Set

# Ensure your OPENAI_API_KEY is exported as an environment variable
# export OPENAI_API_KEY="sk-..."
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Configuration for API optimization
API_CONFIG = {
    "max_parallel_workers": 10,  # Adjust based on your API rate limits
    "batch_size": 5,  # Number of comparisons per batch
    "use_async": True,  # Toggle between async and thread-based parallel (set to False to avoid async issues)
    "use_batch": True,  # Toggle batch processing (set to False if batching fails)
    "rate_limit_delay": 0.1,  # Delay between batches to respect rate limits
}

# --- SINGLE API CALL (Original) ---
def call_o4_mini_same_person(name1: str, name2: str, shared_addresses: list[str]) -> bool:
    """
    Original single API call - kept for compatibility and fallback.
    """
    prompt = f"""
You are an expert data-deduplication specialist. Your task is to determine if two name clusters refer to the EXACT SAME INDIVIDUAL.

**CRITICAL RULE:** Only merge if you are confident these are the SAME PERSON. When in doubt, DO NOT MERGE.

**NEVER MERGE when:**
- First names are completely different (e.g., GLORIA vs TOMMY, JOHN vs MARY, MICHAEL vs SUSAN)
- Names suggest different genders
- Names have no reasonable connection (not nicknames, not variations)

**ONLY MERGE when:**
- Same first name + shared address (likely last name change due to marriage)
- Same first name + same middle name/initial + shared address
- Clear nickname relationship + other strong evidence (e.g., BOB/ROBERT, MIKE/MICHAEL)

**Examples:**

1) Example #1 (MERGE - Same person, maiden/married name)
    Name A: "GLORIA B MIDDLEBROOK"
    Name B: "GLORIA B NEL"
    Shared address: ["10223 LIPSCOMB DR SAN DIEGO CA"]
    Analysis: Same first name (GLORIA), same middle initial (B), shared address
    → {{ "same_person": true }}

2) Example #2 (DO NOT MERGE - Different people)
    Name A: "GLORIA B MIDDLEBROOK"
    Name B: "TOMMY ALAN NOEL"
    Shared address: ["10223 LIPSCOMB DR SAN DIEGO CA"]
    Analysis: Completely different first names (GLORIA vs TOMMY), different genders
    → {{ "same_person": false }}

3) Example #3 (DO NOT MERGE - Family members)
    Name A: "JOHN SMITH"
    Name B: "MARY SMITH"
    Shared address: ["123 MAIN ST"]
    Analysis: Different first names, likely spouses or family
    → {{ "same_person": false }}

4) Example #4 (MERGE - Same person with nickname)
    Name A: "MICHAEL J JONES"
    Name B: "MIKE J JONES"
    Shared address: ["456 OAK AVE"]
    Analysis: MIKE is common nickname for MICHAEL, same middle initial, shared address
    → {{ "same_person": true }}

**Now evaluate:**
Name A: "{name1}"
Name B: "{name2}"
Shared address(es): {json.dumps(shared_addresses)}

Are these the SAME INDIVIDUAL? Return ONLY: {{ "same_person": true/false }}"""

    response = client.chat.completions.create(
        model="o4-mini",  # Using o4-mini as requested
        messages=[{"role": "user", "content": prompt.strip()}],
        temperature=1,  # As requested
        max_completion_tokens=16384  # Maximum tokens allowed
    )
    text = response.choices[0].message.content.strip()
    try:
        json_str = text[text.find('{'):text.rfind('}')+1]
        parsed = json.loads(json_str)
        return bool(parsed.get("same_person", False))
    except (json.JSONDecodeError, AttributeError):
        low = text.lower()
        return '"same_person": true' in low

# --- BATCH API CALL ---
def call_o4_mini_batch(pairs_info: List[Tuple[str, str, List[str]]]) -> List[bool]:
    """
    Process multiple pairs in a single API call for efficiency.
    Returns a list of booleans for each pair.
    """
    if not pairs_info:
        return []

    prompt = """You are an expert data-deduplication specialist. For each pair, determine if they are the SAME INDIVIDUAL.

CRITICAL RULES:
- NEVER merge people with completely different first names (GLORIA vs TOMMY = FALSE)
- NEVER merge obvious family members (JOHN vs MARY at same address = FALSE)
- ONLY merge if confident they are the SAME PERSON
- When in doubt, return FALSE

Valid merges:
- Same first name + shared address (even with different last names)
- Clear nicknames (BOB/ROBERT, MIKE/MICHAEL) + other evidence
- Same first + middle initial + shared address

"""

    for i, (name1, name2, addrs) in enumerate(pairs_info, 1):
        prompt += f"""
Pair {i}:
  Name A: {name1}
  Name B: {name2}
  Shared: {json.dumps(addrs)}
"""

    prompt += "\nReturn ONLY a JSON array of booleans: [true/false, true/false, ...]"

    try:
        response = client.chat.completions.create(
            model="o4-mini",  # Using o4-mini as requested
            messages=[{"role": "user", "content": prompt.strip()}],
            temperature=1,  # As requested
            max_completion_tokens=16384  # Maximum tokens allowed
        )
        text = response.choices[0].message.content.strip()

        # Extract JSON array
        json_str = text[text.find('['):text.rfind(']')+1]
        results = json.loads(json_str)

        # Ensure we have the right number of results
        if len(results) != len(pairs_info):
            print(f"Warning: Expected {len(pairs_info)} results, got {len(results)}. Using fallback.")
            return [call_o4_mini_same_person(n1, n2, addrs)
                    for n1, n2, addrs in pairs_info]

        return [bool(r) for r in results]
    except Exception as e:
        print(f"Batch processing failed: {e}. Using fallback.")
        return [call_o4_mini_same_person(n1, n2, addrs)
                for n1, n2, addrs in pairs_info]

# --- ASYNC API CALLS ---
async def call_o4_mini_async(session, name1: str, name2: str, shared_addresses: List[str]) -> Tuple[str, str, bool]:
    """
    Async version of the API call using aiohttp.
    Returns (name1, name2, result) tuple.
    """
    prompt = f"""Determine if these names are the SAME INDIVIDUAL:
Name A: {name1}
Name B: {name2}
Shared addresses: {json.dumps(shared_addresses)}

CRITICAL:
- NEVER merge different first names (GLORIA vs TOMMY = false)
- ONLY merge if same person (e.g., GLORIA B SMITH vs GLORIA B JONES = true)
- When in doubt, return false

Return only: {{"same_person": true/false}}"""

    headers = {
        "Authorization": f"Bearer {os.getenv('OPENAI_API_KEY')}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "o4-mini",  # Using o4-mini as requested
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 1,  # As requested
        "max_completion_tokens": 16384  # Maximum tokens allowed
    }

    try:
        async with session.post(
            "https://api.openai.com/v1/chat/completions",
            headers=headers,
            json=data
        ) as response:
            result = await response.json()
            text = result['choices'][0]['message']['content'].strip()

            try:
                json_str = text[text.find('{'):text.rfind('}')+1]
                parsed = json.loads(json_str)
                return (name1, name2, bool(parsed.get("same_person", False)))
            except:
                return (name1, name2, '"same_person": true' in text.lower())
    except Exception as e:
        print(f"Async call failed for {name1} vs {name2}: {e}")
        return (name1, name2, False)

async def process_pairs_async(pairs_to_check: List[Tuple[int, int, Set[str]]], cluster_info: Dict) -> List[Tuple[int, int, bool]]:
    """
    Process all pairs using async API calls.
    """
    async with aiohttp.ClientSession() as session:
        tasks = []
        pair_mapping = {}  # Map (name1, name2) to (cid1, cid2)

        for cid1, cid2, shared_addrs in pairs_to_check:
            name1 = cluster_info[cid1]['name']
            name2 = cluster_info[cid2]['name']
            pair_mapping[(name1, name2)] = (cid1, cid2)

            task = call_o4_mini_async(session, name1, name2, list(shared_addrs))
            tasks.append(task)

            # Add small delay to respect rate limits
            if len(tasks) % 10 == 0:
                await asyncio.sleep(API_CONFIG["rate_limit_delay"])

        results = []
        for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Async API Calls"):
            name1, name2, should_merge = await coro
            cid1, cid2 = pair_mapping[(name1, name2)]
            results.append((cid1, cid2, should_merge))

        return results

# --- PARALLEL API CALLS (Thread-based) ---
def process_single_pair(args: Tuple[int, int, Dict, Set[str]]) -> Tuple[int, int, bool]:
    """
    Helper function for parallel processing using threads.
    """
    cid1, cid2, cluster_info, shared_addresses = args
    name1 = cluster_info[cid1]['name']
    name2 = cluster_info[cid2]['name']

    result = call_o4_mini_same_person(name1, name2, list(shared_addresses))
    return (cid1, cid2, result)

def process_pairs_parallel(pairs_to_check: List[Tuple[int, int, Set[str]]], cluster_info: Dict) -> List[Tuple[int, int, bool]]:
    """
    Process pairs using ThreadPoolExecutor for parallel API calls.
    """
    # Prepare arguments for parallel processing
    args_list = [(cid1, cid2, cluster_info, shared_addrs)
                 for cid1, cid2, shared_addrs in pairs_to_check]

    results = []
    with ThreadPoolExecutor(max_workers=API_CONFIG["max_parallel_workers"]) as executor:
        futures = [executor.submit(process_single_pair, args) for args in args_list]

        for future in tqdm(as_completed(futures), total=len(futures), desc="Parallel API Calls"):
            try:
                result = future.result(timeout=30)
                results.append(result)
            except Exception as e:
                print(f"Parallel processing error: {e}")

    return results

# --- BATCH PROCESSING WITH PARALLEL/ASYNC ---
def process_pairs_batch(pairs_to_check: List[Tuple[int, int, Set[str]]], cluster_info: Dict) -> List[Tuple[int, int, bool]]:
    """
    Process pairs in batches for efficiency.
    """
    results = []
    batch_size = API_CONFIG["batch_size"]

    # Prepare batch data
    batch_data = []
    cid_mapping = []

    for cid1, cid2, shared_addrs in pairs_to_check:
        name1 = cluster_info[cid1]['name']
        name2 = cluster_info[cid2]['name']
        batch_data.append((name1, name2, list(shared_addrs)))
        cid_mapping.append((cid1, cid2))

    # Process in batches
    for i in tqdm(range(0, len(batch_data), batch_size), desc="Batch Processing"):
        batch = batch_data[i:i + batch_size]
        batch_cids = cid_mapping[i:i + batch_size]

        batch_results = call_o4_mini_batch(batch)

        for (cid1, cid2), should_merge in zip(batch_cids, batch_results):
            results.append((cid1, cid2, should_merge))

        # Rate limit delay between batches
        if i + batch_size < len(batch_data):
            time.sleep(API_CONFIG["rate_limit_delay"])

    return results

# --- HELPERS (unchanged from original) ---

def get_name_parts(name: str) -> tuple[str, str, str]:
    """Parses a name string into (first_name, middle_name, last_name), uppercased."""
    parts = name.strip().upper().split()
    if not parts:
        return "", "", ""
    if len(parts) == 1:
        # Assume a single token is a last name
        return "", "", parts[0]
    first_name = parts[0]
    last_name = parts[-1]
    middle_name = " ".join(parts[1:-1])
    return first_name, middle_name, last_name

def middle_names_match(m1: str, m2: str) -> bool:
    """
    Middle names match if:
    - exact match (ignoring '.'),
    - or one is the initial of the other (e.g., 'J' vs 'JOANNE').
    Requires both to be non-empty.
    """
    if not m1 or not m2:
        return False
    m1 = m1.replace(".", "")
    m2 = m2.replace(".", "")
    if m1 == m2:
        return True
    if len(m1) == 1 and m2.startswith(m1):
        return True
    if len(m2) == 1 and m1.startswith(m2):
        return True
    return False

def names_are_compatible(name1_parts: tuple[str, str, str], name2_parts: tuple[str, str, str]) -> bool:
    """
    Check if two name tuples could represent the same person.
    ENHANCED: More conservative to avoid false merges.
    """
    first1, middle1, last1 = name1_parts
    first2, middle2, last2 = name2_parts

    # CRITICAL: Different first names = NOT the same person
    # (unless it's a known nickname relationship)
    if first1 and first2:
        # Check for exact match
        if first1 == first2:
            # Same first name - check other conditions
            pass
        else:
            # Check for common nicknames only
            nicknames = {
                ('MIKE', 'MICHAEL'), ('MICHAEL', 'MIKE'),
                ('BOB', 'ROBERT'), ('ROBERT', 'BOB'),
                ('BILL', 'WILLIAM'), ('WILLIAM', 'BILL'),
                ('JIM', 'JAMES'), ('JAMES', 'JIM'),
                ('DICK', 'RICHARD'), ('RICHARD', 'DICK'),
                ('TOM', 'THOMAS'), ('THOMAS', 'TOM'),
                ('CHRIS', 'CHRISTOPHER'), ('CHRISTOPHER', 'CHRIS'),
                ('LIZ', 'ELIZABETH'), ('ELIZABETH', 'LIZ'),
                ('BETH', 'ELIZABETH'), ('ELIZABETH', 'BETH'),
            }
            if (first1, first2) not in nicknames:
                return False  # Different first names, not nicknames = different people

    # Last names must match (unless one is empty)
    if last1 and last2 and last1 != last2:
        return False

    # Case 1: One name is "MIDDLE LAST" and the other is "FIRST MIDDLE LAST"
    if not first1 and middle1 and last1:
        if first2 and middle2 and last2:
            return middle1 == middle2 and last1 == last2
    elif not first2 and middle2 and last2:
        if first1 and middle1 and last1:
            return middle1 == middle2 and last1 == last2

    # Case 2: First+Last match, middles compatible / optional
    if first1 and first2 and first1 == first2 and last1 == last2:
        if not middle1 or not middle2:
            return True
        if middle1 == middle2:
            return True
        # One middle initial vs the other's full
        if len(middle1) == 1 and middle2.startswith(middle1):
            return True
        if len(middle2) == 1 and middle1.startswith(middle2):
            return True

    return False

# --- ENHANCED MERGE LOGIC ---

def merge_clusters(df_name: pd.DataFrame) -> dict[int, int]:
    """
    Merges clusters using:
    1) Deterministic rules for shared addresses and name compatibility.
    2) Bridge logic: connects A and B with same (first, middle) but different last names.
    3) AI fallback with parallel/batch/async processing for remaining pairs.
    """
    parent = {cid: cid for cid in df_name["ClusterID"].astype(int)}

    def find(x: int) -> int:
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(a: int, b: int):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    # --- Pre-computation Step ---
    print("Pre-computing name parts and address lookups...")
    cluster_info = {}
    for _, row in df_name.iterrows():
        cid = int(row["ClusterID"])
        name = row["Name"]
        first, middle, last = get_name_parts(name)
        addresses = set(addr.strip() for addr in row["Addresses"].split(";") if addr.strip())
        cluster_info[cid] = {
            "first": first, "middle": middle, "last": last,
            "name": name, "addresses": addresses,
            "name_parts": (first, middle, last)
        }

    # Lookups for bridge logic
    fm_name_map = defaultdict(list)   # (first, middle) -> [cids]
    last_name_map = defaultdict(list) # last -> [cids]
    for cid, info in cluster_info.items():
        if info['first'] and info['middle']:
            fm_name_map[(info['first'], info['middle'])].append(cid)
        if info['last']:
            last_name_map[info['last']].append(cid)

    # --- Step 1: Deterministic merges for pairs with shared addresses ---
    print("\nStep 1: Applying deterministic rules for shared addresses...")
    all_cids = list(parent.keys())
    total_pairs = len(all_cids) * (len(all_cids) - 1) // 2

    for cid1, cid2 in tqdm(combinations(all_cids, 2), total=total_pairs, desc="Name Compatibility"):
        if find(cid1) == find(cid2):
            continue

        shared_addresses = cluster_info[cid1]['addresses'].intersection(cluster_info[cid2]['addresses'])
        if not shared_addresses:
            continue

        # Fast path: identical names
        if cluster_info[cid1]['name'].strip().upper() == cluster_info[cid2]['name'].strip().upper():
            union(cid1, cid2)
            continue

        # Deterministic rule: same FIRST + MIDDLE + shared address
        f1, m1, _ = cluster_info[cid1]['name_parts']
        f2, m2, _ = cluster_info[cid2]['name_parts']
        if f1 and (f1 == f2) and middle_names_match(m1, m2):
            union(cid1, cid2)
            continue

        # Existing deterministic rule: names_are_compatible
        if names_are_compatible(cluster_info[cid1]['name_parts'], cluster_info[cid2]['name_parts']):
            union(cid1, cid2)
            continue

    # --- Step 2: Bridge logic ---
    print("\nStep 2: Applying the 'bridge' logic for name/address connections...")
    for (first, middle), cids in tqdm(fm_name_map.items(), desc="Bridge Logic"):
        if len(cids) < 2:
            continue

        for cid_a, cid_b in combinations(cids, 2):
            if find(cid_a) == find(cid_b):
                continue

            info_a = cluster_info[cid_a]
            info_b = cluster_info[cid_b]

            if info_a['last'] == info_b['last']:
                continue

            # Bridge Type 1
            bridged = False
            for cid_c in last_name_map.get(info_b['last'], []):
                if cid_c == cid_a or cid_c == cid_b:
                    continue
                info_c = cluster_info[cid_c]
                if info_c['addresses'].intersection(info_a['addresses']):
                    union(cid_a, cid_b)
                    bridged = True
                    break
            if bridged:
                continue

            # Bridge Type 2
            for cid_c in last_name_map.get(info_a['last'], []):
                if cid_c == cid_a or cid_c == cid_b:
                    continue
                info_c = cluster_info[cid_c]
                if info_c['addresses'].intersection(info_b['addresses']):
                    union(cid_a, cid_b)
                    break

    # --- Step 3: Enhanced AI fallback with parallel/batch/async ---
    print("\nStep 3: Comparing remaining clusters using optimized AI calls...")
    print(f"Configuration: Batch={API_CONFIG['use_batch']}, Async={API_CONFIG['use_async']}, Workers={API_CONFIG['max_parallel_workers']}")

    # Collect pairs that need AI evaluation
    pairs_to_check = []
    for cid1, cid2 in combinations(all_cids, 2):
        if find(cid1) == find(cid2):
            continue

        shared_addresses = cluster_info[cid1]['addresses'].intersection(cluster_info[cid2]['addresses'])
        if not shared_addresses:
            continue

        name1 = cluster_info[cid1]['name']
        name2 = cluster_info[cid2]['name']

        # Skip deterministic cases
        f1, m1, _ = cluster_info[cid1]['name_parts']
        f2, m2, _ = cluster_info[cid2]['name_parts']
        if (name1.strip().upper() == name2.strip().upper() or
            (f1 and (f1 == f2) and middle_names_match(m1, m2)) or
            names_are_compatible(cluster_info[cid1]['name_parts'], cluster_info[cid2]['name_parts'])):
            continue

        pairs_to_check.append((cid1, cid2, shared_addresses))

    print(f"Found {len(pairs_to_check)} pairs requiring AI evaluation")

    if pairs_to_check:
        # Choose processing method based on configuration
        if API_CONFIG["use_batch"]:
            results = process_pairs_batch(pairs_to_check, cluster_info)
        elif API_CONFIG["use_async"]:
            # Run async processing
            results = asyncio.run(process_pairs_async(pairs_to_check, cluster_info))
        else:
            # Use thread-based parallel processing
            results = process_pairs_parallel(pairs_to_check, cluster_info)

        # Apply the merge results
        for cid1, cid2, should_merge in results:
            if should_merge:
                union(cid1, cid2)

        print(f"Completed {len(results)} AI evaluations")

    return {cid: find(cid) for cid in parent}

# --- OUTPUT BUILDERS (unchanged) ---

def rebuild_merged_clusters(
    df_name: pd.DataFrame,
    merged_map: dict[int, int]
) -> pd.DataFrame:
    """
    Collapse each union-find set into:
      – Names = all distinct original names
      – AllRecIDs = all RecIDs from every merged cluster
    Returns a DataFrame with columns ["RootID", "Names", "AllRecIDs"].
    """
    temp: dict[int, dict[str, set | list]] = defaultdict(lambda: {
        "Names": set(),
        "AllRecIDs": []
    })

    for _, row in df_name.iterrows():
        orig_cid = int(row["ClusterID"])
        root = merged_map[orig_cid]
        temp[root]["Names"].add(row["Name"])
        rec_list = [r.strip() for r in row["RecIDs"].split(";") if r.strip()]
        temp[root]["AllRecIDs"].extend(rec_list)

    merged_rows = []
    for root, info in temp.items():
        merged_rows.append({
            "RootID": root,
            "Names": sorted(list(info["Names"])),
            "AllRecIDs": sorted(list(set(info["AllRecIDs"])))
        })
    return pd.DataFrame(merged_rows)

def group_by_address_for_merged(
    df_addr: pd.DataFrame,
    merged_clusters_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Re-groups the final merged clusters by address to produce the final output format.
    """
    recid_to_addr: dict[str, str] = {}
    for _, row in df_addr.iterrows():
        addr = row["Addresses"]
        recs = [r.strip() for r in row["RecIDs"].split(";") if r.strip()]
        for r in recs:
            recid_to_addr[r] = addr

    output_rows = []
    for _, merged_row in merged_clusters_df.iterrows():
        root = int(merged_row["RootID"])
        names = merged_row["Names"]
        all_recs = merged_row["AllRecIDs"]

        addr_to_recs: dict[str, list[str]] = defaultdict(list)
        for rec in all_recs:
            if rec in recid_to_addr:
                a = recid_to_addr[rec]
                addr_to_recs[a].append(rec)

        sorted_addrs = sorted(addr_to_recs.keys())
        bracketed_lists = []
        direct_links = []
        for a in sorted_addrs:
            recs_at_a = sorted(addr_to_recs[a])
            bracketed_lists.append(f"[{', '.join(recs_at_a)}]")
            n = len(recs_at_a)
            direct_links.append(str(n * (n - 1) // 2))

        output_rows.append({
            "OldClusterID": root,
            "Name": "; ".join(names),
            "RecIDs": "; ".join(bracketed_lists),
            "Addresses": "; ".join(sorted_addrs),
            "DirectLink": "; ".join(direct_links)
        })

    result_df = pd.DataFrame(output_rows)
    if not result_df.empty:
        old_ids = sorted(result_df["OldClusterID"].astype(int).tolist())
        id_map = {old: new for new, old in enumerate(old_ids)}
        result_df["ClusterID"] = result_df["OldClusterID"].map(id_map)
        result_df = result_df[["ClusterID", "Name", "RecIDs", "Addresses", "DirectLink"]]
    else:
        return pd.DataFrame(columns=["ClusterID", "Name", "RecIDs", "Addresses", "DirectLink"])

    return result_df

# --- MAIN ---

def main():
    """Main execution function."""
    try:
        # Display configuration
        print("=" * 60)
        print("STEP 8: Enhanced Merge with Optimized API Calls")
        print("=" * 60)
        print(f"Configuration:")
        print(f"  - Batch Processing: {'Enabled' if API_CONFIG['use_batch'] else 'Disabled'}")
        print(f"  - Async Processing: {'Enabled' if API_CONFIG['use_async'] else 'Disabled'}")
        print(f"  - Max Parallel Workers: {API_CONFIG['max_parallel_workers']}")
        print(f"  - Batch Size: {API_CONFIG['batch_size']}")
        print(f"  - Rate Limit Delay: {API_CONFIG['rate_limit_delay']}s")
        print("=" * 60)

        # Load input files
        df_name = pd.read_csv("g_grouped_by_name.csv")
        df_addr = pd.read_csv("g_grouped_by_address.csv")

        print(f"\nLoaded {len(df_name)} name clusters and {len(df_addr)} address groups")

        # Process merges
        start_time = time.time()
        merged_map = merge_clusters(df_name)
        merge_time = time.time() - start_time

        # Build output
        merged_clusters_df = rebuild_merged_clusters(df_name, merged_map)
        final_df = group_by_address_for_merged(df_addr, merged_clusters_df)

        # Save results
        final_df.to_csv("h_refined_clusters_new.csv", index=False)

        # Report statistics
        original_clusters = len(df_name)
        final_clusters = len(final_df)
        reduction = original_clusters - final_clusters

        print("\n" + "=" * 60)
        print("PROCESSING COMPLETE")
        print("=" * 60)
        print(f"Original clusters: {original_clusters}")
        print(f"Final clusters: {final_clusters}")
        print(f"Clusters merged: {reduction}")
        print(f"Reduction: {reduction/original_clusters*100:.1f}%")
        print(f"Processing time: {merge_time:.2f} seconds")
        print(f"\nOutput written to: h_refined_clusters_new.csv")
        print("=" * 60)

    except FileNotFoundError as e:
        print(f"Error: {e}")
        print("Please ensure 'g_grouped_by_name.csv' and 'g_grouped_by_address.csv' are in the correct directory.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    # Install required packages if needed
    try:
        import aiohttp
    except ImportError:
        print("Installing aiohttp for async operations...")
        os.system("pip install aiohttp")
        print("Please restart the script after installation.")
        exit(1)

    main()

STEP 8: Enhanced Merge with Optimized API Calls
Configuration:
  - Batch Processing: Enabled
  - Async Processing: Enabled
  - Max Parallel Workers: 10
  - Batch Size: 5
  - Rate Limit Delay: 0.1s

Loaded 728 name clusters and 638 address groups
Pre-computing name parts and address lookups...

Step 1: Applying deterministic rules for shared addresses...


Name Compatibility: 100%|██████████| 264628/264628 [00:00<00:00, 1470470.54it/s]



Step 2: Applying the 'bridge' logic for name/address connections...


Bridge Logic: 100%|██████████| 621/621 [00:00<00:00, 1038127.85it/s]



Step 3: Comparing remaining clusters using optimized AI calls...
Configuration: Batch=True, Async=True, Workers=10
Found 738 pairs requiring AI evaluation


Batch Processing: 100%|██████████| 148/148 [15:25<00:00,  6.25s/it]

Completed 738 AI evaluations

PROCESSING COMPLETE
Original clusters: 728
Final clusters: 713
Clusters merged: 15
Reduction: 2.1%
Processing time: 925.42 seconds

Output written to: h_refined_clusters_new.csv


In [ ]:
#Code needs to be streamlined
#Step 9 - Complete Async Entity Resolution with Enhanced Reasoning and Unified Numbering
import os
import re
import json
import difflib
import itertools
import pandas as pd
import asyncio
import aiohttp
from typing import List, Dict, Set, Tuple
import time

################################################################################
# LLM ADAPTER (OpenAI ONLY — async version)                                    #
################################################################################

class LLMDecision:
    def __init__(self, merge: bool, confidence: float, reasons: List[str], detailed_analysis: str = ""):
        self.merge = bool(merge)
        self.confidence = float(confidence)
        self.reasons = list(reasons)
        self.detailed_analysis = detailed_analysis

    def __repr__(self):
        return f"LLMDecision(merge={self.merge}, confidence={self.confidence:.2f})"


class AsyncOpenAIAdapter:
    """Async version using aiohttp for parallel API calls."""
    def __init__(self, model: str = "gpt-4o-mini", api_key_env: str = "OPENAI_API_KEY", max_concurrent: int = 5):
        api_key = os.getenv(api_key_env)
        if not api_key:
            raise SystemExit("OPENAI_API_KEY is not set. Please export it and re-run.")

        self.api_key = api_key
        self.model = model
        self.max_concurrent = max_concurrent
        self.base_url = "https://api.openai.com/v1/chat/completions"

    async def decide_batch(self, system: str, prompts: List[Tuple[str, str]]) -> List[Tuple[str, LLMDecision]]:
        """Process multiple prompts concurrently"""
        semaphore = asyncio.Semaphore(self.max_concurrent)

        async def process_single(prompt_data):
            pair_key, user_prompt = prompt_data
            async with semaphore:
                try:
                    decision = await self._single_request(system, user_prompt)
                    return (pair_key, decision)
                except Exception as e:
                    print(f"Error processing {pair_key}: {e}")
                    return (pair_key, LLMDecision(False, 0.0, [f"API Error: {e}"]))

        tasks = [process_single(prompt_data) for prompt_data in prompts]
        results = await asyncio.gather(*tasks)
        return results

    async def _single_request(self, system: str, user_prompt: str) -> LLMDecision:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": self.model,
            "temperature": 0.7,
            "response_format": {"type": "json_object"},
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user_prompt}
            ]
        }

        async with aiohttp.ClientSession() as session:
            async with session.post(self.base_url, headers=headers, json=payload) as response:
                if response.status != 200:
                    raise Exception(f"API request failed: {response.status}")

                result = await response.json()
                content = result["choices"][0]["message"]["content"] or "{}"
                data = json.loads(content)

                reasons = data.get("reasons", [])
                if isinstance(reasons, str):
                    reasons = [reasons]

                return LLMDecision(
                    merge=bool(data.get("merge", False)),
                    confidence=float(data.get("confidence", 0.5)),
                    reasons=list(reasons),
                    detailed_analysis=data.get("detailed_analysis", "")
                )

##############################
# Utilities for names/addresses/features
##############################

def first_middle_variants(name_str: str) -> Tuple[List[str], List[str]]:
    variants = [v.strip() for v in name_str.split(';') if v.strip()]
    first_names, keys = [], []
    for v in variants:
        parts = v.split()
        first = parts[0].upper() if parts else ""
        middle = parts[1].upper() if len(parts) > 1 else ""
        first_names.append(first)
        keys.append(f"{first} {middle}".strip())
    return first_names, keys

def last_names(name_str: str) -> Set[str]:
    out = set()
    for variant in name_str.split(";"):
        variant = variant.strip()
        parts = variant.split()
        if len(parts) >= 2:
            out.add(parts[-1].upper())
    return out

def make_addr_set(address_str: str) -> Set[str]:
    return {addr.strip().upper() for addr in address_str.split(";") if addr.strip()}

def make_name_tokens(name_str: str) -> Set[str]:
    tokens = set()
    for variant in name_str.split(";"):
        for word in re.findall(r"\b\w+\b", variant):
            w = word.upper()
            if len(w) > 1:
                tokens.add(w)
    return tokens

def looks_like_po_box(addr: str) -> bool:
    addr_u = addr.upper()
    return any(k in addr_u for k in [" PO BOX", " P.O.", "POST OFFICE", "BOX "])

def address_similarity_score(addr1: str, addr2: str) -> float:
    """Calculate similarity between two addresses considering various factors"""
    addr1_clean = re.sub(r'[^\w\s]', '', addr1.upper())
    addr2_clean = re.sub(r'[^\w\s]', '', addr2.upper())

    base_sim = difflib.SequenceMatcher(None, addr1_clean, addr2_clean).ratio()

    nums1 = set(re.findall(r'\b\d+\b', addr1_clean))
    nums2 = set(re.findall(r'\b\d+\b', addr2_clean))

    common_words = {'ST', 'STREET', 'AVE', 'AVENUE', 'RD', 'ROAD', 'DR', 'DRIVE',
                   'LN', 'LANE', 'CT', 'COURT', 'PL', 'PLACE', 'BLVD', 'BOULEVARD',
                   'WAY', 'APT', 'APARTMENT', 'UNIT', 'STE', 'SUITE'}
    words1 = set(addr1_clean.split()) - common_words
    words2 = set(addr2_clean.split()) - common_words

    if nums1 and nums2:
        if nums1 & nums2:
            base_sim += 0.2

    if words1 and words2:
        word_overlap = len(words1 & words2) / max(len(words1), len(words2))
        base_sim += word_overlap * 0.3

    return min(base_sim, 1.0)

def find_similar_addresses(addrs_a: Set[str], addrs_b: Set[str]) -> List[Tuple[str, str, float]]:
    """Find similar addresses between two sets with similarity scores"""
    similar = []
    for a1 in addrs_a:
        for a2 in addrs_b:
            if not (looks_like_po_box(a1) and looks_like_po_box(a2)):
                sim = address_similarity_score(a1, a2)
                if sim > 0.7:
                    similar.append((a1, a2, sim))
    return sorted(similar, key=lambda x: x[2], reverse=True)

#####################################
# Data enrichment & candidate pairs
#####################################

def enrich(df: pd.DataFrame) -> pd.DataFrame:
    df[["First", "Middle"]] = df["Name"].apply(
        lambda x: pd.Series(
            first_middle_variants(x)[1][0].split()[:2]
            if first_middle_variants(x)[1]
            else ["", ""]
        )
    )
    df["NameKey"] = (df["First"] + " " + df["Middle"]).str.strip()
    df["AddrSet"] = df["Addresses"].apply(make_addr_set)
    df["NameTokens"] = df["Name"].apply(make_name_tokens)
    df["LastNames"] = df["Name"].apply(last_names)
    return df

def build_address_index(df: pd.DataFrame) -> Dict[str, Set[int]]:
    idx: Dict[str, Set[int]] = {}
    for cid, adds in df[["ClusterID", "AddrSet"]].values:
        for a in adds:
            idx.setdefault(a, set()).add(cid)
    return idx

def find_candidate_pairs(df: pd.DataFrame, skip: Set[int]) -> List[Tuple[str, int, int]]:
    key_map: Dict[str, List[int]] = {}
    for _, row in df.iterrows():
        cid = row["ClusterID"]
        _, keys = first_middle_variants(row["Name"])
        for k in keys:
            if k:
                key_map.setdefault(k, []).append(cid)

    pairs = []
    for key, clist in key_map.items():
        uniq = sorted({c for c in clist if c not in skip})
        for a, b in itertools.combinations(uniq, 2):
            pairs.append((key, a, b))
    return pairs

################################
# Merge operation
################################

def merge_clusters(df: pd.DataFrame, merge_ids: List[int],
                  new_cluster_id: int, name_override: str = None) -> pd.DataFrame:
    to_merge = df[df["ClusterID"].isin(merge_ids)].copy()

    addr_lists: Dict[int, List[str]] = {}
    recid_lists: Dict[int, List[List[str]]] = {}

    for _, row in to_merge.iterrows():
        cid = int(row["ClusterID"])
        adds = [a.strip() for a in row["Addresses"].split(";")]
        addr_lists[cid] = adds

        groups: List[List[str]] = []
        for grp in str(row["RecIDs"]).split(";"):
            core = grp.strip().strip("[]")
            ids = [x.strip() for x in core.split(",") if x.strip()]
            groups.append(ids)
        recid_lists[cid] = groups

        if len(adds) != len(groups):
            raise ValueError(f"Cluster {cid} has {len(adds)} addresses but {len(groups)} recid-groups")

    combined_addrs: List[str] = []
    seen = set()
    for cid in merge_ids:
        for a in addr_lists[cid]:
            if a not in seen:
                combined_addrs.append(a)
                seen.add(a)

    combined_groups: List[List[str]] = []
    for addr in combined_addrs:
        grp: List[str] = []
        for cid in merge_ids:
            if addr in addr_lists[cid]:
                idx = addr_lists[cid].index(addr)
                grp.extend(recid_lists[cid][idx])
        combined_groups.append(grp)

    recid_str = "; ".join(f"[{', '.join(g)}]" for g in combined_groups)
    link_str = "; ".join(str(len(g) * (len(g) - 1) // 2) for g in combined_groups)

    if name_override:
        merged_name = name_override
    else:
        all_names = []
        for _, row in to_merge.iterrows():
            for v in str(row["Name"]).split(";"):
                v = v.strip()
                if v and v not in all_names:
                    all_names.append(v)
        merged_name = "; ".join(all_names)

    rest = df[~df["ClusterID"].isin(merge_ids)].copy()
    new_row = {
        "ClusterID": int(new_cluster_id),
        "Name": merged_name,
        "RecIDs": recid_str,
        "Addresses": "; ".join(combined_addrs),
        "DirectLink": link_str
    }
    out = pd.concat([rest, pd.DataFrame([new_row])], ignore_index=True)
    out["ClusterID"] = out["ClusterID"].astype(int)
    out = out.sort_values("ClusterID").reset_index(drop=True)
    return out[["ClusterID", "Name", "RecIDs", "Addresses", "DirectLink"]]

#############################
# Pair features & associates
#############################

def jaro_like(a: str, b: str) -> float:
    return difflib.SequenceMatcher(None, a, b).ratio()

def levenshtein(a: str, b: str) -> int:
    """Classic Levenshtein edit distance."""
    m, n = len(a), len(b)
    if m == 0: return n
    if n == 0: return m
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        ca = a[i-1]
        for j in range(1, n+1):
            cb = b[j-1]
            cost = 0 if ca == cb else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,      # delete
                dp[i][j-1] + 1,      # insert
                dp[i-1][j-1] + cost  # substitute
            )
    return dp[m][n]

def is_adjacent_transposition(a: str, b: str) -> bool:
    """True if b == a with exactly one adjacent character swap."""
    if len(a) != len(b):
        return False
    diffs = [(i, x, y) for i, (x, y) in enumerate(zip(a, b)) if x != y]
    if len(diffs) != 2:
        return False
    (i, ax, ay), (j, bx, by) = diffs
    return j == i+1 and ax == by and ay == bx

def last_name_similarity(set_a: Set[str], set_b: Set[str]) -> float:
    if not set_a or not set_b:
        return 0.0
    best = 0.0
    for la in set_a:
        for lb in set_b:
            best = max(best, jaro_like(la, lb))
    return best

def min_edit_info(set_a: Set[str], set_b: Set[str]) -> dict:
    """Compute best (min) edit distance and whether it's an adjacent transposition."""
    best = {
        "min_distance": None,
        "pair": ["", ""],
        "adjacent_transposition": False
    }
    if not set_a or not set_b:
        return best
    for la in set_a:
        for lb in set_b:
            d = levenshtein(la, lb)
            adj = is_adjacent_transposition(la, lb)
            if (best["min_distance"] is None) or (d < best["min_distance"]) or (d == best["min_distance"] and adj):
                best["min_distance"] = d
                best["pair"] = [la, lb]
                best["adjacent_transposition"] = adj
    return best

def summarize_associates(a_id: int, b_id: int, df: pd.DataFrame, addr_index: Dict[str, Set[int]]) -> dict:
    a_row = df[df["ClusterID"] == a_id].iloc[0]
    b_row = df[df["ClusterID"] == b_id].iloc[0]
    a_addrs = a_row["AddrSet"]
    b_addrs = b_row["AddrSet"]

    associates_a = set()
    associates_b = set()
    for ad in a_addrs:
        associates_a |= (addr_index.get(ad, set()) - {a_id, b_id})
    for ad in b_addrs:
        associates_b |= (addr_index.get(ad, set()) - {a_id, b_id})

    bridge_by_address = len(associates_a & associates_b) > 0

    ln_a = a_row["LastNames"]
    ln_b = b_row["LastNames"]

    assoc_rows: Dict[int, pd.Series] = {}
    for cid in associates_a | associates_b:
        assoc_rows[cid] = df[df["ClusterID"] == cid].iloc[0]

    def addr_overlap_count(s: Set[str], t: Set[str]) -> Tuple[int, List[str]]:
        inter = sorted(list(s & t))
        return (len(inter), inter[:3])

    # surname-bridge from A->B and B->A
    bridge_from_a = []
    for cid in associates_a:
        r = assoc_rows[cid]
        ln_info = min_edit_info(r["LastNames"], ln_b)
        if ln_info["min_distance"] is None:
            continue
        very_close = (ln_info["min_distance"] is not None and ln_info["min_distance"] <= 1) or ln_info["adjacent_transposition"]
        if very_close:
            cnt, shared = addr_overlap_count(r["AddrSet"], a_addrs)
            bridge_from_a.append({
                "cid": int(cid),
                "name": r["Name"],
                "associate_last_names": sorted(list(r["LastNames"])),
                "best_pair": ln_info["pair"],
                "min_distance": ln_info["min_distance"],
                "adjacent_transposition": ln_info["adjacent_transposition"],
                "shared_address_count_with_A": cnt,
                "example_shared_addresses_with_A": shared
            })

    bridge_from_b = []
    for cid in associates_b:
        r = assoc_rows[cid]
        ln_info = min_edit_info(r["LastNames"], ln_a)
        if ln_info["min_distance"] is None:
            continue
        very_close = (ln_info["min_distance"] is not None and ln_info["min_distance"] <= 1) or ln_info["adjacent_transposition"]
        if very_close:
            cnt, shared = addr_overlap_count(r["AddrSet"], b_addrs)
            bridge_from_b.append({
                "cid": int(cid),
                "name": r["Name"],
                "associate_last_names": sorted(list(r["LastNames"])),
                "best_pair": ln_info["pair"],
                "min_distance": ln_info["min_distance"],
                "adjacent_transposition": ln_info["adjacent_transposition"],
                "shared_address_count_with_B": cnt,
                "example_shared_addresses_with_B": shared
            })

    # intra-side same-surname support
    same_surname_support_a = []
    for cid in associates_a:
        r = assoc_rows[cid]
        if r["LastNames"] & ln_a:
            cnt, shared = addr_overlap_count(r["AddrSet"], a_addrs)
            same_surname_support_a.append({
                "cid": int(cid),
                "name": r["Name"],
                "shared_address_count_with_A": cnt,
                "example_shared_addresses_with_A": shared
            })

    same_surname_support_b = []
    for cid in associates_b:
        r = assoc_rows[cid]
        if r["LastNames"] & ln_b:
            cnt, shared = addr_overlap_count(r["AddrSet"], b_addrs)
            same_surname_support_b.append({
                "cid": int(cid),
                "name": r["Name"],
                "shared_address_count_with_B": cnt,
                "example_shared_addresses_with_B": shared
            })

    def describe(cid: int) -> str:
        r = assoc_rows[cid]
        return f"{cid}: {r['Name']}"

    examples_a = ", ".join(describe(cid) for cid in sorted(list(associates_a))[:3])
    examples_b = ", ".join(describe(cid) for cid in sorted(list(associates_b))[:3])

    return {
        "associates_a": sorted(list(associates_a)),
        "associates_b": sorted(list(associates_b)),
        "associates_examples_a": examples_a,
        "associates_examples_b": examples_b,
        "bridge_by_address": bridge_by_address,
        "surname_bridge_from_a": bridge_from_a[:4],
        "surname_bridge_from_b": bridge_from_b[:4],
        "same_surname_support_a": same_surname_support_a[:4],
        "same_surname_support_b": same_surname_support_b[:4],
    }

def build_pair_features(a_id: int, b_id: int, df: pd.DataFrame, addr_index: Dict[str, Set[int]]) -> dict:
    row_a = df[df["ClusterID"] == a_id].iloc[0]
    row_b = df[df["ClusterID"] == b_id].iloc[0]

    a_addrs = row_a["AddrSet"]
    b_addrs = row_b["AddrSet"]
    shared_addrs = sorted(list(a_addrs & b_addrs))
    similar_addrs = find_similar_addresses(a_addrs, b_addrs)

    po_flags = {
        "any_po_a": any(looks_like_po_box(x) for x in a_addrs),
        "any_po_b": any(looks_like_po_box(x) for x in b_addrs),
    }

    ln_sim_ratio = last_name_similarity(row_a["LastNames"], row_b["LastNames"])
    ln_edit = min_edit_info(row_a["LastNames"], row_b["LastNames"])

    assoc = summarize_associates(a_id, b_id, df, addr_index)

    return {
        "a": {
            "id": int(a_id),
            "name": row_a["Name"],
            "first_name": row_a["First"],
            "addresses": list(a_addrs),
            "last_names": sorted(list(row_a["LastNames"])),
        },
        "b": {
            "id": int(b_id),
            "name": row_b["Name"],
            "first_name": row_b["First"],
            "addresses": list(b_addrs),
            "last_names": sorted(list(row_b["LastNames"])),
        },
        "shared_addresses": shared_addrs,
        "shared_address_count": len(shared_addrs),
        "similar_addresses": similar_addrs,
        "last_name_similarity_ratio": ln_sim_ratio,
        "last_name_edit": ln_edit,
        "po_flags": po_flags,
        "associates": assoc,
    }

#############################################
# Enhanced Prompt with detailed reasoning
#############################################

SYSTEM_MSG = """You are an entity-resolution expert deciding if two clusters represent the SAME PERSON.
Make a firm decision each time. Be precise and pragmatic: merge when there is compelling evidence
OR multiple weaker signals that converge; skip otherwise.

REASONING PROCESS:
1. Analyze the names - are they consistent with the same person?
2. Look for name change patterns (e.g., marriage - names like Jennifer, Nancy, Brenda with surname changes)
3. Examine address connections - exact matches, high similarity scores
4. Consider supporting evidence from associates and surname bridges
5. Combine all signals to reach a decision

WEIGHTING SYSTEM:
- VERY HIGH: Exact shared residential addresses (not PO boxes)
- HIGH:
  * Name pattern suggesting marriage (same first/middle, different surname, consider if name sounds feminine)
  * Similar addresses (similarity > 0.85) especially residential
  * Surname-bridge via associate with multiple shared residential addresses
- MEDIUM:
  * Same FIRST+MIDDLE + high surname similarity (>0.90) + compatible geography
  * Adjacent transposition in surnames (likely typos)
  * Edit distance ≤1 in surnames with other supporting evidence
- LOW (supporting only):
  * Shared PO boxes only
  * City/ZIP co-location without address similarity

MARRIAGE NAME PATTERNS: When analyzing names, consider whether the first name suggests a person
who might change their surname through marriage. Same first/middle with different surnames can be
a STRONG positive signal if it fits this pattern.

ADDRESS SIMILARITY: Similar addresses carry significant weight. Consider apartment number changes,
formatting differences, etc.

DETAILED REASONING: Provide a "detailed_analysis" field that walks through your reasoning process
step-by-step in 3-5 sentences explaining the logical flow from evidence to conclusion.

Return STRICT JSON only:
{
  "merge": true|false,
  "confidence": 0..1,
  "reasons": ["4-10 specific bullets about THIS case"],
  "detailed_analysis": "Step-by-step reasoning explanation (3-5 sentences)"
}"""

FEWSHOT = """
Examples:

MERGE (Name change pattern):
{
  "merge": true,
  "confidence": 0.92,
  "reasons": [
    "Same first+middle ('Jennifer M') - exact match",
    "First name 'Jennifer' with different surnames ('SMITH' vs 'JOHNSON') suggests marriage name change",
    "High address similarity: '123 Oak St Apt 2' vs '123 Oak Street Unit 2' (0.94 similarity)",
    "Compatible geographic pattern across all addresses"
  ],
  "detailed_analysis": "The exact match on first and middle names establishes the baseline connection. The first name 'Jennifer' combined with different surnames (SMITH/JOHNSON) provides strong evidence for a marriage name change scenario. The very high address similarity (0.94) between primary addresses suggests these are the same physical location with minor formatting differences. Together, these signals strongly indicate a single person whose name changed through marriage."
}

SKIP (Insufficient evidence):
{
  "merge": false,
  "confidence": 0.75,
  "reasons": [
    "Same first+middle but no direct address matches",
    "Address similarity too low (<0.7) for reliable connection",
    "Different surnames with no marriage change indicator",
    "No surname-bridge via associates"
  ],
  "detailed_analysis": "While the first and middle names match, there are no strong connecting signals beyond this. The address similarity scores are all below 0.7, indicating different locations rather than formatting variations. The different surnames lack any marriage change indicators or typo patterns. Without address connections or name change evidence, the matching first/middle alone is insufficient for a merge decision."
}
""".strip()

def make_prompt(features: dict) -> str:
    # Enhanced feature payload
    similar_addrs_summary = []
    for addr1, addr2, sim in features["similar_addresses"][:3]:
        similar_addrs_summary.append({"addr1": addr1, "addr2": addr2, "similarity": sim})

    features_json = json.dumps({
        "a_id": features["a"]["id"],
        "b_id": features["b"]["id"],
        "a_first_name": features["a"]["first_name"],
        "b_first_name": features["b"]["first_name"],
        "first_middle_match": True,
        "shared_address_count": features["shared_address_count"],
        "shared_addresses": features["shared_addresses"],
        "similar_addresses": similar_addrs_summary,
        "last_name_similarity_ratio": features["last_name_similarity_ratio"],
        "last_name_edit": features["last_name_edit"],
        "a_last_names": features["a"]["last_names"],
        "b_last_names": features["b"]["last_names"],
        "po_flags": features["po_flags"],
        "associates": features["associates"],
    }, ensure_ascii=False)

    a_block = f"- {features['a']['id']}: Name = {features['a']['name']}\n        Addresses = " + "; ".join(features['a']['addresses'])
    b_block = f"- {features['b']['id']}: Name = {features['b']['name']}\n        Addresses = " + "; ".join(features['b']['addresses'])

    similar_block = ""
    if features["similar_addresses"]:
        similar_block = "Similar addresses found:\n"
        for addr1, addr2, sim in features["similar_addresses"][:3]:
            similar_block += f"   • '{addr1}' ↔ '{addr2}' (similarity: {sim:.2f})\n"

    assoc = features["associates"]
    assoc_text = []
    if assoc.get("associates_a"):
        assoc_text.append(f"   • A associates: {assoc['associates_examples_a']}")
    if assoc.get("associates_b"):
        assoc_text.append(f"   • B associates: {assoc['associates_examples_b']}")

    def summarize_bridges(kind: str):
        items = assoc.get(kind, [])
        lines = []
        for it in items[:3]:
            if "shared_address_count_with_A" in it:
                lines.append(
                    f"     - {it['cid']} ({it['name']}): ln_pair={it['best_pair']}, edit={it['min_distance']}, "
                    f"adj_transp={it['adjacent_transposition']}, shared_with_A={it['shared_address_count_with_A']} "
                    f"ex_addresses={it['example_shared_addresses_with_A']}"
                )
            else:
                lines.append(
                    f"     - {it['cid']} ({it['name']}): ln_pair={it['best_pair']}, edit={it['min_distance']}, "
                    f"adj_transp={it['adjacent_transposition']}, shared_with_B={it['shared_address_count_with_B']} "
                    f"ex_addresses={it['example_shared_addresses_with_B']}"
                )
        return "\n".join(lines) if lines else "     - (none)"

    def summarize_same_surname(kind: str):
        items = assoc.get(kind, [])
        lines = []
        for it in items[:3]:
            if "shared_address_count_with_A" in it:
                lines.append(
                    f"     - {it['cid']} ({it['name']}): same-surname intra-side; shared_with_A={it['shared_address_count_with_A']} "
                    f"ex_addresses={it['example_shared_addresses_with_A']}"
                )
            else:
                lines.append(
                    f"     - {it['cid']} ({it['name']}): same-surname intra-side; shared_with_B={it['shared_address_count_with_B']} "
                    f"ex_addresses={it['example_shared_addresses_with_B']}"
                )
        return "\n".join(lines) if lines else "     - (none)"

    body = f"""
{FEWSHOT}

=== Candidate pair (same FIRST+MIDDLE key) ===
 {a_block}
 {b_block}

{similar_block}

Associates (share an address with A/B):
{chr(10).join(assoc_text) if assoc_text else "   • (none detected)"}

Surname bridge from A's associates to B:
{summarize_bridges('surname_bridge_from_a')}

Surname bridge from B's associates to A:
{summarize_bridges('surname_bridge_from_b')}

Intra-side same-surname support (acknowledge as positive but not a bridge):
  A-side:
{summarize_same_surname('same_surname_support_a')}
  B-side:
{summarize_same_surname('same_surname_support_b')}

__FEATURES_BEGIN__
{features_json}
__FEATURES_END__

REMEMBER: Consider if first names suggest marriage name changes. Address similarity ≥0.85 carries HIGH weight.
Return STRICT JSON only.
""".strip()
    return body

########################
# Merge Candidate Class
########################

class MergeCandidate:
    def __init__(self, key: str, a_id: int, b_id: int, decision: LLMDecision, features: dict):
        self.key = key
        self.a_id = a_id
        self.b_id = b_id
        self.decision = decision
        self.features = features
        self.human_decision = None
        self.new_cluster_id = None
        self.merged_name = None

########################
# Display Functions
########################

def format_candidate_summary(candidate: MergeCandidate, df: pd.DataFrame, number: int) -> str:
    """Format a candidate for display with unified numbering"""
    row_a = df[df["ClusterID"] == candidate.a_id].iloc[0]
    row_b = df[df["ClusterID"] == candidate.b_id].iloc[0]

    decision_str = "MERGE" if candidate.decision.merge else "SKIP"
    conf_str = f"{candidate.decision.confidence:.2f}"

    shared_addrs = candidate.features.get("shared_addresses", [])
    similar_addrs = candidate.features.get("similar_addresses", [])

    addr_info = ""
    if shared_addrs:
        addr_info = f" [SHARED: {len(shared_addrs)} addrs]"
    elif similar_addrs:
        best_sim = max(sim for _, _, sim in similar_addrs) if similar_addrs else 0
        addr_info = f" [SIMILAR: {best_sim:.2f}]" if similar_addrs else ""

    name_change_hint = ""
    reasons_text = ' '.join(candidate.decision.reasons).lower()
    if 'marriage' in reasons_text or 'name change' in reasons_text:
        name_change_hint = " [POSSIBLE_NAME_CHANGE]"

    summary = f"""
[{number}] {candidate.key} - {decision_str} ({conf_str}){addr_info}{name_change_hint}
    A({candidate.a_id}): {row_a['Name']}
    B({candidate.b_id}): {row_b['Name']}
    Reasons: {'; '.join(candidate.decision.reasons[:2])}{'...' if len(candidate.decision.reasons) > 2 else ''}
"""
    return summary.strip()

def display_batch_results(candidates: List[MergeCandidate], df: pd.DataFrame) -> None:
    """Display all candidates with unified numbering for batch review"""
    print(f"\n{'='*60}")
    print(f"BATCH PROCESSING COMPLETE: {len(candidates)} total candidates")
    print(f"{'='*60}")

    for i, candidate in enumerate(candidates, 1):
        print(format_candidate_summary(candidate, df, i))
        if candidate.decision.detailed_analysis:
            print(f"    Analysis: {candidate.decision.detailed_analysis[:200]}...")

    merge_count = sum(1 for c in candidates if c.decision.merge)
    skip_count = len(candidates) - merge_count
    print(f"\n{'='*60}")
    print(f"SUMMARY: {merge_count} recommended MERGE, {skip_count} recommended SKIP")
    print(f"Recommended merges: {', '.join(str(i+1) for i, c in enumerate(candidates) if c.decision.merge)}")
    print(f"Recommended skips: {', '.join(str(i+1) for i, c in enumerate(candidates) if not c.decision.merge)}")
    print(f"{'='*60}")

def get_batch_decisions(candidates: List[MergeCandidate], df: pd.DataFrame) -> List[MergeCandidate]:
    """Get human decisions for all candidates with unified numbering"""

    if not candidates:
        print("\nNo candidates to review.")
        return []

    print(f"\n{'='*60}")
    print("BATCH DECISION INTERFACE")
    print(f"{'='*60}")
    print("\nEnter the numbers of candidates you want to MERGE (comma-separated)")
    print("Options:")
    print("  '1,3,5' - merge specific candidates by number")
    print("  'recommended' - accept all LLM-recommended merges")
    print("  'none' - skip all candidates (no merges)")
    print("  'review' - show all candidates again")
    print("  'detail 3' - show detailed analysis for candidate #3")
    print(f"\nTotal candidates: {len(candidates)}")

    recommended_indices = [i+1 for i, c in enumerate(candidates) if c.decision.merge]

    while True:
        response = input(f"\nEnter merge decisions: ").strip().lower()

        if response == 'recommended':
            selected_indices = recommended_indices
            break
        elif response == 'none':
            selected_indices = []
            break
        elif response == 'review':
            display_batch_results(candidates, df)
            continue
        elif response.startswith('detail '):
            try:
                idx = int(response.split()[1])
                if 1 <= idx <= len(candidates):
                    c = candidates[idx-1]
                    print(f"\n{'='*40}")
                    print(format_candidate_summary(c, df, idx))
                    print(f"\nDetailed Analysis:")
                    print(c.decision.detailed_analysis)
                    print(f"\nAll Reasons:")
                    for j, reason in enumerate(c.decision.reasons, 1):
                        print(f"  {j}. {reason}")
                    print(f"{'='*40}")
                else:
                    print(f"Invalid index: {idx}")
            except (ValueError, IndexError):
                print("Invalid command format. Use 'detail 3' to see details for candidate 3")
            continue
        else:
            try:
                if response:
                    selected_indices = [int(x.strip()) for x in response.split(',')]
                    invalid = [idx for idx in selected_indices if idx < 1 or idx > len(candidates)]
                    if invalid:
                        print(f"Invalid indices: {invalid}. Please use numbers 1-{len(candidates)}")
                        continue
                else:
                    selected_indices = []
                break
            except ValueError:
                print(f"Invalid input. Use numbers like '1,3,5' or commands: 'recommended', 'none', 'review', 'detail N'")
                continue

    confirmed = []
    for idx in selected_indices:
        candidate = candidates[idx-1]
        candidate.new_cluster_id = min(candidate.a_id, candidate.b_id)

        row_a = df[df["ClusterID"] == candidate.a_id].iloc[0]
        row_b = df[df["ClusterID"] == candidate.b_id].iloc[0]

        all_names = []
        for nm in (str(row_a["Name"]) + ";" + str(row_b["Name"])).split(";"):
            nm = nm.strip()
            if nm and nm not in all_names:
                all_names.append(nm)
        candidate.merged_name = "; ".join(all_names)

        candidate.human_decision = "merge"
        confirmed.append(candidate)

    not_selected = [i for i in range(1, len(candidates)+1) if i not in selected_indices]

    print(f"\n{'='*60}")
    print(f"DECISION SUMMARY:")
    print(f"  Merging: {len(confirmed)} candidates - {selected_indices if selected_indices else 'none'}")
    print(f"  Skipping: {len(not_selected)} candidates - {not_selected if not_selected else 'none'}")
    print(f"{'='*60}")

    return confirmed

########################
# Async Batch Processing
########################

async def process_batch(df: pd.DataFrame, pairs: List[Tuple[str, int, int]],
                       addr_index: Dict[str, Set[int]], llm: AsyncOpenAIAdapter) -> List[MergeCandidate]:
    """Process a batch of pairs asynchronously"""
    print(f"Processing {len(pairs)} pairs asynchronously...")

    prompts = []
    features_map = {}

    for key, a_id, b_id in pairs:
        features = build_pair_features(a_id, b_id, df, addr_index)
        prompt = make_prompt(features)
        pair_key = f"{key}|{a_id}|{b_id}"
        prompts.append((pair_key, prompt))
        features_map[pair_key] = features

    start_time = time.time()
    results = await llm.decide_batch(SYSTEM_MSG, prompts)
    elapsed = time.time() - start_time

    print(f"Completed {len(results)} LLM decisions in {elapsed:.1f}s")

    candidates = []
    for pair_key, decision in results:
        key, a_id, b_id = pair_key.split("|")
        a_id, b_id = int(a_id), int(b_id)
        features = features_map[pair_key]

        candidate = MergeCandidate(key, a_id, b_id, decision, features)
        candidates.append(candidate)

    return candidates

########################
# Main Function
########################

async def main():
    INPUT = "h_refined_clusters_new.csv"
    OUTPUT = "i_refined_clusters_merged.csv"

    print("Loading data...")
    df = pd.read_csv(INPUT, dtype=str)
    df["ClusterID"] = df["ClusterID"].astype(int)

    llm = AsyncOpenAIAdapter(max_concurrent=10)

    skipped: Set[int] = set()
    pass_num = 1

    while True:
        print(f"\n{'='*20} Pass {pass_num} {'='*20}")
        df = enrich(df)
        addr_index = build_address_index(df)
        pairs = find_candidate_pairs(df, skipped)

        if not pairs:
            print("\nNo more candidate pairs.")
            break

        candidates = await process_batch(df, pairs, addr_index, llm)
        display_batch_results(candidates, df)
        confirmed_merges = get_batch_decisions(candidates, df)

        if not confirmed_merges:
            print("\nNo merges confirmed in this pass.")
            for _, a, b in pairs:
                skipped.update([a, b])
            pass_num += 1
            continue

        merged_this_pass = False
        for candidate in confirmed_merges:
            try:
                df = merge_clusters(df, [candidate.a_id, candidate.b_id],
                                 candidate.new_cluster_id, candidate.merged_name)
                print(f"✓ Merged clusters {candidate.a_id} & {candidate.b_id} into {candidate.new_cluster_id}")
                merged_this_pass = True
            except Exception as e:
                print(f"✗ Error merging {candidate.a_id} & {candidate.b_id}: {e}")

        if merged_this_pass:
            print(f"\nCompleted {len(confirmed_merges)} merges in pass {pass_num}")

        merged_ids = set()
        for candidate in confirmed_merges:
            merged_ids.update([candidate.a_id, candidate.b_id])

        for _, a, b in pairs:
            if a not in merged_ids and b not in merged_ids:
                skipped.update([a, b])

        pass_num += 1

    df.to_csv(OUTPUT, index=False)
    print(f"\n✅ Final results written to {OUTPUT}")
    print(f"Total passes completed: {pass_num - 1}")


await main()

Loading data...

==================== Pass 1 ====================
Processing 20 pairs asynchronously...
Completed 20 LLM decisions in 11.3s

BATCH PROCESSING COMPLETE: 20 total candidates
[1] JAMES ABBOTT - SKIP (0.76) [SIMILAR: 0.84] [POSSIBLE_NAME_CHANGE]
    A(78): JAMES ABBOTT WALKER
    B(79): JAMES ABBOTT WILSON
    Reasons: Same first+middle ('JAMES ABBOTT') - exact match; Different surnames ('WALKER' vs 'WILSON') with no marriage change indicator...
    Analysis: The exact match on first and middle names establishes a connection, but the differing surnames lack evidence of a marriage name change. The address similarity of 0.84, while high, does not meet the th...
[2] STEVEN ABEE - SKIP (0.68)
    A(166): STEVEN ABEE KERR
    B(167): STEVEN ABEE NEIL
    Reasons: Same first+middle name ('STEVEN ABEE') but last names are very different ('KERR' vs 'NEIL'); No shared addresses or high similarity in addresses to indicate they live at the same location...
    Analysis: Although both 

In [ ]:
import pandas as pd
import itertools
import re

def normalize_and_export(
    input_path: str,
    normalized_path: str = "i_refined_clusters_merged.csv",
    households_path: str = "j_households_new.csv",
) -> None:
    df = pd.read_csv(input_path, dtype=str)
    df = df.drop(columns=['First', 'Middle', 'NameKey', 'AddrSet', 'NameTokens', 'LastNames', 'FirstLikelyGender'], errors='ignore')

    overall_sum = 0
    if 'DirectLink' in df.columns:
        def _sum_directlink(s: str) -> int:
            if pd.isna(s): return 0
            return sum(int(n) for n in re.findall(r'-?\d+', str(s)))
        df['DirectLink_Sum'] = df['DirectLink'].apply(_sum_directlink)
        overall_sum = int(df['DirectLink_Sum'].sum())

    df['ClusterID'] = df['ClusterID'].astype(int)
    df = df.sort_values('ClusterID').reset_index(drop=True)
    df['ClusterID'] = df.index
    df.to_csv(normalized_path, index=False)

    cluster_map = {}
    for _, row in df.iterrows():
        addrs = [a.strip() for a in str(row['Addresses']).split(';')]
        recid_groups = []
        for grp in str(row['RecIDs']).split(';'):
            cleaned = grp.strip().strip('[]')
            recid_groups.append([x.strip() for x in cleaned.split(',') if x.strip()])
        cluster_map[row['ClusterID']] = list(zip(addrs[:len(recid_groups)], recid_groups[:len(addrs)]))

    addr_map = {}
    for _, row in df.iterrows():
        cid = row['ClusterID']
        name = row.get('Name', '')
        for addr, recids in cluster_map.get(cid, []):
            addr_map.setdefault(addr, []).append((cid, name, recids))

    households = []
    h_id = 1
    for address, entries in addr_map.items():
        if len(entries) < 2:
            continue
        for (cid1, name1, rec1), (cid2, name2, rec2) in itertools.combinations(entries, 2):
            a, b = ((cid1, name1, rec1), (cid2, name2, rec2)) if len(rec1) <= len(rec2) else ((cid2, name2, rec2), (cid1, name1, rec1))
            households.append({
                'HouseholdId': h_id,
                'ClusterID_a': a[0],
                'Name_a': a[1],
                'RecIDs_a': ';'.join(a[2]),
                'ClusterID_b': b[0],
                'Name_b': b[1],
                'RecIDs_b': ';'.join(b[2]),
                'Address': address
            })
            h_id += 1

    pd.DataFrame(households).to_csv(households_path, index=False)
    print(f"Wrote {normalized_path} and {households_path}. DirectLink overall sum: {overall_sum}")

if __name__ == "__main__":
    normalize_and_export("i_refined_clusters_merged.csv")


Wrote i_refined_clusters_merged.csv and j_households_new.csv. DirectLink overall sum: 6670


In [ ]:
#this code is to make any changes needed
import pandas as pd

def edit_cluster_data(file_path='i_refined_clusters_merged.csv'):
    """
    This program allows a user to edit data in a CSV file based on a ClusterID.
    """
    try:
        # Read the CSV file into a pandas DataFrame
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return

    while True:
        # Prompt the user for a ClusterID
        try:
            cluster_id_input = input("Please enter the ClusterID you want to edit (or 'q' to quit): ")
            if cluster_id_input.lower() == 'q':
                break

            cluster_id = int(cluster_id_input)

            # Find the row with the matching ClusterID
            if cluster_id not in df['ClusterID'].values:
                print(f"ClusterID {cluster_id} not found. Please try again.")
                continue

            # Get the index of the row to edit
            row_index = df[df['ClusterID'] == cluster_id].index[0]

            print(f"\nEditing data for ClusterID: {cluster_id}")
            print("="*30)

            # Iterate over each column for the selected row
            for col in df.columns:
                if col == 'ClusterID': # Don't allow editing of ClusterID
                    continue

                current_value = df.at[row_index, col]
                print(f"\nCurrent column: '{col}'")
                print(f"Current value: {current_value}")

                # Ask the user if they want to edit this column
                edit_choice = input(f"Do you want to edit the '{col}' column? (yes/no): ").lower()

                if edit_choice in ['yes', 'y']:
                    # Get the new value from the user
                    new_value = input(f"Enter the new value for '{col}': ")
                    df.at[row_index, col] = new_value
                    print(f"'{col}' has been updated.")
                else:
                    print(f"Skipping '{col}'.")

            # Save the updated DataFrame back to the CSV file
            df.to_csv(file_path, index=False)
            print("\n" + "="*30)
            print("All changes have been saved to the file.")
            print("="*30 + "\n")


        except ValueError:
            print("Invalid input. Please enter a numerical ClusterID.")
        except Exception as e:
            print(f"An error occurred: {e}")

if __name__ == "__main__":
    edit_cluster_data()

Please enter the ClusterID you want to edit (or 'q' to quit): q


In [ ]:
#Step 11
import pandas as pd

# --- Load data
households_df = pd.read_csv('j_households_new.csv', sep=',')

# --- Fill missing so string operations don't blow up
households_df['RecIDs_a'] = households_df['RecIDs_a'].fillna('')
households_df['RecIDs_b'] = households_df['RecIDs_b'].fillna('')
households_df['Address']   = households_df['Address'].fillna('')

# --- Build an unordered pair key so (a, b) and (b, a) collapse into one group
pair_keys = households_df.apply(
    lambda r: tuple(sorted((r['ClusterID_a'], r['ClusterID_b']))),
    axis=1
)
pair_groups = households_df.groupby(pair_keys)

movements = []

for (c1, c2), group in pair_groups:
    # only interested in pairs that share at least 2 addresses
    if len(group) < 2:
        continue

    # figure out each cluster's name from the first row
    first = group.iloc[0]
    name1 = first['Name_a'] if first['ClusterID_a'] == c1 else first['Name_b']
    name2 = first['Name_a'] if first['ClusterID_a'] == c2 else first['Name_b']

    row = {
        'ClusterID_a': c1,
        'Name_a':      name1,
        'ClusterID_b': c2,
        'Name_b':      name2,
        'ClusterID_c': '',
        'Name_c':      ''
    }

    direct_links = []
    total1 = total2 = 0

    # for up to four shared-address records, extract RecIDs and compute links
    for idx in range(4):
        rec_key = f'RecIDs_{idx+1}'
        addr_key = f'Address_{idx+1}'

        if idx < len(group):
            r = group.iloc[idx]
            # get this row's RecIDs for cluster1 and cluster2 in the sorted order
            recs1 = r['RecIDs_a'] if r['ClusterID_a'] == c1 else r['RecIDs_b']
            recs2 = r['RecIDs_a'] if r['ClusterID_a'] == c2 else r['RecIDs_b']

            row[rec_key] = f"[{recs1}];[{recs2}]"
            row[addr_key] = r['Address']

            # count direct links = n*(n−1)/2 for each side
            list1 = [x.strip() for x in str(recs1).split(';') if x.strip()]
            list2 = [x.strip() for x in str(recs2).split(';') if x.strip()]
            n1 = len(list1)
            n2 = len(list2)
            direct_links.append(f"[{n1*(n1-1)//2},{n2*(n2-1)//2}]")
            total1 += n1
            total2 += n2
        else:
            row[rec_key] = ''
            row[addr_key] = ''

    # aggregate direct- and indirect-links
    row['Direct_Links']   = ";".join(direct_links)
    row['Indirect_Links'] = f"[{total1*(total1-1)//2}];[{total2*(total2-1)//2}]"

    movements.append(row)

# --- Build final DataFrame
movements_df = pd.DataFrame(movements)
movements_df.insert(0, 'HouseholdMovementsID', range(len(movements_df)))

# --- Reorder to your specification
cols = [
    'HouseholdMovementsID',
    'ClusterID_a', 'Name_a',
    'ClusterID_b', 'Name_b',
    'ClusterID_c', 'Name_c',
    'RecIDs_1', 'Address_1',
    'RecIDs_2', 'Address_2',
    'RecIDs_3', 'Address_3',
    'RecIDs_4', 'Address_4',
    'Direct_Links',
    'Indirect_Links'
]
movements_df = movements_df[cols]

# --- Write out
movements_df.to_csv('j_household_movements_new.csv', index=False)
print(f"✓ Generated j_household_movements_new.csv with {len(movements_df)} rows")


✓ Generated j_household_movements_new.csv with 134 rows


In [ ]:
#Step 12
import pandas as pd
import numpy as np
import re
from sklearn.metrics import (
    pair_confusion_matrix,
    adjusted_rand_score,
    rand_score,
    fowlkes_mallows_score,
    homogeneity_score,
    completeness_score,
    v_measure_score,
    normalized_mutual_info_score,
)

# -------- Config --------
INPUT_CSV = "i_refined_clusters_merged.csv"
OUTPUT_CSV = "cluster_eval_metrics.csv"  # Uncomment to save results

# -------- Load & expand labels --------
df_raw = pd.read_csv(INPUT_CSV)

records = []
for _, row in df_raw.iterrows():
    pred = str(row["ClusterID"]).strip()
    recids = re.findall(r"\d+\.\d+", str(row["RecIDs"]))
    for rid in recids:
        true = int(rid.split(".")[0])  # integer part is the true cluster ID
        records.append({"true": true, "pred": pred})

if not records:
    raise ValueError("No records found. Check that 'RecIDs' contains patterns like '123.1; 123.2; ...'")

df = pd.DataFrame(records)

# -------- Pairwise confusion counts --------
tn, fp, fn, tp = pair_confusion_matrix(df["true"], df["pred"]).ravel()

# -------- Pairwise metrics --------
def safe_div(num, den):
    return (num / den) if den else 0.0

pair_precision = safe_div(tp, tp + fp)
pair_recall    = safe_div(tp, tp + fn)
pair_accuracy  = safe_div(tp + tn, tp + tn + fp + fn)
pair_f1        = safe_div(2 * pair_precision * pair_recall, pair_precision + pair_recall)

# -------- Clustering indices --------
ari  = adjusted_rand_score(df["true"], df["pred"])
ri   = rand_score(df["true"], df["pred"])
fmi  = fowlkes_mallows_score(df["true"], df["pred"])
homo = homogeneity_score(df["true"], df["pred"])
compl = completeness_score(df["true"], df["pred"])
vmeasure = v_measure_score(df["true"], df["pred"])
nmi = normalized_mutual_info_score(df["true"], df["pred"])

# Purity: assign each predicted cluster to the most frequent true label
contingency = pd.crosstab(df["true"], df["pred"])
purity = np.sum(np.max(contingency.values, axis=0)) / contingency.values.sum()

# -------- Summarize --------
results = pd.DataFrame({
    "Metric": [
        "Pairwise Precision", "Pairwise Recall", "Pairwise Accuracy", "Pairwise F1",
        "Adjusted Rand Index", "Rand Index", "Fowlkes–Mallows Index",
        "Homogeneity", "Completeness", "V-Measure",
        "Normalized Mutual Information", "Purity"
    ],
    "Value": [
        pair_precision, pair_recall, pair_accuracy, pair_f1,
        ari, ri, fmi,
        homo, compl, vmeasure,
        nmi, purity
    ],
})

print("Pairwise counts (TN, FP, FN, TP):", (tn, fp, fn, tp))
print()
print(results.to_string(index=False))

# Optional: save results
# results.to_csv(OUTPUT_CSV, index=False)
# print(f"\nSaved metrics to {OUTPUT_CSV}")


Pairwise counts (TN, FP, FN, TP): (np.int64(24936632), np.int64(284), np.int64(2870), np.int64(55214))

                       Metric    Value
           Pairwise Precision 0.994883
              Pairwise Recall 0.950589
            Pairwise Accuracy 0.999874
                  Pairwise F1 0.972232
          Adjusted Rand Index 0.972168
                   Rand Index 0.999874
        Fowlkes–Mallows Index 0.972484
                  Homogeneity 0.998524
                 Completeness 0.988867
                    V-Measure 0.993672
Normalized Mutual Information 0.993672
                       Purity 0.996600


In [ ]:
#!/usr/bin/env python3
import pandas as pd

def parse_recids(recids_cell: str) -> list[str]:
    """
    Given a cell like "[1.5]; [1.11, 1.4, 1.8]; [1.10]; ...",
    return a flat list of RecID strings.
    """
    recid_lists = recids_cell.split(';')
    recids = []
    for part in recid_lists:
        # strip brackets and whitespace
        part = part.strip().lstrip('[').rstrip(']')
        if not part:
            continue
        # split on commas
        for rid in part.split(','):
            rid = rid.strip()
            if rid:
                recids.append(rid)
    return recids

def recid_key(r: str) -> tuple[int,int]:
    """
    Convert RecID string 'X.Y' into a tuple (X, Y) of ints for proper numeric sorting.
    """
    major, minor = r.split('.', 1)
    return (int(major), int(minor))

def build_refindex(input_csv: str, output_txt: str):
    # Load the merged clusters file
    df = pd.read_csv(input_csv, dtype=str)

    rows = []
    for _, row in df.iterrows():
        all_recids = parse_recids(row['RecIDs'])
        # find the smallest RecID by numeric (major, minor)
        min_recid = min(all_recids, key=recid_key)
        # map each RecID to that min_recid
        for rid in all_recids:
            rows.append({'RefID': rid, 'ClusterID': min_recid})

    out_df = pd.DataFrame(rows, columns=['RefID','ClusterID'])
    out_df.to_csv(output_txt, index=False)

if __name__ == '__main__':
    build_refindex('i_refined_clusters_merged.csv', 'z_LinkIndex_Taha.txt')


In [ ]:
#!/usr/bin/env python3
import pandas as pd
import re

def parse_recids(cell: str) -> list[str]:
    """
    Given a RecIDs cell like "[1.5]; [1.11, 1.4, 1.8]; [1.10]; …",
    returns a flat list of RecID strings in order:
      ['1.5','1.11','1.4','1.8','1.10', …]
    """
    parts = cell.split(';')
    recids = []
    for part in parts:
        # remove brackets and surrounding whitespace
        part = part.strip().lstrip('[').rstrip(']')
        if not part:
            continue
        # split on commas
        for rid in part.split(','):
            rid = rid.strip()
            if rid:
                recids.append(rid)
    return recids

def build_linked_pairs(input_csv: str, output_txt: str):
    # 1) Load the merged clusters file
    df = pd.read_csv(input_csv, dtype=str)

    rows = []
    # 2) For each cluster, parse and link adjacent RecIDs
    for cell in df['RecIDs']:
        recids = parse_recids(cell)
        for a, b in zip(recids, recids[1:]):
            rows.append({'recid1': a, 'recid2': b})

    # 3) Save as a two‐column CSV (or .txt)
    out = pd.DataFrame(rows, columns=['recid1','recid2'])
    out.to_csv(output_txt, index=False)

if __name__ == '__main__':
    build_linked_pairs(
        input_csv='i_refined_clusters_merged.csv',
        output_txt='z_LinkedPair_Taha.txt'
    )

In [ ]:
#!/usr/bin/env python3
import pandas as pd

def recid_key(r: str) -> tuple[int,int]:
    """
    Convert a RecID like 'X.Y' into a tuple (X, Y) of ints for proper sorting.
    """
    major, minor = r.split('.', 1)
    return (int(major), int(minor))


def build_truthfile(input_csv: str, output_txt: str):
    # 1) Load the concatenated data (assuming comma-separated)
    df = pd.read_csv(input_csv, dtype=str).fillna('')

    # 2) Extract the group key (the part before the first dot)
    # Use named args to avoid positional split error
    df['group'] = df['RecID'].str.split(pat='.', n=1).str[0]

    # 3) Compute the smallest RecID per group using numeric sort
    truth_map = (
        df
        .groupby('group')['RecID']
        .agg(lambda recs: min(recs.tolist(), key=recid_key))
        .to_dict()
    )

    # 4) Map each RecID to its group's IdTruth
    df['IdTruth'] = df['group'].map(truth_map)

    # 5) Write out only RecID and IdTruth
    df[['RecID', 'IdTruth']].to_csv(output_txt, index=False)

if __name__ == '__main__':
    build_truthfile(
        input_csv='a_concatenated.csv',
        output_txt='z_Truthfile_Taha.txt'
    )


In [ ]:
#!/usr/bin/env python
# coding: utf-8

"""
Advanced Link Evaluator System v2.1 - FIXED
A comprehensive entity resolution evaluation framework with enhanced metrics and reporting.
Author: Expert Systems Team
FIXES: Standard mode data parsing and metrics calculation
"""

import sys
import time
import datetime
import re
import json
import logging
from typing import Dict, List, Tuple, Set, Optional, Any
from dataclasses import dataclass, field
from collections import defaultdict, Counter
from pathlib import Path
import csv

# ============================================================================
# Configuration and Constants
# ============================================================================

VERSION = "2.1"
LOG_FORMAT = "%(asctime)s - %(levelname)s - %(message)s"
DATE_FORMAT = "%Y-%m-%d %H:%M:%S"

# ============================================================================
# Data Classes for Better Type Safety
# ============================================================================

@dataclass
class ClusterData:
    """Represents a cluster with its ID and member records."""
    cluster_id: str
    record_ids: List[str] = field(default_factory=list)

@dataclass
class EvaluationMetrics:
    """Comprehensive metrics for entity resolution evaluation."""
    true_positives: float = 0
    false_positives: float = 0
    false_negatives: float = 0
    precision: float = 0
    recall: float = 0
    f1_score: float = 0
    pairwise_precision: float = 0
    pairwise_recall: float = 0
    pairwise_f1: float = 0
    cluster_precision: float = 0
    cluster_recall: float = 0
    cluster_f1: float = 0
    homogeneity: float = 0
    completeness: float = 0
    v_measure: float = 0
    adjusted_rand_index: float = 0
    mutual_info_score: float = 0
    fowlkes_mallows_index: float = 0

# ============================================================================
# Enhanced File Parser Class - FIXED
# ============================================================================

class FileParser:
    """Advanced file parser with support for multiple formats - FIXED."""

    @staticmethod
    def parse_link_index(filepath: str, logger: logging.Logger) -> Tuple[List[Tuple[str, str]], List[str], Dict[str, str]]:
        """
        Parse a link index file (RefID, ClusterID format).
        Returns: (list of pairs, list of unique RefIDs, dict mapping RefID to ClusterID)
        FIXED: Now properly creates pairs from clusters instead of wrong approach
        """
        logger.info(f"Parsing link index file: {filepath}")
        pairs = []
        ref_ids = []
        cluster_dict = defaultdict(list)
        ref_to_cluster = {}

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                # Try to detect the delimiter
                first_line = f.readline().strip()
                f.seek(0)  # Reset to beginning

                # Check if it's comma or tab separated
                if '\t' in first_line:
                    delimiter = '\t'
                else:
                    delimiter = ','

                reader = csv.DictReader(f, delimiter=delimiter, skipinitialspace=True)

                # Handle different column name variations
                for row in reader:
                    # Try different column name variations
                    ref_id = (row.get('RefID') or row.get('refID') or
                             row.get('Ref ID') or row.get('RecID') or '').strip()
                    cluster_id = (row.get('ClusterID') or row.get('clusterID') or
                                row.get('Cluster ID') or row.get('TruthID') or
                                row.get('Truth ID') or '').strip()

                    if ref_id and cluster_id:
                        ref_ids.append(ref_id)
                        cluster_dict[cluster_id].append(ref_id)
                        ref_to_cluster[ref_id] = cluster_id

            # FIXED: Generate pairs correctly - all pairs within each cluster
            logger.info("Generating pairs from link index clusters...")
            for cluster_id, members in cluster_dict.items():
                # Generate all pairs within this cluster (including self-pairs)
                for i in range(len(members)):
                    for j in range(i, len(members)):  # Include i==j for self-pairs
                        pairs.append((members[i], members[j]))
                        # Also add reverse pair if different
                        if i != j:
                            pairs.append((members[j], members[i]))

            logger.info(f"Parsed {len(ref_ids)} references in {len(cluster_dict)} clusters")
            logger.info(f"Generated {len(pairs)} pairs from link index")
            return pairs, ref_ids, ref_to_cluster

        except Exception as e:
            logger.error(f"Error parsing link index file: {e}")
            raise

    @staticmethod
    def parse_linked_pairs(filepath: str, logger: logging.Logger) -> List[Tuple[str, str]]:
        """
        Parse a linked pairs file (RefID1, RefID2 format).
        Returns: list of pairs
        FIXED: Better parsing and bidirectional pairs
        """
        logger.info(f"Parsing linked pairs file: {filepath}")
        pairs = []

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                # Try to detect if first line is header
                first_line = f.readline().strip()
                f.seek(0)

                # Skip header if it contains non-numeric data or common header words
                if any(word in first_line.lower() for word in ['refid', 'id1', 'id2', 'ref_id']):
                    f.readline()  # Skip header
                else:
                    f.seek(0)  # Reset if first line might be data

                # Read pairs
                for line_num, line in enumerate(f, 2):
                    line = line.strip()
                    if not line:
                        continue

                    # Handle different delimiters
                    if '\t' in line:
                        parts = line.split('\t')
                    else:
                        parts = line.split(',')

                    if len(parts) >= 2:
                        ref_id1 = parts[0].strip().strip('"').strip("'")
                        ref_id2 = parts[1].strip().strip('"').strip("'")

                        if ref_id1 and ref_id2:
                            # Add both directions for symmetry
                            pairs.append((ref_id1, ref_id2))
                            pairs.append((ref_id2, ref_id1))

            logger.info(f"Parsed {len(pairs)} linked pairs (including bidirectional)")
            return pairs

        except Exception as e:
            logger.error(f"Error parsing linked pairs file: {e}")
            raise

    @staticmethod
    def parse_truth_file(filepath: str, ref_ids: List[str], logger: logging.Logger) -> Tuple[List[Tuple[str, str]], Dict[str, str]]:
        """
        Parse a truth file and create truth pairs.
        Returns: (list of truth pairs, dict mapping RefID to TruthID)
        FIXED: Better truth pair generation with enhanced parsing
        """
        logger.info(f"Parsing truth file: {filepath}")
        truth_dict = {}
        truth_pairs = []

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                # Read first few lines to analyze format
                lines = []
                for _ in range(5):
                    line = f.readline().strip()
                    if line:
                        lines.append(line)
                f.seek(0)  # Reset

                if not lines:
                    logger.error("Truth file is empty")
                    return [], {}

                # Analyze format
                logger.info("Analyzing truth file format...")
                for i, line in enumerate(lines[:3]):
                    logger.info(f"  Line {i+1}: {line}")

                # Try to detect delimiter
                first_line = lines[0]
                if '\t' in first_line:
                    delimiter = '\t'
                    logger.info("Detected tab delimiter")
                elif ',' in first_line:
                    delimiter = ','
                    logger.info("Detected comma delimiter")
                else:
                    # Try space or assume comma
                    delimiter = ' ' if ' ' in first_line else ','
                    logger.info(f"Defaulting to '{delimiter}' delimiter")

                # Try CSV reader first
                try:
                    f.seek(0)
                    reader = csv.DictReader(f, delimiter=delimiter, skipinitialspace=True)
                    headers = reader.fieldnames
                    logger.info(f"CSV headers detected: {headers}")

                    # Read truth assignments
                    for row_num, row in enumerate(reader, 2):
                        # Try different column name variations
                        ref_id = None
                        truth_id = None

                        # Try to find RefID column
                        for key in row.keys():
                            if key and any(term in key.lower() for term in ['refid', 'ref_id', 'recid', 'rec_id', 'id']):
                                ref_id = row[key].strip().strip('"').strip("'") if row[key] else None
                                break

                        # Try to find TruthID column
                        for key in row.keys():
                            if key and any(term in key.lower() for term in ['truthid', 'truth_id', 'clusterid', 'cluster_id', 'truth', 'cluster']):
                                truth_id = row[key].strip().strip('"').strip("'") if row[key] else None
                                break

                        # If still no columns found, try by position
                        if not ref_id or not truth_id:
                            values = list(row.values())
                            if len(values) >= 2:
                                if not ref_id:
                                    ref_id = values[0].strip().strip('"').strip("'") if values[0] else None
                                if not truth_id:
                                    truth_id = values[1].strip().strip('"').strip("'") if values[1] else None

                        # Only process if RefID is in our reference list
                        if ref_id and truth_id and ref_id in ref_ids:
                            truth_dict[ref_id] = truth_id
                        elif ref_id and truth_id:
                            # RefID not in reference list - this might indicate a mismatch
                            if row_num <= 5:  # Only log first few mismatches
                                logger.warning(f"RefID '{ref_id}' from truth file not found in link index")

                except Exception as csv_error:
                    # CSV parsing failed, try simple line-by-line parsing
                    logger.warning(f"CSV parsing failed: {csv_error}")
                    logger.info("Trying line-by-line parsing...")

                    f.seek(0)
                    lines = f.readlines()

                    # Skip potential header
                    start_line = 0
                    if lines and any(term in lines[0].lower() for term in ['refid', 'truthid', 'id', 'cluster']):
                        start_line = 1
                        logger.info("Skipping header line")

                    for line_num, line in enumerate(lines[start_line:], start_line + 1):
                        line = line.strip()
                        if not line:
                            continue

                        # Split by delimiter
                        parts = [p.strip().strip('"').strip("'") for p in line.split(delimiter)]

                        if len(parts) >= 2:
                            ref_id = parts[0]
                            truth_id = parts[1]

                            if ref_id and truth_id and ref_id in ref_ids:
                                truth_dict[ref_id] = truth_id

                logger.info(f"Successfully parsed {len(truth_dict)} truth assignments")

                # Create truth pairs from truth assignments
                truth_clusters = defaultdict(list)
                for ref_id, truth_id in truth_dict.items():
                    truth_clusters[truth_id].append(ref_id)

                # Generate pairs from truth clusters
                logger.info("Generating truth pairs from truth clusters...")
                for truth_id, members in truth_clusters.items():
                    # Generate all pairs within this cluster (including self-pairs)
                    for i in range(len(members)):
                        for j in range(i, len(members)):  # Include self-pairs
                            truth_pairs.append((members[i], members[j]))
                            # Also add reverse pair if different
                            if i != j:
                                truth_pairs.append((members[j], members[i]))

            logger.info(f"Created {len(truth_pairs)} truth pairs from {len(truth_clusters)} truth clusters")

            # Additional diagnostics
            if len(truth_dict) == 0:
                logger.error("No truth assignments found! Possible issues:")
                logger.error("  1. File format not recognized")
                logger.error("  2. Column names don't match expected patterns")
                logger.error("  3. RefIDs in truth file don't match RefIDs in link index")
                logger.error("  4. File encoding issues")

            return truth_pairs, truth_dict

        except Exception as e:
            logger.error(f"Error parsing truth file: {e}")
            raise

    @staticmethod
    def parse_complex_cluster_file(filepath: str, logger: logging.Logger) -> Tuple[List[ClusterData], Dict[str, str], Dict[str, str]]:
        """
        Parse complex cluster file format with RecIDs in brackets.
        Example: 20 BARBARA CHAVEZ [1.3, 1.4, 1.8]; [1.5, 1.6]; ...
        Or CSV format: 0,A "[328.1, 328.2]"
        Returns: (list of ClusterData objects, dict mapping RefID to ClusterID, dict mapping RefID to TruthID)
        """
        logger.info(f"Parsing complex cluster file: {filepath}")
        clusters = []
        ref_to_cluster = {}
        ref_to_truth = {}  # For automatic truth generation

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                for line_num, line in enumerate(f, 1):
                    line = line.strip()
                    if not line:
                        continue

                    # Try to extract cluster ID and RecIDs
                    rec_ids = []
                    cluster_id = None

                    # Check if it's CSV format with comma-separated cluster ID
                    if ',' in line and '[' in line:
                        # Format like: 0,A "[328.1, 328.2]" or "0,A" [328.1, 328.2]
                        comma_pos = line.find(',')
                        # Extract just the numeric part before the comma
                        cluster_id = line[:comma_pos].strip().strip('"')
                    else:
                        # Format like: 20 BARBARA CHAVEZ [1.3, 1.4, 1.8]
                        parts = line.split(maxsplit=1)
                        if parts:
                            cluster_id = parts[0]

                    # Extract all RecIDs from brackets
                    bracket_pattern = r'\[(.*?)\]'
                    matches = re.findall(bracket_pattern, line)

                    for match in matches:
                        # Split by comma and clean
                        ids = [id.strip().strip('"').strip("'") for id in match.split(',') if id.strip()]
                        rec_ids.extend(ids)

                    if rec_ids and cluster_id is not None:
                        # Sort RecIDs to find the smallest one
                        sorted_rec_ids = sorted(rec_ids)

                        # Use the smallest RecID as the new cluster ID
                        new_cluster_id = sorted_rec_ids[0]

                        cluster = ClusterData(cluster_id=new_cluster_id, record_ids=rec_ids)
                        clusters.append(cluster)

                        # Map each RefID to its new cluster ID
                        for ref_id in rec_ids:
                            ref_to_cluster[ref_id] = new_cluster_id

                            # Extract truth cluster from RecID pattern
                            if '.' in ref_id:
                                # For pattern like "328.1" -> truth is "328.1" (base + .1)
                                base_num = ref_id.split('.')[0]
                                truth_id = f"{base_num}.1"
                                ref_to_truth[ref_id] = truth_id
                            else:
                                # If no dot pattern, add .1 to make it consistent
                                ref_to_truth[ref_id] = f"{ref_id}.1"

            logger.info(f"Parsed {len(clusters)} clusters with {len(ref_to_cluster)} total records")
            logger.info(f"Automatically derived truth clusters from RecID patterns")
            logger.info(f"Renamed cluster IDs to smallest RecID in each cluster")
            return clusters, ref_to_cluster, ref_to_truth

        except Exception as e:
            logger.error(f"Error parsing complex cluster file: {e}")
            raise

    @staticmethod
    def create_cluster_file(clusters: List[ClusterData], output_path: str, logger: logging.Logger):
        """Create a clean cluster file with ClusterID and RecIDs columns."""
        logger.info(f"Creating cluster file: {output_path}")

        try:
            with open(output_path, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(['ClusterID', 'RecIDs'])

                for cluster in clusters:
                    rec_ids_str = ';'.join(cluster.record_ids)
                    writer.writerow([cluster.cluster_id, rec_ids_str])

            logger.info(f"Successfully created cluster file with {len(clusters)} clusters")

        except Exception as e:
            logger.error(f"Error creating cluster file: {e}")
            raise

# ============================================================================
# Enhanced Transitive Closure with Optimization
# ============================================================================

class TransitiveClosure:
    """Optimized transitive closure implementation using Union-Find algorithm."""

    def __init__(self):
        self.parent = {}
        self.rank = {}

    def find(self, x: str) -> str:
        """Find with path compression."""
        if x not in self.parent:
            self.parent[x] = x
            self.rank[x] = 0

        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x: str, y: str):
        """Union by rank."""
        px, py = self.find(x), self.find(y)

        if px == py:
            return

        if self.rank[px] < self.rank[py]:
            px, py = py, px

        self.parent[py] = px
        if self.rank[px] == self.rank[py]:
            self.rank[px] += 1

    def compute_closure(self, pairs: List[Tuple[str, str]], logger: logging.Logger) -> List[Tuple[str, str]]:
        """Compute transitive closure using Union-Find."""
        logger.info("Computing transitive closure using Union-Find algorithm")

        # Build union-find structure
        for a, b in pairs:
            self.union(a, b)

        # Build clusters
        clusters = defaultdict(list)
        for node in self.parent:
            root = self.find(node)
            clusters[root].append(node)

        # Generate all pairs within each cluster (including self-pairs)
        closure_pairs = []
        for root, members in clusters.items():
            for i in range(len(members)):
                for j in range(i, len(members)):  # Include self-pairs
                    closure_pairs.append((members[i], members[j]))
                    # Add reverse if different
                    if i != j:
                        closure_pairs.append((members[j], members[i]))

        logger.info(f"Transitive closure computed: {len(closure_pairs)} pairs from {len(clusters)} clusters")
        return sorted(list(set(closure_pairs)))

# ============================================================================
# Advanced Metrics Calculator - FIXED
# ============================================================================

class MetricsCalculator:
    """Comprehensive metrics calculation for entity resolution evaluation - FIXED."""

    @staticmethod
    def calculate_all_metrics(predicted_pairs: List[Tuple[str, str]],
                            truth_pairs: List[Tuple[str, str]],
                            logger: logging.Logger) -> EvaluationMetrics:
        """Calculate comprehensive evaluation metrics with progress indication - FIXED."""
        metrics = EvaluationMetrics()

        # Handle empty inputs gracefully
        if not predicted_pairs or not truth_pairs:
            logger.warning("Empty predicted or truth pairs - returning zero metrics")
            return metrics

        print("  Step 1/4: Converting pairs to sets for comparison...")
        # Convert to sets for efficient operations (normalize pair order)
        pred_set = set()
        truth_set = set()

        # Normalize pairs to ensure consistent ordering (smaller ID first)
        for a, b in predicted_pairs:
            if a <= b:
                pred_set.add((a, b))
            else:
                pred_set.add((b, a))

        for a, b in truth_pairs:
            if a <= b:
                truth_set.add((a, b))
            else:
                truth_set.add((b, a))

        print(f"    Normalized to {len(pred_set)} predicted pairs and {len(truth_set)} truth pairs")

        print("  Step 2/4: Calculating basic pairwise metrics...")
        # Basic pairwise metrics
        tp = len(pred_set & truth_set)
        fp = len(pred_set - truth_set)
        fn = len(truth_set - pred_set)

        metrics.true_positives = tp
        metrics.false_positives = fp
        metrics.false_negatives = fn

        print(f"    TP: {tp}, FP: {fp}, FN: {fn}")

        # Precision, Recall, F1
        metrics.precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        metrics.recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        metrics.f1_score = (2 * metrics.precision * metrics.recall) / (metrics.precision + metrics.recall) \
                          if (metrics.precision + metrics.recall) > 0 else 0

        # Store pairwise metrics
        metrics.pairwise_precision = metrics.precision
        metrics.pairwise_recall = metrics.recall
        metrics.pairwise_f1 = metrics.f1_score

        print("  Step 3/4: Computing cluster-level metrics...")
        # Calculate cluster-level metrics
        pred_clusters = MetricsCalculator._pairs_to_clusters(list(pred_set))
        truth_clusters = MetricsCalculator._pairs_to_clusters(list(truth_set))

        # Homogeneity, Completeness, V-measure
        metrics.homogeneity = MetricsCalculator._calculate_homogeneity(pred_clusters, truth_clusters)
        metrics.completeness = MetricsCalculator._calculate_completeness(pred_clusters, truth_clusters)
        metrics.v_measure = (2 * metrics.homogeneity * metrics.completeness) / \
                           (metrics.homogeneity + metrics.completeness) \
                           if (metrics.homogeneity + metrics.completeness) > 0 else 0

        print("  Step 4/4: Computing advanced metrics...")
        # Fowlkes-Mallows Index
        metrics.fowlkes_mallows_index = MetricsCalculator._calculate_fowlkes_mallows(
            pred_clusters, truth_clusters, tp, fp, fn
        )

        # Adjusted Rand Index (using fast approximation for large datasets)
        if len(predicted_pairs) > 10000 or len(truth_pairs) > 10000:
            print("    Using fast approximation for Adjusted Rand Index (large dataset)...")
            metrics.adjusted_rand_index = MetricsCalculator._calculate_adjusted_rand_index_fast(
                pred_clusters, truth_clusters, tp, fp, fn
            )
        else:
            metrics.adjusted_rand_index = MetricsCalculator._calculate_adjusted_rand_index(
                pred_clusters, truth_clusters
            )

        logger.info("All metrics calculated successfully")
        return metrics

    @staticmethod
    def _calculate_adjusted_rand_index_fast(pred_clusters: Dict, truth_clusters: Dict,
                                           tp: int, fp: int, fn: int) -> float:
        """Fast approximation of Adjusted Rand Index for large datasets."""
        # Use the confusion matrix approach for efficiency
        n = len(set().union(*pred_clusters.values(), *truth_clusters.values()))
        if n < 2:
            return 0

        # Total pairs
        total_pairs = n * (n - 1) / 2
        if total_pairs == 0:
            return 0

        # Agreement pairs (TP + TN)
        tn = total_pairs - tp - fp - fn
        rand_index = (tp + tn) / total_pairs

        # Simple adjustment for chance
        expected_index = 0.5  # Simplified expectation
        max_index = 1.0

        if max_index - expected_index == 0:
            return 0

        adjusted = (rand_index - expected_index) / (max_index - expected_index)
        return max(-1, min(1, adjusted))  # Bound between -1 and 1

    @staticmethod
    def _pairs_to_clusters(pairs: List[Tuple[str, str]]) -> Dict[str, Set[str]]:
        """Convert pairs to cluster representation using Union-Find."""
        # Use Union-Find to properly build clusters from pairs
        parent = {}

        def find(x):
            if x not in parent:
                parent[x] = x
            if parent[x] != x:
                parent[x] = find(parent[x])
            return parent[x]

        def union(x, y):
            px, py = find(x), find(y)
            if px != py:
                parent[py] = px

        # Build union-find structure
        for a, b in pairs:
            union(a, b)

        # Build clusters
        clusters = defaultdict(set)
        for node in parent:
            root = find(node)
            clusters[root].add(node)

        return dict(clusters)

    @staticmethod
    def _calculate_homogeneity(pred_clusters: Dict, truth_clusters: Dict) -> float:
        """Calculate homogeneity score."""
        # Simplified homogeneity calculation
        total_items = sum(len(cluster) for cluster in pred_clusters.values())
        if total_items == 0:
            return 0

        homogeneity_sum = 0
        for pred_key, pred_members in pred_clusters.items():
            max_overlap = 0
            for truth_key, truth_members in truth_clusters.items():
                overlap = len(pred_members & truth_members)
                max_overlap = max(max_overlap, overlap)
            homogeneity_sum += max_overlap

        return homogeneity_sum / total_items if total_items > 0 else 0

    @staticmethod
    def _calculate_completeness(pred_clusters: Dict, truth_clusters: Dict) -> float:
        """Calculate completeness score."""
        total_items = sum(len(cluster) for cluster in truth_clusters.values())
        if total_items == 0:
            return 0

        completeness_sum = 0
        for truth_key, truth_members in truth_clusters.items():
            max_overlap = 0
            for pred_key, pred_members in pred_clusters.items():
                overlap = len(truth_members & pred_members)
                max_overlap = max(max_overlap, overlap)
            completeness_sum += max_overlap

        return completeness_sum / total_items if total_items > 0 else 0

    @staticmethod
    def _calculate_fowlkes_mallows(pred_clusters: Dict, truth_clusters: Dict,
                                  tp: int, fp: int, fn: int) -> float:
        """Calculate Fowlkes-Mallows Index."""
        if tp + fp == 0 or tp + fn == 0:
            return 0
        precision = tp / (tp + fp)
        recall = tp / (tp + fn)
        return (precision * recall) ** 0.5

    @staticmethod
    def _calculate_adjusted_rand_index(pred_clusters: Dict, truth_clusters: Dict) -> float:
        """Calculate Adjusted Rand Index (simplified version)."""
        # This is a simplified calculation
        # For production, consider using sklearn.metrics.adjusted_rand_score
        all_items = set()
        for cluster in pred_clusters.values():
            all_items.update(cluster)
        for cluster in truth_clusters.values():
            all_items.update(cluster)

        n = len(all_items)
        if n < 2:
            return 0

        # Count agreements and disagreements
        agreements = 0
        total_pairs = 0

        items_list = list(all_items)
        for i in range(len(items_list)):
            for j in range(i + 1, len(items_list)):
                item1, item2 = items_list[i], items_list[j]
                total_pairs += 1

                # Check if items are in same cluster in both pred and truth
                in_same_pred = any(item1 in cluster and item2 in cluster
                                  for cluster in pred_clusters.values())
                in_same_truth = any(item1 in cluster and item2 in cluster
                                   for cluster in truth_clusters.values())

                if in_same_pred == in_same_truth:
                    agreements += 1

        return (agreements / total_pairs) if total_pairs > 0 else 0

# ============================================================================
# Report Generator
# ============================================================================

class ReportGenerator:
    """Generate comprehensive evaluation reports."""

    @staticmethod
    def generate_metrics_report(metrics: EvaluationMetrics, logger: logging.Logger, log_file):
        """Generate detailed metrics report with explanations."""

        report_lines = [
            "\n" + "="*80,
            "COMPREHENSIVE ENTITY RESOLUTION EVALUATION REPORT",
            "="*80,
            "\n### PAIRWISE METRICS ###\n",

            f"True Positive Pairs: {metrics.true_positives:.0f}",
            "  → Correctly identified entity pairs that belong together",

            f"False Positive Pairs: {metrics.false_positives:.0f}",
            "  → Incorrectly linked pairs that don't belong together",

            f"False Negative Pairs: {metrics.false_negatives:.0f}",
            "  → Missed pairs that should have been linked",

            f"\nPrecision: {metrics.precision:.4f}",
            "  → Proportion of identified links that are correct",
            "  → Higher precision means fewer false matches",
            "  → Range: [0,1] where 1 is perfect precision",

            f"\nRecall: {metrics.recall:.4f}",
            "  → Proportion of true links that were identified",
            "  → Higher recall means fewer missed matches",
            "  → Range: [0,1] where 1 is perfect recall",

            f"\nF1 Score: {metrics.f1_score:.4f}",
            "  → Harmonic mean of precision and recall",
            "  → Balanced measure of overall performance",
            "  → Range: [0,1] where 1 is perfect performance",

            "\n### CLUSTER QUALITY METRICS ###\n",

            f"Homogeneity: {metrics.homogeneity:.4f}",
            "  → Measures if clusters contain only members of a single true class",
            "  → Higher homogeneity means purer clusters",
            "  → Range: [0,1] where 1 means perfectly homogeneous clusters",

            f"\nCompleteness: {metrics.completeness:.4f}",
            "  → Measures if all members of a true class are in the same cluster",
            "  → Higher completeness means fewer split entities",
            "  → Range: [0,1] where 1 means no entity splitting",

            f"\nV-Measure: {metrics.v_measure:.4f}",
            "  → Harmonic mean of homogeneity and completeness",
            "  → Overall measure of clustering quality",
            "  → Range: [0,1] where 1 is perfect clustering",

            f"\nFowlkes-Mallows Index: {metrics.fowlkes_mallows_index:.4f}",
            "  → Geometric mean of precision and recall",
            "  → Alternative balanced measure of performance",
            "  → Range: [0,1] where 1 indicates perfect agreement",

            f"\nAdjusted Rand Index: {metrics.adjusted_rand_index:.4f}",
            "  → Measures similarity between predicted and true clusters",
            "  → Adjusted for chance (random clustering would score ~0)",
            "  → Range: [-1,1] where 1 is perfect agreement, 0 is random",

            "\n### PERFORMANCE INTERPRETATION ###\n",
        ]

        # Add performance interpretation
        if metrics.f1_score >= 0.9:
            report_lines.append("★ EXCELLENT: Very high quality entity resolution")
        elif metrics.f1_score >= 0.8:
            report_lines.append("★ GOOD: Strong entity resolution performance")
        elif metrics.f1_score >= 0.7:
            report_lines.append("★ FAIR: Acceptable performance with room for improvement")
        elif metrics.f1_score >= 0.6:
            report_lines.append("★ POOR: Significant improvements needed")
        else:
            report_lines.append("★ VERY POOR: Major issues with entity resolution")

        # Identify specific issues
        report_lines.append("\n### DIAGNOSTIC INSIGHTS ###\n")

        if metrics.precision < metrics.recall:
            report_lines.append("⚠ Low Precision: Too many false matches (over-linking)")
            report_lines.append("  Recommendation: Tighten matching criteria")
        elif metrics.recall < metrics.precision:
            report_lines.append("⚠ Low Recall: Too many missed matches (under-linking)")
            report_lines.append("  Recommendation: Relax matching criteria or add more features")

        if metrics.homogeneity < 0.8:
            report_lines.append("⚠ Low Homogeneity: Clusters contain mixed entities")
            report_lines.append("  Recommendation: Review clustering algorithm or similarity threshold")

        if metrics.completeness < 0.8:
            report_lines.append("⚠ Low Completeness: Same entities split across clusters")
            report_lines.append("  Recommendation: Improve transitive closure or linkage strategy")

        report_lines.append("\n" + "="*80)

        # Write to console and log file
        for line in report_lines:
            print(line)
            print(line, file=log_file)

        logger.info("Metrics report generated successfully")

    @staticmethod
    def generate_cluster_profile(clusters: List[ClusterData], logger: logging.Logger, log_file):
        """Generate detailed cluster size profile - FIXED division by zero."""

        if not clusters:
            print("\n### CLUSTER SIZE DISTRIBUTION ###", file=log_file)
            print("No clusters to analyze.", file=log_file)
            logger.warning("No clusters provided for profile generation")
            return

        size_distribution = Counter(len(cluster.record_ids) for cluster in clusters)

        print("\n### CLUSTER SIZE DISTRIBUTION ###", file=log_file)
        print("\nCluster Size Distribution:", file=log_file)
        print("Size\tCount\tRecords\tCumulative%", file=log_file)
        print("-"*50, file=log_file)

        total_records = sum(size * count for size, count in size_distribution.items())
        cumulative = 0

        for size in sorted(size_distribution.keys()):
            count = size_distribution[size]
            records = size * count
            cumulative += records
            cum_percent = (cumulative / total_records * 100) if total_records > 0 else 0

            print(f"{size}\t{count}\t{records}\t{cum_percent:.1f}%", file=log_file)

        print(f"\nTotal Clusters: {len(clusters)}", file=log_file)
        print(f"Total Records: {total_records}", file=log_file)
        print(f"Average Cluster Size: {total_records/len(clusters):.2f}" if len(clusters) > 0 else "Average Cluster Size: 0.00", file=log_file)
        print(f"Singleton Clusters: {size_distribution.get(1, 0)}", file=log_file)

        logger.info("Cluster profile generated successfully")

# ============================================================================
# Main Application Class - ENHANCED
# ============================================================================

class LinkEvaluator:
    """Main application class for link evaluation - ENHANCED."""

    def __init__(self):
        self.version = VERSION
        self.logger = self._setup_logger()
        self.start_time = time.time()
        self.log_file = None

    def _setup_logger(self) -> logging.Logger:
        """Setup logging configuration."""
        logger = logging.getLogger('LinkEvaluator')
        logger.setLevel(logging.INFO)

        # Console handler
        console_handler = logging.StreamHandler()
        console_handler.setLevel(logging.INFO)
        formatter = logging.Formatter(LOG_FORMAT, DATE_FORMAT)
        console_handler.setFormatter(formatter)
        logger.addHandler(console_handler)

        return logger

    def _create_log_file(self) -> Any:
        """Create timestamped log file."""
        now = datetime.datetime.now()
        timestamp = now.strftime("%Y%m%d_%H%M%S")
        log_filename = f"LinkEvaluator_Log_{timestamp}.txt"

        self.logger.info(f"Creating log file: {log_filename}")
        return open(log_filename, 'w', encoding='utf-8')

    def run(self):
        """Main execution flow."""
        try:
            # Initialize log file
            self.log_file = self._create_log_file()

            # Print header
            self._print_header()

            # Get input mode
            mode = self._get_input_mode()

            if mode == '1':
                self._process_standard_mode()
            elif mode == '2':
                self._process_complex_cluster_mode()
            else:
                self.logger.error("Invalid mode selected")
                return

            # Print execution time
            self._print_footer()

        except KeyboardInterrupt:
            self.logger.info("Process interrupted by user")
        except Exception as e:
            self.logger.error(f"Unexpected error: {e}", exc_info=True)
        finally:
            if self.log_file:
                self.log_file.close()

    def _print_header(self):
        """Print application header."""
        header = [
            "="*80,
            f"Advanced Link Evaluator System v{self.version} - FIXED VERSION",
            f"Execution Date: {datetime.datetime.now().strftime(DATE_FORMAT)}",
            "FIXES: Standard mode data parsing and metrics calculation",
            "="*80,
        ]
        for line in header:
            print(line)
            print(line, file=self.log_file)

    def _print_footer(self):
        """Print execution summary."""
        elapsed_time = time.time() - self.start_time
        footer = [
            "="*80,
            f"Total Execution Time: {elapsed_time/60:.2f} minutes",
            "Process completed successfully",
            "="*80,
        ]
        for line in footer:
            print(line)
            print(line, file=self.log_file)

    def _get_input_mode(self) -> str:
        """Get processing mode from user."""
        print("\nSelect Input Mode:")
        print("1. Standard Mode (Link Index + Linked Pairs + Truth File) - FIXED")
        print("2. Complex Cluster Mode (Process complex cluster format)")

        mode = input("\nEnter mode (1 or 2): ").strip()
        print(f"Selected mode: {mode}", file=self.log_file)
        return mode

    def _process_standard_mode(self):
        """Process standard link index format - FIXED."""
        parser = FileParser()

        print("\n" + "="*50)
        print("STANDARD MODE - ENHANCED DATA VALIDATION")
        print("="*50)

        # Get link index file
        link_index_file = input("\nEnter link index filename: ").strip()

        # Enhanced parsing with validation
        base_pairs, ref_ids, ref_to_cluster = parser.parse_link_index(link_index_file, self.logger)

        print(f"\n✓ Link Index Analysis:")
        print(f"  - Total RefIDs: {len(ref_ids)}")
        print(f"  - Unique Clusters: {len(set(ref_to_cluster.values()))}")
        print(f"  - Pairs from clusters: {len(base_pairs)}")

        # Validate data quality
        cluster_sizes = Counter(ref_to_cluster.values())
        singleton_clusters = sum(1 for count in cluster_sizes.values() if count == 1)
        multi_clusters = len(cluster_sizes) - singleton_clusters

        print(f"  - Singleton clusters: {singleton_clusters}")
        print(f"  - Multi-record clusters: {multi_clusters}")

        # Get linked pairs file
        linked_pairs_file = input("\nEnter linked pairs filename: ").strip()
        additional_pairs = parser.parse_linked_pairs(linked_pairs_file, self.logger)

        print(f"\n✓ Linked Pairs Analysis:")
        print(f"  - Additional pairs: {len(additional_pairs)}")

        # Combine pairs with deduplication
        print(f"\n✓ Combining pairs...")
        all_pairs_set = set(base_pairs + additional_pairs)
        all_pairs = list(all_pairs_set)

        print(f"  - Total unique pairs before closure: {len(all_pairs)}")

        # Get truth file
        truth_file = input("\nEnter truth filename: ").strip()
        truth_pairs, truth_dict = parser.parse_truth_file(truth_file, ref_ids, self.logger)

        print(f"\n✓ Truth Data Analysis:")
        print(f"  - RefIDs with truth: {len(truth_dict)}")
        print(f"  - Truth clusters: {len(set(truth_dict.values()))}")
        print(f"  - Truth pairs: {len(truth_pairs)}")

        # Validate truth coverage
        missing_truth = set(ref_ids) - set(truth_dict.keys())
        if missing_truth:
            print(f"  - ⚠ WARNING: {len(missing_truth)} RefIDs missing truth assignments")
            print(f"    Sample missing: {list(missing_truth)[:5]}")

            # If no truth data found, provide detailed diagnostics
            if len(truth_dict) == 0:
                print(f"\n⚠⚠⚠ CRITICAL: No truth assignments found!")
                print(f"Possible issues:")
                print(f"  1. Wrong file format or delimiter")
                print(f"  2. Column headers don't match expected names")
                print(f"  3. RefIDs in truth file don't match those in link index")
                print(f"  4. File encoding problems")

                # Try to show some sample RefIDs for comparison
                print(f"\nSample RefIDs from link index: {ref_ids[:10]}")

                # Try to peek at truth file format
                try:
                    with open(truth_file, 'r', encoding='utf-8') as f:
                        sample_lines = [f.readline().strip() for _ in range(3)]
                    print(f"Sample lines from truth file:")
                    for i, line in enumerate(sample_lines):
                        print(f"  Line {i+1}: {line}")
                except:
                    print("  Could not read sample lines from truth file")

                # Ask user if they want to continue without evaluation
                continue_choice = input("\nNo truth data found. Continue without evaluation? (y/n): ").strip().lower()
                if continue_choice != 'y':
                    print("Exiting...")
                    return
                else:
                    print("Continuing with clustering analysis only...")

        # Run transitive closure
        print(f"\n✓ Computing transitive closure...")
        closure_engine = TransitiveClosure()
        closure_pairs = closure_engine.compute_closure(all_pairs, self.logger)

        print(f"  - Pairs after closure: {len(closure_pairs)}")

        # Generate cluster profile from closure (even without truth data)
        print(f"\n✓ Generating cluster profile from predicted clusters...")
        clusters_dict = defaultdict(list)

        # Build clusters from all closure pairs
        uf_parent = {}
        def uf_find(x):
            if x not in uf_parent:
                uf_parent[x] = x
            if uf_parent[x] != x:
                uf_parent[x] = uf_find(uf_parent[x])
            return uf_parent[x]

        def uf_union(x, y):
            px, py = uf_find(x), uf_find(y)
            if px != py:
                uf_parent[py] = px

        # Build clusters using Union-Find
        for a, b in closure_pairs:
            uf_union(a, b)

        # Group items by root
        root_clusters = defaultdict(list)
        for item in uf_parent:
            root = uf_find(item)
            root_clusters[root].append(item)

        clusters = [ClusterData(cluster_id=cid, record_ids=list(set(rids)))
                   for cid, rids in root_clusters.items() if rids]

        reporter = ReportGenerator()
        reporter.generate_cluster_profile(clusters, self.logger, self.log_file)

        # Only proceed with evaluation if we have truth data
        if len(truth_dict) > 0:
            # Filter pairs to only include those with truth data
            print(f"\n✓ Filtering pairs for evaluation...")
            truth_ref_ids = set(truth_dict.keys())

            filtered_closure_pairs = [(a, b) for a, b in closure_pairs
                                    if a in truth_ref_ids and b in truth_ref_ids]
            filtered_truth_pairs = [(a, b) for a, b in truth_pairs
                                  if a in truth_ref_ids and b in truth_ref_ids]

            print(f"  - Predicted pairs (filtered): {len(filtered_closure_pairs)}")
            print(f"  - Truth pairs (filtered): {len(filtered_truth_pairs)}")

            if len(filtered_closure_pairs) > 0 and len(filtered_truth_pairs) > 0:
                # Calculate metrics
                print(f"\n✓ Calculating evaluation metrics...")
                calculator = MetricsCalculator()
                metrics = calculator.calculate_all_metrics(filtered_closure_pairs, filtered_truth_pairs, self.logger)

                # Generate reports
                reporter.generate_metrics_report(metrics, self.logger, self.log_file)
            else:
                print(f"\n⚠ Cannot calculate metrics: insufficient filtered pairs")
        else:
            print(f"\n⚠ Skipping metrics calculation: no truth data available")

        # Additional diagnostic information
        print(f"\n### DIAGNOSTIC SUMMARY ###")
        print(f"Input Quality Assessment:")
        print(f"  - Link Index Coverage: {len(ref_ids)} RefIDs")
        print(f"  - Truth Coverage: {len(truth_dict)}/{len(ref_ids)} ({100*len(truth_dict)/len(ref_ids):.1f}%)" if len(ref_ids) > 0 else "  - Truth Coverage: 0/0 (0.0%)")
        print(f"  - Additional Links: {len(additional_pairs)} pairs")
        print(f"  - Transitive Expansion: {len(all_pairs)} → {len(closure_pairs)} pairs")
        print(f"  - Final Clusters: {len(clusters)}")

        if len(truth_dict) > 0 and hasattr(self, 'metrics') and hasattr(self.metrics, 'precision') and self.metrics.precision < 0.5:
            print(f"\n⚠ LOW PRECISION ALERT:")
            print(f"  This suggests data quality issues. Check:")
            print(f"  1. Are the file formats correct?")
            print(f"  2. Do ClusterIDs in link index match truth assignments?")
            print(f"  3. Are there too many spurious links in linked pairs file?")
            print(f"  4. Is transitive closure creating over-connected components?")
        elif len(truth_dict) == 0:
            print(f"\n💡 RECOMMENDATION:")
            print(f"  Fix the truth file parsing to enable evaluation metrics.")
            print(f"  Current analysis shows only clustering structure without accuracy assessment.")

    def _process_complex_cluster_mode(self):
        """Process complex cluster file format."""
        parser = FileParser()

        # Get complex cluster file
        cluster_file = input("\nEnter complex cluster filename: ").strip()
        clusters, ref_to_cluster, ref_to_truth = parser.parse_complex_cluster_file(cluster_file, self.logger)

        # Create output files
        output_prefix = Path(cluster_file).stem

        # Create clean cluster file
        clean_cluster_file = f"{output_prefix}_clusters.csv"
        parser.create_cluster_file(clusters, clean_cluster_file, self.logger)
        print(f"Created cluster file: {clean_cluster_file}")

        # Create link index file with properly formatted ClusterIDs
        link_index_file = f"{output_prefix}_linkindex.csv"
        with open(link_index_file, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['RefID', 'ClusterID'])

            # Sort by RefID for better readability
            for ref_id in sorted(ref_to_cluster.keys()):
                cluster_id = ref_to_cluster[ref_id]
                writer.writerow([ref_id, cluster_id])

        print(f"Created link index file: {link_index_file}")
        self.logger.info(f"Link index uses smallest RecID as ClusterID for each cluster")

        # Generate cluster profile
        reporter = ReportGenerator()
        reporter.generate_cluster_profile(clusters, self.logger, self.log_file)

        # Ask about truth file
        truth_option = input("\nDo you have a separate truth file for evaluation? (y/n): ").strip().lower()

        if truth_option == 'y':
            # Use provided truth file
            truth_file = input("Enter truth filename: ").strip()
            ref_ids = list(ref_to_cluster.keys())
            truth_pairs, _ = parser.parse_truth_file(truth_file, ref_ids, self.logger)
            print("Using provided truth file for evaluation")
        else:
            # Use automatic truth derivation from RecID patterns
            print("\nUsing automatic truth derivation from RecID patterns")
            print("Truth pattern: Records like 328.1, 328.2 → Truth Cluster 328.1")
            print("Processing truth clusters...")

            # Create truth pairs from automatic truth assignments
            truth_clusters = defaultdict(list)
            for ref_id, truth_id in ref_to_truth.items():
                truth_clusters[truth_id].append(ref_id)

            # Generate truth pairs with progress indicator
            print("Generating truth pairs...")
            truth_pairs = []
            total_clusters = len(truth_clusters)
            processed = 0

            for truth_id, members in truth_clusters.items():
                if len(members) > 1:
                    for i in range(len(members)):
                        for j in range(i, len(members)):
                            truth_pairs.append((members[i], members[j]))
                            if i != j:
                                truth_pairs.append((members[j], members[i]))
                else:
                    # Single member clusters still need self-pairs
                    truth_pairs.append((members[0], members[0]))

                processed += 1
                if processed % 100 == 0 or processed == total_clusters:
                    print(f"  Progress: {processed}/{total_clusters} clusters processed", end='\r')

            print(f"\n✓ Generated {len(truth_pairs)} truth pairs from {len(truth_clusters)} automatic truth clusters")
            self.logger.info(f"Generated {len(truth_pairs)} truth pairs from {len(truth_clusters)} automatic truth clusters")

            # Create and save automatic truth file for reference
            auto_truth_file = f"{output_prefix}_auto_truth.csv"
            with open(auto_truth_file, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(['RefID', 'TruthID'])

                # Sort for better readability
                for ref_id in sorted(ref_to_truth.keys()):
                    truth_id = ref_to_truth[ref_id]
                    writer.writerow([ref_id, truth_id])

            print(f"Created automatic truth file for reference: {auto_truth_file}")

        # Convert predicted clusters to pairs for evaluation with progress
        print("\nGenerating predicted pairs...")
        predicted_pairs = []
        total_pred_clusters = len(clusters)

        for idx, cluster in enumerate(clusters, 1):
            if len(cluster.record_ids) > 1:
                for i in range(len(cluster.record_ids)):
                    for j in range(i, len(cluster.record_ids)):
                        predicted_pairs.append((cluster.record_ids[i], cluster.record_ids[j]))
                        if i != j:
                            predicted_pairs.append((cluster.record_ids[j], cluster.record_ids[i]))
            else:
                # Single member clusters
                predicted_pairs.append((cluster.record_ids[0], cluster.record_ids[0]))

            if idx % 100 == 0 or idx == total_pred_clusters:
                print(f"  Progress: {idx}/{total_pred_clusters} clusters processed", end='\r')

        print(f"\n✓ Generated {len(predicted_pairs)} predicted pairs from {len(clusters)} clusters")

        # Calculate metrics with progress indication
        print("\nCalculating evaluation metrics...")
        print(f"  Comparing {len(predicted_pairs)} predicted pairs with {len(truth_pairs)} truth pairs")
        print("  This may take a moment for large datasets...")

        calculator = MetricsCalculator()
        metrics = calculator.calculate_all_metrics(predicted_pairs, truth_pairs, self.logger)

        print("✓ Metrics calculation complete")

        # Generate report
        reporter.generate_metrics_report(metrics, self.logger, self.log_file)

        # Additional analysis for automatic truth
        if truth_option != 'y':
            print("\n### AUTOMATIC TRUTH ANALYSIS ###")
            print(f"Truth clusters detected: {len(truth_clusters)}")
            print(f"Predicted clusters: {len(clusters)}")

            # Also write to log file
            print("\n### AUTOMATIC TRUTH ANALYSIS ###", file=self.log_file)
            print(f"Truth clusters detected: {len(truth_clusters)}", file=self.log_file)
            print(f"Predicted clusters: {len(clusters)}", file=self.log_file)

            # Show sample truth assignments
            print("\nSample Cluster Assignments (first 10):")
            print("\nSample Cluster Assignments (first 10):", file=self.log_file)

            sample_items = list(ref_to_truth.items())[:10]
            for ref_id, truth_id in sample_items:
                predicted_id = ref_to_cluster[ref_id]
                # Check if the base numbers match
                truth_base = truth_id.split('.')[0] if '.' in truth_id else truth_id
                pred_base = predicted_id.split('.')[0] if '.' in predicted_id else predicted_id
                match = "✓" if truth_base == pred_base else "✗"

                print(f"  {ref_id} → Truth: {truth_id}, Predicted: {predicted_id} {match}")
                print(f"  {ref_id} → Truth: {truth_id}, Predicted: {predicted_id} {match}", file=self.log_file)

# ============================================================================
# Entry Point
# ============================================================================

def main():
    """Main entry point."""
    app = LinkEvaluator()
    app.run()

if __name__ == "__main__":
    main()

2025-09-07 08:08:15 - INFO - Creating log file: LinkEvaluator_Log_20250907_080815.txt
2025-09-07 08:08:15 - INFO - Creating log file: LinkEvaluator_Log_20250907_080815.txt
2025-09-07 08:08:15 - INFO - Creating log file: LinkEvaluator_Log_20250907_080815.txt
INFO:LinkEvaluator:Creating log file: LinkEvaluator_Log_20250907_080815.txt


Advanced Link Evaluator System v2.1 - FIXED VERSION
Execution Date: 2025-09-07 08:08:15
FIXES: Standard mode data parsing and metrics calculation

Select Input Mode:
1. Standard Mode (Link Index + Linked Pairs + Truth File) - FIXED
2. Complex Cluster Mode (Process complex cluster format)

Enter mode (1 or 2): 1

STANDARD MODE - ENHANCED DATA VALIDATION

Enter link index filename: z_LinkIndex_Taha.txt


2025-09-07 08:08:32 - INFO - Parsing link index file: z_LinkIndex_Taha.txt
2025-09-07 08:08:32 - INFO - Parsing link index file: z_LinkIndex_Taha.txt
2025-09-07 08:08:32 - INFO - Parsing link index file: z_LinkIndex_Taha.txt
INFO:LinkEvaluator:Parsing link index file: z_LinkIndex_Taha.txt
2025-09-07 08:08:32 - INFO - Generating pairs from link index clusters...
2025-09-07 08:08:32 - INFO - Generating pairs from link index clusters...
2025-09-07 08:08:32 - INFO - Generating pairs from link index clusters...
INFO:LinkEvaluator:Generating pairs from link index clusters...
2025-09-07 08:08:32 - INFO - Parsed 5000 references in 550 clusters
2025-09-07 08:08:32 - INFO - Parsed 5000 references in 550 clusters
2025-09-07 08:08:32 - INFO - Parsed 5000 references in 550 clusters
INFO:LinkEvaluator:Parsed 5000 references in 550 clusters
2025-09-07 08:08:32 - INFO - Generated 60498 pairs from link index
2025-09-07 08:08:32 - INFO - Generated 60498 pairs from link index
2025-09-07 08:08:32 - INFO -


✓ Link Index Analysis:
  - Total RefIDs: 5000
  - Unique Clusters: 550
  - Pairs from clusters: 60498
  - Singleton clusters: 15
  - Multi-record clusters: 535

Enter linked pairs filename: z_LinkedPair_Taha.txt


2025-09-07 08:08:52 - INFO - Parsing linked pairs file: z_LinkedPair_Taha.txt
2025-09-07 08:08:52 - INFO - Parsing linked pairs file: z_LinkedPair_Taha.txt
2025-09-07 08:08:52 - INFO - Parsing linked pairs file: z_LinkedPair_Taha.txt
INFO:LinkEvaluator:Parsing linked pairs file: z_LinkedPair_Taha.txt
2025-09-07 08:08:52 - INFO - Parsed 8900 linked pairs (including bidirectional)
2025-09-07 08:08:52 - INFO - Parsed 8900 linked pairs (including bidirectional)
2025-09-07 08:08:52 - INFO - Parsed 8900 linked pairs (including bidirectional)
INFO:LinkEvaluator:Parsed 8900 linked pairs (including bidirectional)



✓ Linked Pairs Analysis:
  - Additional pairs: 8900

✓ Combining pairs...
  - Total unique pairs before closure: 60498

Enter truth filename: z_Truthfile_Taha.txt


2025-09-07 08:09:01 - INFO - Parsing truth file: z_Truthfile_Taha.txt
2025-09-07 08:09:01 - INFO - Parsing truth file: z_Truthfile_Taha.txt
2025-09-07 08:09:01 - INFO - Parsing truth file: z_Truthfile_Taha.txt
INFO:LinkEvaluator:Parsing truth file: z_Truthfile_Taha.txt
2025-09-07 08:09:01 - INFO - Analyzing truth file format...
2025-09-07 08:09:01 - INFO - Analyzing truth file format...
2025-09-07 08:09:01 - INFO - Analyzing truth file format...
INFO:LinkEvaluator:Analyzing truth file format...
2025-09-07 08:09:01 - INFO -   Line 1: RecID,IdTruth
2025-09-07 08:09:01 - INFO -   Line 1: RecID,IdTruth
2025-09-07 08:09:01 - INFO -   Line 1: RecID,IdTruth
INFO:LinkEvaluator:  Line 1: RecID,IdTruth
2025-09-07 08:09:01 - INFO -   Line 2: 1.1,1.1
2025-09-07 08:09:01 - INFO -   Line 2: 1.1,1.1
2025-09-07 08:09:01 - INFO -   Line 2: 1.1,1.1
INFO:LinkEvaluator:  Line 2: 1.1,1.1
2025-09-07 08:09:01 - INFO -   Line 3: 1.2,1.1
2025-09-07 08:09:01 - INFO -   Line 3: 1.2,1.1
2025-09-07 08:09:01 - INFO


✓ Truth Data Analysis:
  - RefIDs with truth: 5000
  - Truth clusters: 496
  - Truth pairs: 63084

✓ Computing transitive closure...
  - Pairs after closure: 60498

✓ Generating cluster profile from predicted clusters...

✓ Filtering pairs for evaluation...
  - Predicted pairs (filtered): 60498
  - Truth pairs (filtered): 63084

✓ Calculating evaluation metrics...
  Step 1/4: Converting pairs to sets for comparison...
    Normalized to 32749 predicted pairs and 34042 truth pairs
  Step 2/4: Calculating basic pairwise metrics...
    TP: 32607, FP: 142, FN: 1435
  Step 3/4: Computing cluster-level metrics...


2025-09-07 08:09:02 - INFO - All metrics calculated successfully
2025-09-07 08:09:02 - INFO - All metrics calculated successfully
2025-09-07 08:09:02 - INFO - All metrics calculated successfully
INFO:LinkEvaluator:All metrics calculated successfully
2025-09-07 08:09:02 - INFO - Metrics report generated successfully
2025-09-07 08:09:02 - INFO - Metrics report generated successfully
2025-09-07 08:09:02 - INFO - Metrics report generated successfully
INFO:LinkEvaluator:Metrics report generated successfully


  Step 4/4: Computing advanced metrics...
    Using fast approximation for Adjusted Rand Index (large dataset)...

COMPREHENSIVE ENTITY RESOLUTION EVALUATION REPORT

### PAIRWISE METRICS ###

True Positive Pairs: 32607
  → Correctly identified entity pairs that belong together
False Positive Pairs: 142
  → Incorrectly linked pairs that don't belong together
False Negative Pairs: 1435
  → Missed pairs that should have been linked

Precision: 0.9957
  → Proportion of identified links that are correct
  → Higher precision means fewer false matches
  → Range: [0,1] where 1 is perfect precision

Recall: 0.9578
  → Proportion of true links that were identified
  → Higher recall means fewer missed matches
  → Range: [0,1] where 1 is perfect recall

F1 Score: 0.9764
  → Harmonic mean of precision and recall
  → Balanced measure of overall performance
  → Range: [0,1] where 1 is perfect performance

### CLUSTER QUALITY METRICS ###

Homogeneity: 0.9966
  → Measures if clusters contain only membe